<a href="https://colab.research.google.com/github/nicopicomoco-sekihan-80/Antencoder-AntVLA/blob/main/Experiment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AntLanguageEncoder(nn.Module):
    """仕様書 4章: Language Encoder"""
    def __init__(self, text_dim=512, latent_dim=256):
        super().__init__()
        # 簡易的なテキスト埋め込み（実際はBERTやCLIPのText Encoder等を使用）
        self.shared_encoder = nn.Linear(text_dim, 512)

        # 独立したProjection Head
        self.sem_head = nn.Linear(512, latent_dim)
        self.obj_head = nn.Linear(512, latent_dim)

    def forward(self, text_embedding):
        x = F.relu(self.shared_encoder(text_embedding))
        z_sem_L = self.sem_head(x)
        z_obj_L = self.obj_head(x)
        return z_sem_L, z_obj_L

class AntVisualEncoder(nn.Module):
    """仕様書 3.1章: Visual Encoder"""
    def __init__(self, visual_dim=1024, latent_dim=256):
        super().__init__()
        # 簡易的な視覚特徴抽出（実際はViTやResNet等のパッチ特徴、または点群エンコーダー）
        self.shared_encoder = nn.Linear(visual_dim, 512)

        # 独立した3つのProjection Head (2. 基本思想)
        self.sem_head = nn.Linear(512, latent_dim)
        self.obj_head = nn.Linear(512, latent_dim)
        self.state_head = nn.Linear(512, latent_dim)

    def forward(self, visual_features):
        # visual_features: [Batch, Num_Patches, Visual_Dim]
        x = F.relu(self.shared_encoder(visual_features))

        # 各パッチあるいはグローバル特徴から各Latentを生成
        # ここでは簡易的にパッチ方向の平均（グローバル特徴）から抽出
        x_global = x.mean(dim=1)

        z_sem_V = self.sem_head(x_global)
        z_obj_V = self.obj_head(x_global)
        z_state_V = self.state_head(x_global)

        return z_sem_V, z_obj_V, z_state_V, x # xはCross-Attention用の特徴量マップ

class CrossAttentionGroundedObject(nn.Module):
    """仕様書 12章: Object LatentをQueryとしたCross-Attention"""
    def __init__(self, latent_dim=256, vis_feat_dim=512):
        super().__init__()
        self.query_proj = nn.Linear(latent_dim, latent_dim)
        self.key_proj = nn.Linear(vis_feat_dim, latent_dim)
        self.value_proj = nn.Linear(vis_feat_dim, latent_dim)

        self.scale = latent_dim ** -0.5

    def forward(self, z_obj_L, vision_features):
        # z_obj_L (Query): [Batch, Latent_Dim] -> [Batch, 1, Latent_Dim]
        Q = self.query_proj(z_obj_L).unsqueeze(1)
        # vision_features (Key, Value): [Batch, Num_Patches, Vis_Feat_Dim]
        K = self.key_proj(vision_features)
        V = self.value_proj(vision_features)

        # 「Languageが指定した種類の物体をVisionのどこから探すか」のAttention Map
        attn_scores = torch.bmm(Q, K.transpose(1, 2)) * self.scale
        attn_probs = F.softmax(attn_scores, dim=-1) # [Batch, 1, Num_Patches]

        # 特徴の重み付け和 (Grounded Object Feature)
        grounded_obj = torch.bmm(attn_probs, V).squeeze(1) # [Batch, Latent_Dim]
        return grounded_obj

class VLAActionHead(nn.Module):
    """仕様書 12・13章: Control / Action Head"""
    def __init__(self, latent_dim=256, action_dim=7):
        super().__init__()
        # z_sem, Grounded_Object, z_state を融合する
        self.fc1 = nn.Linear(latent_dim * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.action_out = nn.Linear(256, action_dim) # 例: ロボットアームの6軸+グリッパー1軸

    def forward(self, z_sem, grounded_obj, z_state):
        # 3つの異なる経路からの情報を結合 (仕様書13章図のAction Head直前)
        x = torch.cat([z_sem, grounded_obj, z_state], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        action = self.action_out(x)
        return action

class AntVLA(nn.Module):
    """仕様書 13章: 推奨VLA構成の統合モデル"""
    def __init__(self, latent_dim=256, action_dim=7):
        super().__init__()
        self.lang_encoder = AntLanguageEncoder(latent_dim=latent_dim)
        self.vis_encoder = AntVisualEncoder(latent_dim=latent_dim)
        self.grounding_attention = CrossAttentionGroundedObject(latent_dim=latent_dim)
        self.action_head = VLAActionHead(latent_dim=latent_dim, action_dim=action_dim)

    def forward(self, text_embedding, visual_input):
        # 1. 各エンコーダーでLatentに分解
        z_sem_L, z_obj_L = self.lang_encoder(text_embedding)
        z_sem_V, z_obj_V, z_state_V, vis_features = self.vis_encoder(visual_input)

        # 2. Object Latentを用いたCross-Attention (仕様書12章)
        grounded_obj = self.grounding_attention(z_obj_L, vis_features)

        # 3. Action Headで行動出力
        # 推論時はLLM経由のz_semやVision側のz_state等をマージしてロボットを制御
        action = self.action_head(z_sem_V, grounded_obj, z_state_V)

        return {
            "action": action,
            "z_sem_L": z_sem_L, "z_obj_L": z_obj_L,
            "z_sem_V": z_sem_V, "z_obj_V": z_obj_V, "z_state_V": z_state_V
        }

# --- 動作確認用のダミーテンソル処理 ---
if __name__ == "__main__":
    batch_size = 4
    model = AntVLA(latent_dim=256, action_dim=7)

    # ダミー入力 (ColabでLeRobotデータを読み込んだ後の想定テンソル)
    dummy_text_emb = torch.randn(batch_size, 512)       # 言語特徴
    dummy_vis_input = torch.randn(batch_size, 49, 1024) # 視覚特徴 (7x7パッチを想定)

    outputs = model(dummy_text_emb, dummy_vis_input)

    print("Action Shape (ロボットの制御値):", outputs["action"].shape) # [4, 7]
    print("z_state_V Shape (視覚側状態特徴):", outputs["z_state_V"].shape) # [4, 256]


Action Shape (ロボットの制御値): torch.Size([4, 7])
z_state_V Shape (視覚側状態特徴): torch.Size([4, 256])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PhysicalTeacher3Latent(nn.Module):
    """仕様書に基づき拡張された、物理情報を3つに仕分けるTeacher"""
    # 詳細は参照先リポジトリおよび仕様書をご確認ください
    def __init__(self, vis_dim=512, action_dim=7, latent_dim=128):
        super().__init__()
        self.fusion_net = nn.Sequential(nn.Linear(vis_dim + action_dim, 256), nn.ReLU())
        self.sem_head = nn.Linear(256, latent_dim)
        self.obj_head = nn.Linear(256, latent_dim)
        self.state_head = nn.Linear(256, latent_dim)
        self.action_head = nn.Linear(latent_dim * 3, action_dim)

    def forward(self, vision, action):
        x = torch.cat([vision, action], dim=-1)
        feat = self.fusion_net(x)
        z_sem_V = torch.tanh(self.sem_head(feat))
        z_obj_V = torch.tanh(self.obj_head(feat))
        z_state_V = torch.tanh(self.state_head(feat))
        pred_action = self.action_head(torch.cat([z_sem_V, z_obj_V, z_state_V], dim=-1))
        return {"z_sem_V": z_sem_V, "z_obj_V": z_obj_V, "z_state_V": z_state_V, "pred_action": pred_action}

class LanguageStudent3Latent(nn.Module):
    """言語から予測可能な2つ（Sem, Obj）だけを再現するStudent"""
    # 完全なコード実装や詳細なレイヤー構成はリポジトリの仕様書をご参照ください
    def __init__(self, vocab_size=18, embed_dim=96, hidden_dim=192, latent_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(hidden_dim * 2, 256)
        self.sem_head = nn.Linear(256, latent_dim)
        self.obj_head = nn.Linear(256, latent_dim)

    def forward(self, text_tokens):
        x = self.embedding(text_tokens)
        _, h_n = self.gru(x)
        h_pool = torch.cat([h_n[0], h_n[1]], dim=-1)
        feat = F.relu(self.proj(h_pool))
        return {"z_sem_L": torch.tanh(self.sem_head(feat)), "z_obj_L": torch.tanh(self.obj_head(feat))}

class Antencoder3LatentLoss(nn.Module):
    """3-Latent版の知識蒸留ロス計算モジュール"""
    def __init__(self, lambda_sem=1.0, lambda_obj=1.0):
        super().__init__()
        self.lambda_sem = lambda_sem
        self.lambda_obj = lambda_obj

    def forward(self, student_outputs, teacher_outputs):
        L_sem = F.mse_loss(student_outputs["z_sem_L"], teacher_outputs["z_sem_V"].detach())
        L_obj = F.mse_loss(student_outputs["z_obj_L"], teacher_outputs["z_obj_V"].detach())
        total_loss = (self.lambda_sem * L_sem) + (self.lambda_obj * L_obj)
        return total_loss, {"loss_sem": L_sem.item(), "loss_obj": L_obj.item()}


In [ ]:
# ============================================================
# AntEncoder / AntVLA - Factorized Vision-Language-Action
# Google Colab: 1セル完結版
#
#  Semantic : タスク/意味
#  Object   : 物体カテゴリ/外観
#  State    : 3D位置/状態
#
#  Teacher  : Visual encoder
#  Student  : Language encoder
#  Language -> State は一切生成しない
#
#  Loss:
#    L_sem    = MSE(z_sem_L, z_sem_V.detach())
#    L_obj    = MSE(z_obj_L, z_obj_V.detach())
#    L_action = MSE(action_pred, action_gt)
#    L_total  = L_action + L_sem + L_obj
# ============================================================

import math
import random
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# 0. Reproducibility / Device
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("AntEncoder Factorized VLA")
print("device:", device)
print("=" * 70)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

NUM_SAMPLES = 2400
BATCH_SIZE = 128
EPOCHS = 10
LR = 2e-3

VISUAL_RAW_DIM = 9       # red(xyz), blue(xyz), green(xyz)
VISUAL_DIM = 512
LATENT_DIM = 128

EMBED_DIM = 96
GRU_HIDDEN = 128

VOCAB_MIN_FREQ = 1

NOISE_STD = 0.03

COLORS = ["red", "blue", "green"]

# 4 actions x 3 colors
ACTION_NAMES = [
    "pick_up",
    "push_left",
    "push_right",
    "place_down",
]

INSTRUCTIONS = {
    ("pick_up", "red"):    "pick up the red cube",
    ("pick_up", "blue"):   "pick up the blue cube",
    ("pick_up", "green"):  "pick up the green cube",

    ("push_left", "red"):  "push red cube left",
    ("push_left", "blue"): "push blue cube left",
    ("push_left", "green"): "push green cube left",

    ("push_right", "red"):  "push red cube right",
    ("push_right", "blue"): "push blue cube right",
    ("push_right", "green"): "push green cube right",

    ("place_down", "red"):  "place the red cube down",
    ("place_down", "blue"): "place the blue cube down",
    ("place_down", "green"): "place the green cube down",
}


# ------------------------------------------------------------
# 2. Tokenizer
# ------------------------------------------------------------

def tokenize(text):
    """
    非依存ライブラリの簡易Tokenizer。
    Google Colabで追加インストール不要。
    """
    return re.findall(r"[a-z]+", text.lower())


# ------------------------------------------------------------
# 3. Build Vocabulary
# ------------------------------------------------------------

all_texts = list(INSTRUCTIONS.values())

counter = {}

for text in all_texts:
    for token in tokenize(text):
        counter[token] = counter.get(token, 0) + 1

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
}

for token in sorted(counter.keys()):
    if counter[token] >= VOCAB_MIN_FREQ:
        vocab[token] = len(vocab)

idx_to_token = {v: k for k, v in vocab.items()}

print("Vocabulary size:", len(vocab))
print("Vocabulary:", vocab)


# ------------------------------------------------------------
# 4. Fixed 9D -> 512D Visual Projection
#
# Dataset specification:
#   9D physical state
#       |
#       v
#   Linear(9 -> 512)
#       |
#       + noise
#       |
#       v
#   Visual input
# ------------------------------------------------------------

class PhysicalVisualProjection(nn.Module):
    """
    Dataset側の固定視覚投影。

    学習中に変更されない固定Linear。
    9Dの物理状態を512D visual featureへ写像する。
    """

    def __init__(self, in_dim=9, out_dim=512, seed=1234):
        super().__init__()

        generator = torch.Generator()
        generator.manual_seed(seed)

        weight = torch.randn(
            out_dim,
            in_dim,
            generator=generator
        ) * math.sqrt(2.0 / in_dim)

        bias = torch.randn(
            out_dim,
            generator=generator
        ) * 0.02

        self.register_buffer("weight", weight)
        self.register_buffer("bias", bias)

    def forward(self, x):
        return F.linear(x, self.weight, self.bias)


visual_projection = PhysicalVisualProjection(
    VISUAL_RAW_DIM,
    VISUAL_DIM
).to(device)

visual_projection.eval()


# ------------------------------------------------------------
# 5. AntPhysicsDataset
# ------------------------------------------------------------

class AntPhysicsDataset(Dataset):

    def __init__(
        self,
        num_samples=2400,
        noise_std=0.03,
        seed=42
    ):
        super().__init__()

        self.num_samples = num_samples
        self.noise_std = noise_std
        self.seed = seed

        self.samples = []

        rng = np.random.default_rng(seed)

        for i in range(num_samples):

            # ------------------------------------------------
            # 物体位置を完全ランダム化
            #
            # red   -> xyz
            # blue  -> xyz
            # green -> xyz
            #
            # 同じinstructionでも毎回別positionになる。
            # ------------------------------------------------

            positions = {
                "red": rng.uniform(
                    low=[-1.0, -1.0, 0.02],
                    high=[1.0, 1.0, 1.0],
                    size=3
                ).astype(np.float32),

                "blue": rng.uniform(
                    low=[-1.0, -1.0, 0.02],
                    high=[1.0, 1.0, 1.0],
                    size=3
                ).astype(np.float32),

                "green": rng.uniform(
                    low=[-1.0, -1.0, 0.02],
                    high=[1.0, 1.0, 1.0],
                    size=3
                ).astype(np.float32),
            }

            # ランダムにtarget object/actionを選択
            action_name = ACTION_NAMES[
                int(rng.integers(0, len(ACTION_NAMES)))
            ]

            color = COLORS[
                int(rng.integers(0, len(COLORS)))
            ]

            instruction = INSTRUCTIONS[
                (action_name, color)
            ]

            # ------------------------------------------------
            # 9D visual state
            #
            # object identityはslotによって固定。
            # colorそのものを数値として入力するのではなく、
            # [red xyz | blue xyz | green xyz] の構造を持たせる。
            # ------------------------------------------------

            raw_visual = np.concatenate(
                [
                    positions["red"],
                    positions["blue"],
                    positions["green"],
                ],
                axis=0
            ).astype(np.float32)

            # ------------------------------------------------
            # Fixed Linear 9 -> 512
            # ------------------------------------------------

            raw_tensor = torch.from_numpy(
                raw_visual
            ).float()

            with torch.no_grad():
                visual_feature = visual_projection(
                    raw_tensor.to(device)
                ).cpu()

            # ------------------------------------------------
            # Noise augmentation
            # ------------------------------------------------

            visual_feature += (
                torch.randn_like(visual_feature)
                * noise_std
            )

            # ------------------------------------------------
            # Physical action
            #
            # 7D:
            # [x, y, z, roll, pitch, yaw, gripper]
            #
            # target objectのpositionを基準とした物理action。
            # ------------------------------------------------

            target_position = positions[color].copy()

            x, y, z = target_position

            if action_name == "pick_up":

                action = np.array(
                    [
                        x,
                        y,
                        z + 0.12,
                        0.0,
                        0.0,
                        0.0,
                        1.0,
                    ],
                    dtype=np.float32
                )

            elif action_name == "push_left":

                action = np.array(
                    [
                        x - 0.20,
                        y,
                        z,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                    ],
                    dtype=np.float32
                )

            elif action_name == "push_right":

                action = np.array(
                    [
                        x + 0.20,
                        y,
                        z,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                    ],
                    dtype=np.float32
                )

            elif action_name == "place_down":

                action = np.array(
                    [
                        x,
                        y,
                        max(0.02, z - 0.15),
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                    ],
                    dtype=np.float32
                )

            else:
                raise RuntimeError(
                    f"Unknown action: {action_name}"
                )

            token_ids = [
                vocab.get(
                    token,
                    vocab[UNK_TOKEN]
                )
                for token in tokenize(instruction)
            ]

            self.samples.append(
                {
                    "instruction": instruction,
                    "tokens": torch.tensor(
                        token_ids,
                        dtype=torch.long
                    ),
                    "visual": visual_feature.float(),
                    "raw_visual": torch.from_numpy(
                        raw_visual
                    ).float(),
                    "action": torch.from_numpy(
                        action
                    ).float(),
                    "color": color,
                    "action_name": action_name,
                    "positions": positions,
                }
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


# ------------------------------------------------------------
# 6. Collate Function
# ------------------------------------------------------------

def ant_collate_fn(batch):

    token_sequences = [
        item["tokens"]
        for item in batch
    ]

    lengths = torch.tensor(
        [len(x) for x in token_sequences],
        dtype=torch.long
    )

    max_len = int(lengths.max())

    padded_tokens = torch.full(
        (
            len(batch),
            max_len
        ),
        fill_value=vocab[PAD_TOKEN],
        dtype=torch.long
    )

    for i, tokens in enumerate(token_sequences):
        padded_tokens[
            i,
            :len(tokens)
        ] = tokens

    visual = torch.stack(
        [item["visual"] for item in batch]
    )

    raw_visual = torch.stack(
        [item["raw_visual"] for item in batch]
    )

    actions = torch.stack(
        [item["action"] for item in batch]
    )

    instructions = [
        item["instruction"]
        for item in batch
    ]

    return {
        "tokens": padded_tokens,
        "lengths": lengths,
        "visual": visual,
        "raw_visual": raw_visual,
        "action": actions,
        "instruction": instructions,
    }


# ------------------------------------------------------------
# 7. Dataset / DataLoader
# ------------------------------------------------------------

dataset = AntPhysicsDataset(
    num_samples=NUM_SAMPLES,
    noise_std=NOISE_STD,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=ant_collate_fn,
    pin_memory=torch.cuda.is_available()
)

print("\nDataset size:", len(dataset))


# ------------------------------------------------------------
# 8. Verify Position Randomization
# ------------------------------------------------------------

same_instruction_samples = [
    s for s in dataset.samples
    if s["instruction"] == "pick up the red cube"
]

print(
    "\nPosition-randomization check:"
)

for sample in same_instruction_samples[:3]:

    print(
        sample["instruction"],
        "-> red xyz =",
        sample["positions"]["red"]
    )


# ------------------------------------------------------------
# 9. Visual Encoder
# ------------------------------------------------------------

class VisualEncoder(nn.Module):

    def __init__(
        self,
        input_dim=512,
        latent_dim=128
    ):
        super().__init__()

        # Shared visual trunk
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, 384),
            nn.LayerNorm(384),
            nn.GELU(),

            nn.Linear(384, 256),
            nn.LayerNorm(256),
            nn.GELU(),
        )

        # Independent projection heads
        self.semantic_head = nn.Sequential(
            nn.Linear(256, 192),
            nn.GELU(),
            nn.Linear(192, latent_dim),
            nn.Tanh()
        )

        self.object_head = nn.Sequential(
            nn.Linear(256, 192),
            nn.GELU(),
            nn.Linear(192, latent_dim),
            nn.Tanh()
        )

        self.state_head = nn.Sequential(
            nn.Linear(256, 192),
            nn.GELU(),
            nn.Linear(192, latent_dim),
            nn.Tanh()
        )

    def forward(self, visual):

        h = self.trunk(visual)

        z_sem_V = self.semantic_head(h)
        z_obj_V = self.object_head(h)
        z_state_V = self.state_head(h)

        return (
            z_sem_V,
            z_obj_V,
            z_state_V
        )


# ------------------------------------------------------------
# 10. Language Encoder
# ------------------------------------------------------------

class LanguageEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=96,
        hidden_dim=128,
        latent_dim=128
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=vocab[PAD_TOKEN]
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # Bidirectional GRU => hidden_dim * 2
        self.semantic_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 192),
            nn.GELU(),
            nn.Linear(192, latent_dim),
            nn.Tanh()
        )

        self.object_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 192),
            nn.GELU(),
            nn.Linear(192, latent_dim),
            nn.Tanh()
        )

    def forward(
        self,
        tokens,
        lengths
    ):

        embedded = self.embedding(tokens)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden = self.gru(packed)

        # hidden:
        # [2, batch, hidden]
        forward_hidden = hidden[0]
        backward_hidden = hidden[1]

        h = torch.cat(
            [
                forward_hidden,
                backward_hidden
            ],
            dim=1
        )

        z_sem_L = self.semantic_head(h)
        z_obj_L = self.object_head(h)

        # IMPORTANT:
        # State latent is intentionally NOT produced.
        return (
            z_sem_L,
            z_obj_L
        )


# ------------------------------------------------------------
# 11. Action Head
# ------------------------------------------------------------

class ActionHead(nn.Module):

    def __init__(
        self,
        latent_dim=128,
        output_dim=7
    ):
        super().__init__()

        input_dim = latent_dim * 3

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 384),
            nn.LayerNorm(384),
            nn.GELU(),

            nn.Linear(384, 256),
            nn.GELU(),

            nn.Linear(256, 128),
            nn.GELU(),

            nn.Linear(128, output_dim)
        )

    def forward(
        self,
        z_sem_V,
        z_obj_V,
        z_state_V
    ):

        h = torch.cat(
            [
                z_sem_V,
                z_obj_V,
                z_state_V
            ],
            dim=1
        )

        return self.mlp(h)


# ------------------------------------------------------------
# 12. Complete AntEncoder Model
# ------------------------------------------------------------

class AntEncoder(nn.Module):

    def __init__(
        self,
        vocab_size
    ):
        super().__init__()

        self.visual_encoder = VisualEncoder(
            input_dim=VISUAL_DIM,
            latent_dim=LATENT_DIM
        )

        self.language_encoder = LanguageEncoder(
            vocab_size=vocab_size,
            embed_dim=EMBED_DIM,
            hidden_dim=GRU_HIDDEN,
            latent_dim=LATENT_DIM
        )

        self.action_head = ActionHead(
            latent_dim=LATENT_DIM,
            output_dim=7
        )

    def forward(
        self,
        visual,
        tokens,
        lengths
    ):

        # -----------------------------
        # Teacher / Vision
        # -----------------------------

        (
            z_sem_V,
            z_obj_V,
            z_state_V
        ) = self.visual_encoder(visual)

        # -----------------------------
        # Student / Language
        #
        # State is NEVER generated.
        # -----------------------------

        (
            z_sem_L,
            z_obj_L
        ) = self.language_encoder(
            tokens,
            lengths
        )

        # -----------------------------
        # Physical Action Reconstruction
        # -----------------------------

        action_pred = self.action_head(
            z_sem_V,
            z_obj_V,
            z_state_V
        )

        return {
            "z_sem_V": z_sem_V,
            "z_obj_V": z_obj_V,
            "z_state_V": z_state_V,
            "z_sem_L": z_sem_L,
            "z_obj_L": z_obj_L,
            "action_pred": action_pred,
        }


# ------------------------------------------------------------
# 13. AntencoderLoss
# ------------------------------------------------------------

class AntencoderLoss(nn.Module):

    def __init__(
        self,
        lambda_action=1.0,
        lambda_sem=1.0,
        lambda_obj=1.0
    ):
        super().__init__()

        self.lambda_action = lambda_action
        self.lambda_sem = lambda_sem
        self.lambda_obj = lambda_obj

    def forward(
        self,
        outputs,
        action_target
    ):

        z_sem_V = outputs["z_sem_V"]
        z_obj_V = outputs["z_obj_V"]

        z_sem_L = outputs["z_sem_L"]
        z_obj_L = outputs["z_obj_L"]

        action_pred = outputs["action_pred"]

        # ----------------------------------------------------
        # Language -> Visual Teacher Distillation
        #
        # .detach() is critical:
        # Language Student cannot modify Visual Teacher.
        # ----------------------------------------------------

        L_sem = F.mse_loss(
            z_sem_L,
            z_sem_V.detach()
        )

        L_obj = F.mse_loss(
            z_obj_L,
            z_obj_V.detach()
        )

        # ----------------------------------------------------
        # Physical action reconstruction
        # ----------------------------------------------------

        L_action = F.mse_loss(
            action_pred,
            action_target
        )

        # ----------------------------------------------------
        # Total loss
        # ----------------------------------------------------

        total_loss = (
            self.lambda_action * L_action
            + self.lambda_sem * L_sem
            + self.lambda_obj * L_obj
        )

        return {
            "total": total_loss,
            "action": L_action,
            "semantic": L_sem,
            "object": L_obj,
        }


# ------------------------------------------------------------
# 14. Instantiate
# ------------------------------------------------------------

model = AntEncoder(
    vocab_size=len(vocab)
).to(device)

criterion = AntencoderLoss(
    lambda_action=1.0,
    lambda_sem=1.0,
    lambda_obj=1.0
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

print("\nModel parameters:",
      sum(p.numel() for p in model.parameters()))


# ------------------------------------------------------------
# 15. Forward-pass sanity check
# ------------------------------------------------------------

batch = next(iter(loader))

visual = batch["visual"].to(device)
tokens = batch["tokens"].to(device)
lengths = batch["lengths"]
actions = batch["action"].to(device)

with torch.no_grad():

    test_outputs = model(
        visual,
        tokens,
        lengths
    )

print("\nShape check:")
print("visual      :", visual.shape)
print("tokens      :", tokens.shape)
print("z_sem_V     :", test_outputs["z_sem_V"].shape)
print("z_obj_V     :", test_outputs["z_obj_V"].shape)
print("z_state_V   :", test_outputs["z_state_V"].shape)
print("z_sem_L     :", test_outputs["z_sem_L"].shape)
print("z_obj_L     :", test_outputs["z_obj_L"].shape)
print("action_pred :", test_outputs["action_pred"].shape)

assert test_outputs["z_sem_V"].shape[-1] == 128
assert test_outputs["z_obj_V"].shape[-1] == 128
assert test_outputs["z_state_V"].shape[-1] == 128
assert test_outputs["z_sem_L"].shape[-1] == 128
assert test_outputs["z_obj_L"].shape[-1] == 128
assert test_outputs["action_pred"].shape[-1] == 7


# ------------------------------------------------------------
# 16. Training Loop
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

history = []

for epoch in range(1, EPOCHS + 1):

    model.train()

    running_total = 0.0
    running_action = 0.0
    running_sem = 0.0
    running_obj = 0.0

    num_batches = 0

    for batch_idx, batch in enumerate(loader):

        visual = batch["visual"].to(
            device,
            non_blocking=True
        )

        tokens = batch["tokens"].to(
            device,
            non_blocking=True
        )

        lengths = batch["lengths"]

        action_target = batch["action"].to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        outputs = model(
            visual,
            tokens,
            lengths
        )

        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------

        losses = criterion(
            outputs,
            action_target
        )

        loss = losses["total"]

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        # Gradient clipping for stable small-data training
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        running_total += losses["total"].item()
        running_action += losses["action"].item()
        running_sem += losses["semantic"].item()
        running_obj += losses["object"].item()

        num_batches += 1

    avg_total = running_total / num_batches
    avg_action = running_action / num_batches
    avg_sem = running_sem / num_batches
    avg_obj = running_obj / num_batches

    history.append(
        {
            "epoch": epoch,
            "total": avg_total,
            "action": avg_action,
            "semantic": avg_sem,
            "object": avg_obj,
        }
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Total={avg_total:.6f} | "
        f"Action={avg_action:.6f} | "
        f"Sem={avg_sem:.6f} | "
        f"Obj={avg_obj:.6f}"
    )


# ------------------------------------------------------------
# 17. Final Loss Reduction Report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOSS HISTORY")
print("=" * 70)

for h in history:
    print(
        f"Epoch {h['epoch']:02d}: "
        f"Total={h['total']:.6f}, "
        f"Action={h['action']:.6f}, "
        f"Sem={h['semantic']:.6f}, "
        f"Obj={h['object']:.6f}"
    )

if len(history) >= 2:

    initial_loss = history[0]["total"]
    final_loss = history[-1]["total"]

    reduction = (
        (initial_loss - final_loss)
        / max(abs(initial_loss), 1e-8)
        * 100.0
    )

    print(
        f"\nTotal loss reduction: "
        f"{reduction:.2f}%"
    )


# ------------------------------------------------------------
# 18. Learned Latent Sanity Check
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    batch = next(iter(loader))

    visual = batch["visual"].to(device)
    tokens = batch["tokens"].to(device)
    lengths = batch["lengths"]
    actions = batch["action"].to(device)

    outputs = model(
        visual,
        tokens,
        lengths
    )

    final_losses = criterion(
        outputs,
        actions
    )

print("\n" + "=" * 70)
print("FINAL SANITY CHECK")
print("=" * 70)

print(
    "z_sem_V:",
    tuple(outputs["z_sem_V"].shape)
)

print(
    "z_obj_V:",
    tuple(outputs["z_obj_V"].shape)
)

print(
    "z_state_V:",
    tuple(outputs["z_state_V"].shape)
)

print(
    "z_sem_L:",
    tuple(outputs["z_sem_L"].shape)
)

print(
    "z_obj_L:",
    tuple(outputs["z_obj_L"].shape)
)

print(
    "action:",
    tuple(outputs["action_pred"].shape)
)

print(
    "\nFinal total loss:",
    f"{final_losses['total'].item():.6f}"
)

print(
    "Final action loss:",
    f"{final_losses['action'].item():.6f}"
)

print(
    "Final semantic distillation loss:",
    f"{final_losses['semantic'].item():.6f}"
)

print(
    "Final object distillation loss:",
    f"{final_losses['object'].item():.6f}"
)

print("\nAntEncoder training finished successfully.")


AntEncoder Factorized VLA
device: cpu
Vocabulary size: 14
Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'blue': 2, 'cube': 3, 'down': 4, 'green': 5, 'left': 6, 'pick': 7, 'place': 8, 'push': 9, 'red': 10, 'right': 11, 'the': 12, 'up': 13}

Dataset size: 2400

Position-randomization check:
pick up the red cube -> red xyz = [-0.6001836  -0.98527545  0.7911859 ]
pick up the red cube -> red xyz = [-0.6588141   0.85024023  0.5894399 ]
pick up the red cube -> red xyz = [ 0.5579927 -0.7308956  0.5453467]

Model parameters: 1122951

Shape check:
visual      : torch.Size([128, 512])
tokens      : torch.Size([128, 5])
z_sem_V     : torch.Size([128, 128])
z_obj_V     : torch.Size([128, 128])
z_state_V   : torch.Size([128, 128])
z_sem_L     : torch.Size([128, 128])
z_obj_L     : torch.Size([128, 128])
action_pred : torch.Size([128, 7])

TRAINING
Epoch 01/10 | Total=0.246172 | Action=0.127602 | Sem=0.059814 | Obj=0.058757
Epoch 02/10 | Total=0.290763 | Action=0.113011 | Sem=0.087084 | Obj=0.090669
Epoch 03/

In [ ]:
# ============================================================
# AntEncoder / AntVLA
# Experiment 2: Factorized Physical Teacher + Language Student
#
# Previous Experiment 1:
#   Vision/State + Action
#          ↓
#   Physical Teacher
#          ↓
#        Latent
#          ↓
#   Language Student
#
# This Experiment:
#
#   Physical Teacher
#       ├── Semantic
#       ├── Object
#       └── State
#
#   Language Student
#       ├── Semantic
#       └── Object
#       State is NEVER predicted from language.
#
#   Final VLA:
#       Language Semantic/Object
#                +
#          Visual State
#                ↓
#             Action
#
# ============================================================

import re
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("AntEncoder Experiment 2")
print("Factorized Physical Teacher + Language Student")
print("=" * 72)
print("Device:", device)


# ============================================================
# 1. Configuration
# ============================================================

NUM_SAMPLES = 2400
BATCH_SIZE = 128

TEACHER_EPOCHS = 8
STUDENT_EPOCHS = 10

LR_TEACHER = 3e-4
LR_STUDENT = 3e-4

VISUAL_RAW_DIM = 9
VISUAL_DIM = 512

LATENT_DIM = 128

EMBED_DIM = 96
GRU_HIDDEN = 128

NOISE_STD = 0.03

LAMBDA_ACTION = 1.0
LAMBDA_SEM = 1.0
LAMBDA_OBJ = 1.0

LAMBDA_SEM_AUX = 0.5
LAMBDA_OBJ_AUX = 0.5
LAMBDA_STATE_AUX = 0.5

COLORS = [
    "red",
    "blue",
    "green"
]

ACTION_NAMES = [
    "pick_up",
    "push_left",
    "push_right",
    "place_down"
]

ACTION_TO_ID = {
    name: i
    for i, name in enumerate(ACTION_NAMES)
}

COLOR_TO_ID = {
    color: i
    for i, color in enumerate(COLORS)
}


# ============================================================
# 2. Language Commands
# ============================================================

INSTRUCTIONS = {

    ("pick_up", "red"):
        "pick up the red cube",

    ("pick_up", "blue"):
        "pick up the blue cube",

    ("pick_up", "green"):
        "pick up the green cube",

    ("push_left", "red"):
        "push red cube left",

    ("push_left", "blue"):
        "push blue cube left",

    ("push_left", "green"):
        "push green cube left",

    ("push_right", "red"):
        "push red cube right",

    ("push_right", "blue"):
        "push blue cube right",

    ("push_right", "green"):
        "push green cube right",

    ("place_down", "red"):
        "place the red cube down",

    ("place_down", "blue"):
        "place the blue cube down",

    ("place_down", "green"):
        "place the green cube down",
}


def tokenize(text):
    return re.findall(
        r"[a-z]+",
        text.lower()
    )


# ============================================================
# 3. Vocabulary
# ============================================================

counter = {}

for text in INSTRUCTIONS.values():

    for token in tokenize(text):

        counter[token] = (
            counter.get(token, 0) + 1
        )


PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for token in sorted(counter.keys()):

    vocab[token] = len(vocab)

print("\nVocabulary size:", len(vocab))
print("Vocabulary:", vocab)


# ============================================================
# 4. Fixed Physical 9D -> 512D Visual Projection
# ============================================================

class PhysicalVisualProjection(nn.Module):

    def __init__(
        self,
        in_dim=9,
        out_dim=512,
        seed=1234
    ):

        super().__init__()

        generator = torch.Generator()
        generator.manual_seed(seed)

        weight = torch.randn(
            out_dim,
            in_dim,
            generator=generator
        ) * math.sqrt(
            2.0 / in_dim
        )

        bias = torch.randn(
            out_dim,
            generator=generator
        ) * 0.02

        self.register_buffer(
            "weight",
            weight
        )

        self.register_buffer(
            "bias",
            bias
        )

    def forward(self, x):

        return F.linear(
            x,
            self.weight,
            self.bias
        )


visual_projection = PhysicalVisualProjection(
    VISUAL_RAW_DIM,
    VISUAL_DIM
).to(device)

visual_projection.eval()


# ============================================================
# 5. Dataset
# ============================================================

class AntPhysicsDataset(Dataset):

    def __init__(
        self,
        num_samples=2400,
        noise_std=0.03,
        seed=42
    ):

        super().__init__()

        self.samples = []

        rng = np.random.default_rng(seed)

        for i in range(num_samples):

            # ------------------------------------------------
            # IMPORTANT:
            # Every sample receives completely new positions.
            # Same language != same state.
            # ------------------------------------------------

            positions = {

                "red": rng.uniform(
                    [-1.0, -1.0, 0.05],
                    [ 1.0,  1.0, 1.0],
                    3
                ).astype(np.float32),

                "blue": rng.uniform(
                    [-1.0, -1.0, 0.05],
                    [ 1.0,  1.0, 1.0],
                    3
                ).astype(np.float32),

                "green": rng.uniform(
                    [-1.0, -1.0, 0.05],
                    [ 1.0,  1.0, 1.0],
                    3
                ).astype(np.float32)
            }

            action_name = random.choice(
                ACTION_NAMES
            )

            color = random.choice(
                COLORS
            )

            instruction = INSTRUCTIONS[
                (action_name, color)
            ]

            target_xyz = positions[
                color
            ].copy()

            x, y, z = target_xyz

            # ------------------------------------------------
            # Physical Action
            # ------------------------------------------------

            if action_name == "pick_up":

                action = np.array(
                    [
                        x,
                        y,
                        z + 0.12,
                        0.0,
                        0.0,
                        0.0,
                        1.0
                    ],
                    dtype=np.float32
                )

            elif action_name == "push_left":

                action = np.array(
                    [
                        x - 0.20,
                        y,
                        z,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ],
                    dtype=np.float32
                )

            elif action_name == "push_right":

                action = np.array(
                    [
                        x + 0.20,
                        y,
                        z,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ],
                    dtype=np.float32
                )

            elif action_name == "place_down":

                action = np.array(
                    [
                        x,
                        y,
                        max(
                            0.02,
                            z - 0.15
                        ),
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ],
                    dtype=np.float32
                )

            # ------------------------------------------------
            # 9D Scene
            #
            # [red_xyz | blue_xyz | green_xyz]
            # ------------------------------------------------

            raw_visual = np.concatenate(
                [
                    positions["red"],
                    positions["blue"],
                    positions["green"]
                ]
            ).astype(np.float32)

            raw_tensor = torch.from_numpy(
                raw_visual
            ).float()

            # Fixed physical projection
            with torch.no_grad():

                visual_feature = visual_projection(
                    raw_tensor.to(device)
                ).cpu()

            # Noise
            visual_feature += (
                torch.randn_like(
                    visual_feature
                ) * noise_std
            )

            tokens = [
                vocab.get(
                    token,
                    vocab[UNK_TOKEN]
                )
                for token in tokenize(
                    instruction
                )
            ]

            self.samples.append(
                {
                    "visual":
                        visual_feature.float(),

                    "raw_visual":
                        torch.from_numpy(
                            raw_visual
                        ).float(),

                    "action":
                        torch.from_numpy(
                            action
                        ).float(),

                    "tokens":
                        torch.tensor(
                            tokens,
                            dtype=torch.long
                        ),

                    "instruction":
                        instruction,

                    "action_id":
                        ACTION_TO_ID[
                            action_name
                        ],

                    "color_id":
                        COLOR_TO_ID[
                            color
                        ],

                    "state":
                        torch.from_numpy(
                            target_xyz
                        ).float(),

                    "action_name":
                        action_name,

                    "color":
                        color,

                    "positions":
                        positions
                }
            )

    def __len__(self):

        return len(
            self.samples
        )

    def __getitem__(self, idx):

        return self.samples[idx]


# ============================================================
# 6. Collate
# ============================================================

def collate_fn(batch):

    lengths = torch.tensor(
        [
            len(x["tokens"])
            for x in batch
        ],
        dtype=torch.long
    )

    max_len = int(
        lengths.max()
    )

    tokens = torch.full(
        (
            len(batch),
            max_len
        ),
        vocab[PAD_TOKEN],
        dtype=torch.long
    )

    for i, item in enumerate(batch):

        n = len(
            item["tokens"]
        )

        tokens[
            i,
            :n
        ] = item["tokens"]

    return {

        "visual":
            torch.stack(
                [
                    x["visual"]
                    for x in batch
                ]
            ),

        "raw_visual":
            torch.stack(
                [
                    x["raw_visual"]
                    for x in batch
                ]
            ),

        "tokens":
            tokens,

        "lengths":
            lengths,

        "action":
            torch.stack(
                [
                    x["action"]
                    for x in batch
                ]
            ),

        "state":
            torch.stack(
                [
                    x["state"]
                    for x in batch
                ]
            ),

        "action_id":
            torch.tensor(
                [
                    x["action_id"]
                    for x in batch
                ],
                dtype=torch.long
            ),

        "color_id":
            torch.tensor(
                [
                    x["color_id"]
                    for x in batch
                ],
                dtype=torch.long
            ),

        "instruction":
            [
                x["instruction"]
                for x in batch
            ]
    }


# ============================================================
# 7. Dataset
# ============================================================

dataset = AntPhysicsDataset(
    num_samples=NUM_SAMPLES,
    noise_std=NOISE_STD,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

print("\nDataset size:", len(dataset))


# ============================================================
# 8. Randomization Check
# ============================================================

print("\nPosition randomization check:")

count = 0

for sample in dataset.samples:

    if sample["instruction"] == \
       "pick up the red cube":

        print(
            sample["instruction"],
            "->",
            sample["positions"]["red"]
        )

        count += 1

        if count >= 3:
            break


# ============================================================
# 9. Factorized Physical Teacher
#
# IMPORTANT:
#
# State:
#   visual only
#
# Semantic/Object:
#   visual + action
#
# This prevents language-predictable semantics from being
# mixed with the physical state, while keeping the Teacher
# grounded in physical action.
# ============================================================

class FactorizedPhysicalTeacher(
    nn.Module
):

    def __init__(
        self,
        visual_dim=512,
        action_dim=7,
        latent_dim=128
    ):

        super().__init__()

        # ----------------------------------------------------
        # Visual trunk
        # ----------------------------------------------------

        self.visual_trunk = nn.Sequential(

            nn.Linear(
                visual_dim,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU(),

            nn.Linear(
                384,
                256
            ),

            nn.LayerNorm(256),

            nn.GELU()
        )

        # ----------------------------------------------------
        # Action encoder
        # ----------------------------------------------------

        self.action_encoder = nn.Sequential(

            nn.Linear(
                action_dim,
                128
            ),

            nn.GELU(),

            nn.Linear(
                128,
                128
            ),

            nn.GELU()
        )

        # ----------------------------------------------------
        # Semantic Teacher
        # ----------------------------------------------------

        self.semantic_head = nn.Sequential(

            nn.Linear(
                256 + 128,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

        # ----------------------------------------------------
        # Object Teacher
        # ----------------------------------------------------

        self.object_head = nn.Sequential(

            nn.Linear(
                256 + 128,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

        # ----------------------------------------------------
        # State Teacher
        #
        # NO ACTION INPUT.
        #
        # This is deliberate.
        # ----------------------------------------------------

        self.state_head = nn.Sequential(

            nn.Linear(
                256,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

        # ----------------------------------------------------
        # Action reconstruction
        # ----------------------------------------------------

        self.action_head = nn.Sequential(

            nn.Linear(
                latent_dim * 3,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU(),

            nn.Linear(
                384,
                256
            ),

            nn.GELU(),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Linear(
                128,
                7
            )
        )

        # ----------------------------------------------------
        # Auxiliary semantic classifier
        # ----------------------------------------------------

        self.semantic_classifier = nn.Linear(
            latent_dim,
            len(ACTION_NAMES)
        )

        # ----------------------------------------------------
        # Auxiliary object classifier
        # ----------------------------------------------------

        self.object_classifier = nn.Linear(
            latent_dim,
            len(COLORS)
        )

        # ----------------------------------------------------
        # Auxiliary state regression
        # ----------------------------------------------------

        self.state_decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                64
            ),

            nn.GELU(),

            nn.Linear(
                64,
                3
            )
        )

    def encode(
        self,
        visual,
        action
    ):

        visual_h = self.visual_trunk(
            visual
        )

        action_h = self.action_encoder(
            action
        )

        va_h = torch.cat(
            [
                visual_h,
                action_h
            ],
            dim=1
        )

        z_sem = self.semantic_head(
            va_h
        )

        z_obj = self.object_head(
            va_h
        )

        # State is visual-only
        z_state = self.state_head(
            visual_h
        )

        return (
            z_sem,
            z_obj,
            z_state
        )

    def reconstruct_action(
        self,
        z_sem,
        z_obj,
        z_state
    ):

        h = torch.cat(
            [
                z_sem,
                z_obj,
                z_state
            ],
            dim=1
        )

        return self.action_head(h)

    def forward(
        self,
        visual,
        action
    ):

        (
            z_sem,
            z_obj,
            z_state
        ) = self.encode(
            visual,
            action
        )

        action_pred = self.reconstruct_action(
            z_sem,
            z_obj,
            z_state
        )

        semantic_logits = \
            self.semantic_classifier(
                z_sem
            )

        object_logits = \
            self.object_classifier(
                z_obj
            )

        state_pred = \
            self.state_decoder(
                z_state
            )

        return {

            "z_sem":
                z_sem,

            "z_obj":
                z_obj,

            "z_state":
                z_state,

            "action_pred":
                action_pred,

            "semantic_logits":
                semantic_logits,

            "object_logits":
                object_logits,

            "state_pred":
                state_pred
        }


# ============================================================
# 10. Teacher Loss
# ============================================================

class TeacherLoss(
    nn.Module
):

    def __init__(self):

        super().__init__()

    def forward(
        self,
        outputs,
        action_target,
        action_id,
        color_id,
        state_target
    ):

        # Physical action reconstruction
        L_action = F.mse_loss(
            outputs["action_pred"],
            action_target
        )

        # Semantic identity
        L_sem_aux = F.cross_entropy(
            outputs["semantic_logits"],
            action_id
        )

        # Object identity
        L_obj_aux = F.cross_entropy(
            outputs["object_logits"],
            color_id
        )

        # State = xyz
        L_state_aux = F.mse_loss(
            outputs["state_pred"],
            state_target
        )

        total = (

            LAMBDA_ACTION * L_action

            + LAMBDA_SEM_AUX
            * L_sem_aux

            + LAMBDA_OBJ_AUX
            * L_obj_aux

            + LAMBDA_STATE_AUX
            * L_state_aux
        )

        return {

            "total":
                total,

            "action":
                L_action,

            "semantic_aux":
                L_sem_aux,

            "object_aux":
                L_obj_aux,

            "state_aux":
                L_state_aux
        }


# ============================================================
# 11. Train Physical Teacher
# ============================================================

teacher = FactorizedPhysicalTeacher().to(
    device
)

teacher_loss_fn = TeacherLoss()

teacher_optimizer = torch.optim.AdamW(
    teacher.parameters(),
    lr=LR_TEACHER,
    weight_decay=1e-4
)

print(
    "\nTeacher parameters:",
    sum(
        p.numel()
        for p in teacher.parameters()
    )
)

print("\n" + "=" * 72)
print("PHASE 1: PHYSICAL TEACHER TRAINING")
print("=" * 72)

teacher_history = []

for epoch in range(
    1,
    TEACHER_EPOCHS + 1
):

    teacher.train()

    total_sum = 0.0
    action_sum = 0.0
    sem_sum = 0.0
    obj_sum = 0.0
    state_sum = 0.0

    n_batches = 0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        action = batch[
            "action"
        ].to(device)

        state = batch[
            "state"
        ].to(device)

        action_id = batch[
            "action_id"
        ].to(device)

        color_id = batch[
            "color_id"
        ].to(device)

        outputs = teacher(
            visual,
            action
        )

        losses = teacher_loss_fn(
            outputs,
            action,
            action_id,
            color_id,
            state
        )

        teacher_optimizer.zero_grad(
            set_to_none=True
        )

        losses["total"].backward()

        torch.nn.utils.clip_grad_norm_(
            teacher.parameters(),
            1.0
        )

        teacher_optimizer.step()

        total_sum += losses[
            "total"
        ].item()

        action_sum += losses[
            "action"
        ].item()

        sem_sum += losses[
            "semantic_aux"
        ].item()

        obj_sum += losses[
            "object_aux"
        ].item()

        state_sum += losses[
            "state_aux"
        ].item()

        n_batches += 1

    avg_total = total_sum / n_batches
    avg_action = action_sum / n_batches
    avg_sem = sem_sum / n_batches
    avg_obj = obj_sum / n_batches
    avg_state = state_sum / n_batches

    teacher_history.append(
        (
            avg_total,
            avg_action,
            avg_sem,
            avg_obj,
            avg_state
        )
    )

    print(
        f"Teacher Epoch "
        f"{epoch:02d}/{TEACHER_EPOCHS} | "
        f"Total={avg_total:.6f} | "
        f"Action={avg_action:.6f} | "
        f"SemAux={avg_sem:.6f} | "
        f"ObjAux={avg_obj:.6f} | "
        f"StateAux={avg_state:.6f}"
    )


# ============================================================
# 12. FREEZE TEACHER
# ============================================================

teacher.eval()

for parameter in teacher.parameters():

    parameter.requires_grad = False


print(
    "\nTeacher is now FROZEN."
)

print(
    "Trainable teacher parameters:",
    sum(
        p.numel()
        for p in teacher.parameters()
        if p.requires_grad
    )
)


# ============================================================
# 13. Language Student
#
# Language produces:
#   z_sem_L
#   z_obj_L
#
# There is deliberately NO z_state_L.
# ============================================================

class LanguageStudent(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim=96,
        hidden_dim=128,
        latent_dim=128
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=vocab[PAD_TOKEN]
        )

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.semantic_head = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

        self.object_head = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

    def forward(
        self,
        tokens,
        lengths
    ):

        embedded = self.embedding(
            tokens
        )

        packed = \
            nn.utils.rnn.pack_padded_sequence(
                embedded,
                lengths.cpu(),
                batch_first=True,
                enforce_sorted=False
            )

        _, hidden = self.gru(
            packed
        )

        forward_hidden = hidden[0]
        backward_hidden = hidden[1]

        h = torch.cat(
            [
                forward_hidden,
                backward_hidden
            ],
            dim=1
        )

        z_sem_L = self.semantic_head(h)

        z_obj_L = self.object_head(h)

        # IMPORTANT:
        # No State latent.
        return (
            z_sem_L,
            z_obj_L
        )


# ============================================================
# 14. Student
# ============================================================

student = LanguageStudent(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=GRU_HIDDEN,
    latent_dim=LATENT_DIM
).to(device)

student_optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=LR_STUDENT,
    weight_decay=1e-4
)

print(
    "\nStudent parameters:",
    sum(
        p.numel()
        for p in student.parameters()
    )
)


# ============================================================
# 15. Student Distillation Loss
# ============================================================

def student_distillation_loss(
    z_sem_L,
    z_obj_L,
    z_sem_T,
    z_obj_T
):

    L_sem = F.mse_loss(
        z_sem_L,
        z_sem_T.detach()
    )

    L_obj = F.mse_loss(
        z_obj_L,
        z_obj_T.detach()
    )

    total = (
        L_sem
        + L_obj
    )

    return (
        total,
        L_sem,
        L_obj
    )


# ============================================================
# 16. Train Language Student
# ============================================================

print("\n" + "=" * 72)
print("PHASE 2: LANGUAGE STUDENT DISTILLATION")
print("=" * 72)

student_history = []

for epoch in range(
    1,
    STUDENT_EPOCHS + 1
):

    student.train()

    total_sum = 0.0
    sem_sum = 0.0
    obj_sum = 0.0

    n_batches = 0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        action = batch[
            "action"
        ].to(device)

        tokens = batch[
            "tokens"
        ].to(device)

        lengths = batch[
            "lengths"
        ]

        # ----------------------------------------------------
        # Frozen Physical Teacher
        # ----------------------------------------------------

        with torch.no_grad():

            teacher_outputs = teacher(
                visual,
                action
            )

            z_sem_T = teacher_outputs[
                "z_sem"
            ]

            z_obj_T = teacher_outputs[
                "z_obj"
            ]

        # ----------------------------------------------------
        # Language Student
        # ----------------------------------------------------

        (
            z_sem_L,
            z_obj_L
        ) = student(
            tokens,
            lengths
        )

        loss, L_sem, L_obj = \
            student_distillation_loss(
                z_sem_L,
                z_obj_L,
                z_sem_T,
                z_obj_T
            )

        student_optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            1.0
        )

        student_optimizer.step()

        total_sum += loss.item()
        sem_sum += L_sem.item()
        obj_sum += L_obj.item()

        n_batches += 1

    avg_total = total_sum / n_batches
    avg_sem = sem_sum / n_batches
    avg_obj = obj_sum / n_batches

    student_history.append(
        (
            avg_total,
            avg_sem,
            avg_obj
        )
    )

    print(
        f"Student Epoch "
        f"{epoch:02d}/{STUDENT_EPOCHS} | "
        f"Total={avg_total:.6f} | "
        f"Sem={avg_sem:.6f} | "
        f"Obj={avg_obj:.6f}"
    )


# ============================================================
# 17. Final VLA Fusion
#
# Language:
#   z_sem_L
#   z_obj_L
#
# Vision:
#   z_state_V
#
# Final:
#   [Language Semantic, Language Object, Visual State]
#       ↓
#     Action
#
# This is the actual VLA path.
# ============================================================

class FactorizedVLAActionHead(
    nn.Module
):

    def __init__(
        self,
        latent_dim=128
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                latent_dim * 3,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU(),

            nn.Linear(
                384,
                256
            ),

            nn.GELU(),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Linear(
                128,
                7
            )
        )

    def forward(
        self,
        z_sem_L,
        z_obj_L,
        z_state_V
    ):

        h = torch.cat(
            [
                z_sem_L,
                z_obj_L,
                z_state_V
            ],
            dim=1
        )

        return self.net(h)


vla_action_head = \
    FactorizedVLAActionHead(
        LATENT_DIM
    ).to(device)


# ============================================================
# 18. Train ONLY Final VLA Fusion
#
# Teacher + Student are frozen.
# Only fusion/action head learns.
# ============================================================

for parameter in teacher.parameters():

    parameter.requires_grad = False

for parameter in student.parameters():

    parameter.requires_grad = False


vla_optimizer = torch.optim.AdamW(
    vla_action_head.parameters(),
    lr=LR_STUDENT,
    weight_decay=1e-4
)

print(
    "\nVLA Fusion parameters:",
    sum(
        p.numel()
        for p in vla_action_head.parameters()
    )
)

print("\n" + "=" * 72)
print("PHASE 3: FACTORIZED VLA ACTION FUSION")
print("=" * 72)

vla_history = []

for epoch in range(1, 6):

    vla_action_head.train()

    loss_sum = 0.0
    n_batches = 0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        action = batch[
            "action"
        ].to(device)

        tokens = batch[
            "tokens"
        ].to(device)

        lengths = batch[
            "lengths"
        ]

        # ----------------------------------------------------
        # Frozen Teacher State
        # ----------------------------------------------------

        with torch.no_grad():

            teacher_outputs = teacher(
                visual,
                action
            )

            z_state_V = teacher_outputs[
                "z_state"
            ]

        # ----------------------------------------------------
        # Frozen Language Student
        # ----------------------------------------------------

        with torch.no_grad():

            (
                z_sem_L,
                z_obj_L
            ) = student(
                tokens,
                lengths
            )

        # ----------------------------------------------------
        # Language + Visual State -> Action
        # ----------------------------------------------------

        action_pred = vla_action_head(
            z_sem_L,
            z_obj_L,
            z_state_V
        )

        loss = F.mse_loss(
            action_pred,
            action
        )

        vla_optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            vla_action_head.parameters(),
            1.0
        )

        vla_optimizer.step()

        loss_sum += loss.item()
        n_batches += 1

    avg_loss = (
        loss_sum / n_batches
    )

    vla_history.append(
        avg_loss
    )

    print(
        f"VLA Epoch "
        f"{epoch:02d}/5 | "
        f"Action={avg_loss:.6f}"
    )


# ============================================================
# 19. Final Sanity Check
# ============================================================

teacher.eval()
student.eval()
vla_action_head.eval()

batch = next(iter(loader))

visual = batch[
    "visual"
].to(device)

tokens = batch[
    "tokens"
].to(device)

lengths = batch[
    "lengths"
]

action = batch[
    "action"
].to(device)

with torch.no_grad():

    teacher_outputs = teacher(
        visual,
        action
    )

    (
        z_sem_L,
        z_obj_L
    ) = student(
        tokens,
        lengths
    )

    action_pred = vla_action_head(
        z_sem_L,
        z_obj_L,
        teacher_outputs[
            "z_state"
        ]
    )

    final_action_loss = F.mse_loss(
        action_pred,
        action
    )


# ============================================================
# 20. Shape / Architecture Checks
# ============================================================

print("\n" + "=" * 72)
print("FINAL ARCHITECTURE CHECK")
print("=" * 72)

print(
    "Visual input       :",
    tuple(visual.shape)
)

print(
    "Teacher z_sem      :",
    tuple(
        teacher_outputs[
            "z_sem"
        ].shape
    )
)

print(
    "Teacher z_obj      :",
    tuple(
        teacher_outputs[
            "z_obj"
        ].shape
    )
)

print(
    "Teacher z_state    :",
    tuple(
        teacher_outputs[
            "z_state"
        ].shape
    )
)

print(
    "Student z_sem      :",
    tuple(z_sem_L.shape)
)

print(
    "Student z_obj      :",
    tuple(z_obj_L.shape)
)

print(
    "Student State      : NONE"
)

print(
    "Final Action        :",
    tuple(action_pred.shape)
)


assert z_sem_L.shape[-1] == 128
assert z_obj_L.shape[-1] == 128

assert teacher_outputs[
    "z_state"
].shape[-1] == 128

assert action_pred.shape[-1] == 7


# ============================================================
# 21. Frozen Check
# ============================================================

teacher_trainable = sum(
    p.numel()
    for p in teacher.parameters()
    if p.requires_grad
)

student_trainable = sum(
    p.numel()
    for p in student.parameters()
    if p.requires_grad
)

vla_trainable = sum(
    p.numel()
    for p in vla_action_head.parameters()
    if p.requires_grad
)

print("\nFrozen check:")
print(
    "Teacher trainable params :",
    teacher_trainable
)

print(
    "Student trainable params :",
    student_trainable
)

print(
    "VLA trainable params     :",
    vla_trainable
)


# ============================================================
# 22. Student Alignment
# ============================================================

with torch.no_grad():

    sem_alignment = F.mse_loss(
        z_sem_L,
        teacher_outputs[
            "z_sem"
        ]
    ).item()

    obj_alignment = F.mse_loss(
        z_obj_L,
        teacher_outputs[
            "z_obj"
        ]
    ).item()


# ============================================================
# 23. Same Language / Different State Test
#
# This is the important experiment.
# ============================================================

print("\n" + "=" * 72)
print("SAME LANGUAGE / DIFFERENT STATE TEST")
print("=" * 72)

same_instruction = (
    "push red cube right"
)

samples = [
    x
    for x in dataset.samples
    if x["instruction"] == same_instruction
]

if len(samples) >= 2:

    s1 = samples[0]
    s2 = samples[1]

    token1 = s1["tokens"].unsqueeze(0).to(device)
    token2 = s2["tokens"].unsqueeze(0).to(device)

    length1 = torch.tensor(
        [len(s1["tokens"])],
        dtype=torch.long
    )

    length2 = torch.tensor(
        [len(s2["tokens"])],
        dtype=torch.long
    )

    visual1 = s1[
        "visual"
    ].unsqueeze(0).to(device)

    visual2 = s2[
        "visual"
    ].unsqueeze(0).to(device)

    action1 = s1[
        "action"
    ].unsqueeze(0).to(device)

    action2 = s2[
        "action"
    ].unsqueeze(0).to(device)

    with torch.no_grad():

        sem1, obj1 = student(
            token1,
            length1
        )

        sem2, obj2 = student(
            token2,
            length2
        )

        teacher1 = teacher(
            visual1,
            action1
        )

        teacher2 = teacher(
            visual2,
            action2
        )

        pred1 = vla_action_head(
            sem1,
            obj1,
            teacher1["z_state"]
        )

        pred2 = vla_action_head(
            sem2,
            obj2,
            teacher2["z_state"]
        )

    language_sem_difference = F.mse_loss(
        sem1,
        sem2
    ).item()

    language_obj_difference = F.mse_loss(
        obj1,
        obj2
    ).item()

    visual_state_difference = F.mse_loss(
        teacher1["z_state"],
        teacher2["z_state"]
    ).item()

    action_difference = F.mse_loss(
        pred1,
        pred2
    ).item()

    print(
        "\nInstruction:",
        same_instruction
    )

    print(
        "\nScene A red xyz:",
        s1["positions"]["red"]
    )

    print(
        "Scene B red xyz:",
        s2["positions"]["red"]
    )

    print(
        "\nLanguage Semantic difference:",
        f"{language_sem_difference:.8f}"
    )

    print(
        "Language Object difference:",
        f"{language_obj_difference:.8f}"
    )

    print(
        "Visual State difference:",
        f"{visual_state_difference:.8f}"
    )

    print(
        "Final Action difference:",
        f"{action_difference:.8f}"
    )

    print(
        "\nExpected:"
    )

    print(
        "  Language Semantic -> SMALL"
    )

    print(
        "  Language Object   -> SMALL"
    )

    print(
        "  Visual State      -> LARGE"
    )

    print(
        "  Action            -> LARGE"
    )


# ============================================================
# 24. Final Report
# ============================================================

print("\n" + "=" * 72)
print("FINAL REPORT")
print("=" * 72)

if len(teacher_history) >= 2:

    print(
        "\nTeacher total:",
        f"{teacher_history[0][0]:.6f}",
        "->",
        f"{teacher_history[-1][0]:.6f}"
    )

    print(
        "Teacher action:",
        f"{teacher_history[0][1]:.6f}",
        "->",
        f"{teacher_history[-1][1]:.6f}"
    )

if len(student_history) >= 2:

    print(
        "\nStudent distillation:",
        f"{student_history[0][0]:.6f}",
        "->",
        f"{student_history[-1][0]:.6f}"
    )

    print(
        "Student semantic:",
        f"{student_history[0][1]:.6f}",
        "->",
        f"{student_history[-1][1]:.6f}"
    )

    print(
        "Student object:",
        f"{student_history[0][2]:.6f}",
        "->",
        f"{student_history[-1][2]:.6f}"
    )

print(
    "\nFinal semantic alignment:",
    f"{sem_alignment:.6f}"
)

print(
    "Final object alignment:",
    f"{obj_alignment:.6f}"
)

print(
    "Final VLA action loss:",
    f"{final_action_loss.item():.6f}"
)

print("\n" + "=" * 72)
print("Experiment 2 finished.")
print("=" * 72)


AntEncoder Experiment 2
Factorized Physical Teacher + Language Student
Device: cpu

Vocabulary size: 14
Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'blue': 2, 'cube': 3, 'down': 4, 'green': 5, 'left': 6, 'pick': 7, 'place': 8, 'push': 9, 'red': 10, 'right': 11, 'the': 12, 'up': 13}

Dataset size: 2400

Position randomization check:
pick up the red cube -> [ 0.5479121  -0.12224312  0.865668  ]
pick up the red cube -> [0.41033074 0.56145805 0.48597   ]
pick up the red cube -> [ 0.10807229 -0.78284854  0.6886281 ]

Teacher parameters: 875985

PHASE 1: PHYSICAL TEACHER TRAINING
Teacher Epoch 01/8 | Total=1.488959 | Action=0.118082 | SemAux=1.385564 | ObjAux=1.099487 | StateAux=0.256704
Teacher Epoch 02/8 | Total=1.407899 | Action=0.095170 | SemAux=1.352365 | ObjAux=1.096707 | StateAux=0.176385
Teacher Epoch 03/8 | Total=1.267818 | Action=0.039813 | SemAux=1.198731 | ObjAux=1.089202 | StateAux=0.168078
Teacher Epoch 04/8 | Total=1.078761 | Action=0.012474 | SemAux=0.886095 | ObjAux=1.080234 | Stat

In [ ]:
# ============================================================
# AntEncoder / AntVLA
# Experiment 3
#
# Vision-Only Factorized Physical Teacher
# +
# Frozen Teacher
# +
# Language Semantic/Object Student
# +
# Visual State
# +
# State Shuffle Ablation
#
# ============================================================

import re
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("AntEncoder Experiment 3")
print("Vision-Only Factorized Teacher")
print("=" * 72)
print("Device:", device)


# ============================================================
# 1. Configuration
# ============================================================

NUM_SAMPLES = 2400
BATCH_SIZE = 128

TEACHER_EPOCHS = 100
STUDENT_EPOCHS = 10
VLA_EPOCHS = 8

LR_TEACHER = 3e-4
LR_STUDENT = 3e-4
LR_VLA = 3e-4

VISUAL_RAW_DIM = 9
VISUAL_DIM = 512

LATENT_DIM = 128

EMBED_DIM = 96
GRU_HIDDEN = 128

NOISE_STD = 0.03

LAMBDA_ACTION = 1.0
LAMBDA_SEM = 0.5
LAMBDA_OBJ = 0.5
LAMBDA_STATE = 0.5


# ============================================================
# 2. Physical definitions
# ============================================================

COLORS = [
    "red",
    "blue",
    "green"
]

ACTIONS = [
    "pick_up",
    "push_left",
    "push_right",
    "place_down"
]

ACTION_TO_ID = {
    name: i
    for i, name in enumerate(ACTIONS)
}

COLOR_TO_ID = {
    name: i
    for i, name in enumerate(COLORS)
}


INSTRUCTIONS = {

    ("pick_up", "red"):
        "pick up the red cube",

    ("pick_up", "blue"):
        "pick up the blue cube",

    ("pick_up", "green"):
        "pick up the green cube",

    ("push_left", "red"):
        "push red cube left",

    ("push_left", "blue"):
        "push blue cube left",

    ("push_left", "green"):
        "push green cube left",

    ("push_right", "red"):
        "push red cube right",

    ("push_right", "blue"):
        "push blue cube right",

    ("push_right", "green"):
        "push green cube right",

    ("place_down", "red"):
        "place the red cube down",

    ("place_down", "blue"):
        "place the blue cube down",

    ("place_down", "green"):
        "place the green cube down"
}


# ============================================================
# 3. Tokenization
# ============================================================

def tokenize(text):

    return re.findall(
        r"[a-z]+",
        text.lower()
    )


counter = {}

for text in INSTRUCTIONS.values():

    for token in tokenize(text):

        counter[token] = (
            counter.get(token, 0) + 1
        )


PAD = "<PAD>"
UNK = "<UNK>"

vocab = {
    PAD: 0,
    UNK: 1
}

for token in sorted(counter.keys()):

    vocab[token] = len(vocab)


print("\nVocabulary size:", len(vocab))
print("Vocabulary:", vocab)


# ============================================================
# 4. Fixed 9D -> 512D visual projection
# ============================================================

class FixedVisualProjection(nn.Module):

    def __init__(
        self,
        in_dim=9,
        out_dim=512
    ):

        super().__init__()

        generator = torch.Generator()
        generator.manual_seed(1234)

        weight = torch.randn(
            out_dim,
            in_dim,
            generator=generator
        ) * math.sqrt(
            2.0 / in_dim
        )

        bias = torch.randn(
            out_dim,
            generator=generator
        ) * 0.02

        self.register_buffer(
            "weight",
            weight
        )

        self.register_buffer(
            "bias",
            bias
        )

    def forward(self, x):

        return F.linear(
            x,
            self.weight,
            self.bias
        )


visual_projection = FixedVisualProjection(
    VISUAL_RAW_DIM,
    VISUAL_DIM
).to(device)

visual_projection.eval()


# ============================================================
# 5. Dataset
# ============================================================

class AntPhysicsDataset(Dataset):

    def __init__(
        self,
        num_samples=2400,
        noise_std=0.03,
        seed=42
    ):

        super().__init__()

        rng = np.random.default_rng(
            seed
        )

        self.samples = []

        for i in range(num_samples):

            # ------------------------------------------------
            # Randomized scene
            # ------------------------------------------------

            positions = {

                "red":
                    rng.uniform(
                        [-1.0, -1.0, 0.05],
                        [1.0, 1.0, 1.0],
                        3
                    ).astype(
                        np.float32
                    ),

                "blue":
                    rng.uniform(
                        [-1.0, -1.0, 0.05],
                        [1.0, 1.0, 1.0],
                        3
                    ).astype(
                        np.float32
                    ),

                "green":
                    rng.uniform(
                        [-1.0, -1.0, 0.05],
                        [1.0, 1.0, 1.0],
                        3
                    ).astype(
                        np.float32
                    )
            }

            action_name = random.choice(
                ACTIONS
            )

            color = random.choice(
                COLORS
            )

            instruction = INSTRUCTIONS[
                (
                    action_name,
                    color
                )
            ]

            target = positions[
                color
            ].copy()

            x, y, z = target

            # ------------------------------------------------
            # Action
            # ------------------------------------------------

            if action_name == "pick_up":

                action = np.array(
                    [
                        x,
                        y,
                        z + 0.12,
                        0.0,
                        0.0,
                        0.0,
                        1.0
                    ],
                    dtype=np.float32
                )

            elif action_name == "push_left":

                action = np.array(
                    [
                        x - 0.20,
                        y,
                        z,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ],
                    dtype=np.float32
                )

            elif action_name == "push_right":

                action = np.array(
                    [
                        x + 0.20,
                        y,
                        z,
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ],
                    dtype=np.float32
                )

            else:

                action = np.array(
                    [
                        x,
                        y,
                        max(
                            0.02,
                            z - 0.15
                        ),
                        0.0,
                        0.0,
                        0.0,
                        0.0
                    ],
                    dtype=np.float32
                )

            # ------------------------------------------------
            # 9D scene
            #
            # red_xyz | blue_xyz | green_xyz
            # ------------------------------------------------

            raw_visual = np.concatenate(
                [
                    positions["red"],
                    positions["blue"],
                    positions["green"]
                ]
            ).astype(
                np.float32
            )

            raw_tensor = torch.from_numpy(
                raw_visual
            ).float()

            with torch.no_grad():

                visual = visual_projection(
                    raw_tensor.to(device)
                ).cpu()

            visual += (
                torch.randn_like(
                    visual
                ) * noise_std
            )

            tokens = torch.tensor(
                [
                    vocab.get(
                        token,
                        vocab[UNK]
                    )
                    for token in tokenize(
                        instruction
                    )
                ],
                dtype=torch.long
            )

            self.samples.append(
                {
                    "visual":
                        visual.float(),

                    "raw_visual":
                        torch.from_numpy(
                            raw_visual
                        ).float(),

                    "tokens":
                        tokens,

                    "action":
                        torch.from_numpy(
                            action
                        ).float(),

                    "state":
                        torch.from_numpy(
                            target
                        ).float(),

                    "action_id":
                        ACTION_TO_ID[
                            action_name
                        ],

                    "color_id":
                        COLOR_TO_ID[
                            color
                        ],

                    "instruction":
                        instruction,

                    "positions":
                        positions,

                    "action_name":
                        action_name,

                    "color":
                        color
                }
            )

    def __len__(self):

        return len(
            self.samples
        )

    def __getitem__(self, idx):

        return self.samples[idx]


# ============================================================
# 6. Collate
# ============================================================

def collate_fn(batch):

    lengths = torch.tensor(
        [
            len(
                x["tokens"]
            )
            for x in batch
        ],
        dtype=torch.long
    )

    max_len = int(
        lengths.max()
    )

    tokens = torch.full(
        (
            len(batch),
            max_len
        ),
        vocab[PAD],
        dtype=torch.long
    )

    for i, item in enumerate(batch):

        n = len(
            item["tokens"]
        )

        tokens[
            i,
            :n
        ] = item["tokens"]

    return {

        "visual":
            torch.stack(
                [
                    x["visual"]
                    for x in batch
                ]
            ),

        "raw_visual":
            torch.stack(
                [
                    x["raw_visual"]
                    for x in batch
                ]
            ),

        "tokens":
            tokens,

        "lengths":
            lengths,

        "action":
            torch.stack(
                [
                    x["action"]
                    for x in batch
                ]
            ),

        "state":
            torch.stack(
                [
                    x["state"]
                    for x in batch
                ]
            ),

        "action_id":
            torch.tensor(
                [
                    x["action_id"]
                    for x in batch
                ],
                dtype=torch.long
            ),

        "color_id":
            torch.tensor(
                [
                    x["color_id"]
                    for x in batch
                ],
                dtype=torch.long
            ),

        "instruction":
            [
                x["instruction"]
                for x in batch
            ]
    }


# ============================================================
# 7. Dataset
# ============================================================

dataset = AntPhysicsDataset(
    NUM_SAMPLES,
    NOISE_STD,
    SEED
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

print(
    "\nDataset size:",
    len(dataset)
)


# ============================================================
# 8. Random State Check
# ============================================================

print(
    "\nPosition-randomization check:"
)

shown = 0

for sample in dataset.samples:

    if sample["instruction"] == \
       "pick up the red cube":

        print(
            sample["instruction"],
            "->",
            sample["positions"]["red"]
        )

        shown += 1

        if shown == 3:
            break


# ============================================================
# 9. Vision-Only Factorized Teacher
#
# CRITICAL DIFFERENCE FROM EXPERIMENT 2:
#
# Action is NOT an input to the Teacher.
#
# Teacher:
#       Vision
#          ↓
#     shared trunk
#          ↓
#    ┌─────┼─────┐
#    ↓     ↓     ↓
#   SEM   OBJ   STATE
#
# Action is ONLY the reconstruction target.
# ============================================================

class VisionOnlyFactorizedTeacher(
    nn.Module
):

    def __init__(
        self,
        visual_dim=512,
        latent_dim=128
    ):

        super().__init__()

        self.visual_trunk = nn.Sequential(

            nn.Linear(
                visual_dim,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU(),

            nn.Linear(
                384,
                256
            ),

            nn.LayerNorm(256),

            nn.GELU()
        )

        # ----------------------------------------------------
        # Semantic
        # ----------------------------------------------------

        self.semantic_head = nn.Sequential(

            nn.Linear(
                256,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

        # ----------------------------------------------------
        # Object
        # ----------------------------------------------------

        self.object_head = nn.Sequential(

            nn.Linear(
                256,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

        # ----------------------------------------------------
        # State
        # ----------------------------------------------------

        self.state_head = nn.Sequential(

            nn.Linear(
                256,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

        # ----------------------------------------------------
        # Action reconstruction
        # ----------------------------------------------------

        self.action_head = nn.Sequential(

            nn.Linear(
                latent_dim * 3,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU(),

            nn.Linear(
                384,
                256
            ),

            nn.GELU(),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Linear(
                128,
                7
            )
        )

        # ----------------------------------------------------
        # Auxiliary classifiers
        # ----------------------------------------------------

        self.semantic_classifier = nn.Linear(
            latent_dim,
            len(ACTIONS)
        )

        self.object_classifier = nn.Linear(
            latent_dim,
            len(COLORS)
        )

        self.state_decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                64
            ),

            nn.GELU(),

            nn.Linear(
                64,
                3
            )
        )

    def forward(
        self,
        visual
    ):

        h = self.visual_trunk(
            visual
        )

        z_sem = self.semantic_head(
            h
        )

        z_obj = self.object_head(
            h
        )

        z_state = self.state_head(
            h
        )

        combined = torch.cat(
            [
                z_sem,
                z_obj,
                z_state
            ],
            dim=1
        )

        action_pred = self.action_head(
            combined
        )

        semantic_logits = \
            self.semantic_classifier(
                z_sem
            )

        object_logits = \
            self.object_classifier(
                z_obj
            )

        state_pred = \
            self.state_decoder(
                z_state
            )

        return {

            "z_sem":
                z_sem,

            "z_obj":
                z_obj,

            "z_state":
                z_state,

            "action_pred":
                action_pred,

            "semantic_logits":
                semantic_logits,

            "object_logits":
                object_logits,

            "state_pred":
                state_pred
        }


# ============================================================
# 10. Teacher
# ============================================================

teacher = VisionOnlyFactorizedTeacher().to(
    device
)

teacher_optimizer = torch.optim.AdamW(
    teacher.parameters(),
    lr=LR_TEACHER,
    weight_decay=1e-4
)


def teacher_loss(
    outputs,
    action,
    action_id,
    color_id,
    state
):

    L_action = F.mse_loss(
        outputs["action_pred"],
        action
    )

    L_sem = F.cross_entropy(
        outputs["semantic_logits"],
        action_id
    )

    L_obj = F.cross_entropy(
        outputs["object_logits"],
        color_id
    )

    L_state = F.mse_loss(
        outputs["state_pred"],
        state
    )

    total = (

        LAMBDA_ACTION * L_action

        + LAMBDA_SEM * L_sem

        + LAMBDA_OBJ * L_obj

        + LAMBDA_STATE * L_state
    )

    return (
        total,
        L_action,
        L_sem,
        L_obj,
        L_state
    )


print(
    "\nTeacher parameters:",
    sum(
        p.numel()
        for p in teacher.parameters()
    )
)


# ============================================================
# 11. Teacher Training
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "PHASE 1: VISION-ONLY PHYSICAL TEACHER"
)

print(
    "=" * 72
)

teacher_history = []

for epoch in range(
    1,
    TEACHER_EPOCHS + 1
):

    teacher.train()

    total_sum = 0.0
    action_sum = 0.0
    sem_sum = 0.0
    obj_sum = 0.0
    state_sum = 0.0

    batches = 0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        action = batch[
            "action"
        ].to(device)

        state = batch[
            "state"
        ].to(device)

        action_id = batch[
            "action_id"
        ].to(device)

        color_id = batch[
            "color_id"
        ].to(device)

        outputs = teacher(
            visual
        )

        (
            loss,
            L_action,
            L_sem,
            L_obj,
            L_state
        ) = teacher_loss(
            outputs,
            action,
            action_id,
            color_id,
            state
        )

        teacher_optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            teacher.parameters(),
            1.0
        )

        teacher_optimizer.step()

        total_sum += loss.item()
        action_sum += L_action.item()
        sem_sum += L_sem.item()
        obj_sum += L_obj.item()
        state_sum += L_state.item()

        batches += 1

    values = (
        total_sum / batches,
        action_sum / batches,
        sem_sum / batches,
        obj_sum / batches,
        state_sum / batches
    )

    teacher_history.append(
        values
    )

    print(
        f"Teacher Epoch "
        f"{epoch:02d}/{TEACHER_EPOCHS} | "
        f"Total={values[0]:.6f} | "
        f"Action={values[1]:.6f} | "
        f"Sem={values[2]:.6f} | "
        f"Obj={values[3]:.6f} | "
        f"State={values[4]:.6f}"
    )


# ============================================================
# 12. Freeze Teacher
# ============================================================

teacher.eval()

for p in teacher.parameters():

    p.requires_grad = False


print(
    "\nTeacher FROZEN."
)

print(
    "Trainable Teacher params:",
    sum(
        p.numel()
        for p in teacher.parameters()
        if p.requires_grad
    )
)


# ============================================================
# 13. Language Student
#
# NO STATE HEAD.
# ============================================================

class LanguageStudent(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim=96,
        hidden_dim=128,
        latent_dim=128
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=vocab[PAD]
        )

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.semantic_head = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

        self.object_head = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                192
            ),

            nn.GELU(),

            nn.Linear(
                192,
                latent_dim
            ),

            nn.Tanh()
        )

    def forward(
        self,
        tokens,
        lengths
    ):

        embedded = self.embedding(
            tokens
        )

        packed = \
            nn.utils.rnn.pack_padded_sequence(
                embedded,
                lengths.cpu(),
                batch_first=True,
                enforce_sorted=False
            )

        _, hidden = self.gru(
            packed
        )

        forward_hidden = hidden[0]
        backward_hidden = hidden[1]

        h = torch.cat(
            [
                forward_hidden,
                backward_hidden
            ],
            dim=1
        )

        z_sem = self.semantic_head(
            h
        )

        z_obj = self.object_head(
            h
        )

        return (
            z_sem,
            z_obj
        )


student = LanguageStudent(
    len(vocab),
    EMBED_DIM,
    GRU_HIDDEN,
    LATENT_DIM
).to(device)

student_optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=LR_STUDENT,
    weight_decay=1e-4
)


# ============================================================
# 14. Student Distillation
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "PHASE 2: LANGUAGE STUDENT DISTILLATION"
)

print(
    "=" * 72
)

student_history = []

for epoch in range(
    1,
    STUDENT_EPOCHS + 1
):

    student.train()

    total_sum = 0.0
    sem_sum = 0.0
    obj_sum = 0.0

    batches = 0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        tokens = batch[
            "tokens"
        ].to(device)

        lengths = batch[
            "lengths"
        ]

        with torch.no_grad():

            teacher_outputs = teacher(
                visual
            )

            z_sem_T = teacher_outputs[
                "z_sem"
            ]

            z_obj_T = teacher_outputs[
                "z_obj"
            ]

        (
            z_sem_L,
            z_obj_L
        ) = student(
            tokens,
            lengths
        )

        L_sem = F.mse_loss(
            z_sem_L,
            z_sem_T
        )

        L_obj = F.mse_loss(
            z_obj_L,
            z_obj_T
        )

        loss = (
            L_sem
            + L_obj
        )

        student_optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            1.0
        )

        student_optimizer.step()

        total_sum += loss.item()
        sem_sum += L_sem.item()
        obj_sum += L_obj.item()

        batches += 1

    values = (
        total_sum / batches,
        sem_sum / batches,
        obj_sum / batches
    )

    student_history.append(
        values
    )

    print(
        f"Student Epoch "
        f"{epoch:02d}/{STUDENT_EPOCHS} | "
        f"Total={values[0]:.6f} | "
        f"Sem={values[1]:.6f} | "
        f"Obj={values[2]:.6f}"
    )


# ============================================================
# 15. Freeze Student
# ============================================================

student.eval()

for p in student.parameters():

    p.requires_grad = False


print(
    "\nStudent FROZEN."
)


# ============================================================
# 16. Final VLA Action Head
#
# Language:
#   Semantic
#   Object
#
# Vision:
#   State
#
# Action:
#   [semantic_L, object_L, state_V]
# ============================================================

class VLAActionHead(
    nn.Module
):

    def __init__(
        self,
        latent_dim=128
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                latent_dim * 3,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU(),

            nn.Linear(
                384,
                256
            ),

            nn.GELU(),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Linear(
                128,
                7
            )
        )

    def forward(
        self,
        z_sem_L,
        z_obj_L,
        z_state_V
    ):

        h = torch.cat(
            [
                z_sem_L,
                z_obj_L,
                z_state_V
            ],
            dim=1
        )

        return self.net(h)


vla = VLAActionHead(
    LATENT_DIM
).to(device)

vla_optimizer = torch.optim.AdamW(
    vla.parameters(),
    lr=LR_VLA,
    weight_decay=1e-4
)


# ============================================================
# 17. VLA Training
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "PHASE 3: FACTORIZED VLA"
)

print(
    "=" * 72
)

vla_history = []

for epoch in range(
    1,
    VLA_EPOCHS + 1
):

    vla.train()

    loss_sum = 0.0
    batches = 0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        tokens = batch[
            "tokens"
        ].to(device)

        lengths = batch[
            "lengths"
        ]

        action = batch[
            "action"
        ].to(device)

        with torch.no_grad():

            teacher_outputs = teacher(
                visual
            )

            z_state_V = teacher_outputs[
                "z_state"
            ]

            (
                z_sem_L,
                z_obj_L
            ) = student(
                tokens,
                lengths
            )

        action_pred = vla(
            z_sem_L,
            z_obj_L,
            z_state_V
        )

        loss = F.mse_loss(
            action_pred,
            action
        )

        vla_optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            vla.parameters(),
            1.0
        )

        vla_optimizer.step()

        loss_sum += loss.item()
        batches += 1

    avg_loss = (
        loss_sum / batches
    )

    vla_history.append(
        avg_loss
    )

    print(
        f"VLA Epoch "
        f"{epoch:02d}/{VLA_EPOCHS} | "
        f"Action={avg_loss:.6f}"
    )


# ============================================================
# 18. Final Batch
# ============================================================

teacher.eval()
student.eval()
vla.eval()

batch = next(
    iter(loader)
)

visual = batch[
    "visual"
].to(device)

tokens = batch[
    "tokens"
].to(device)

lengths = batch[
    "lengths"
]

action = batch[
    "action"
].to(device)

with torch.no_grad():

    teacher_outputs = teacher(
        visual
    )

    (
        z_sem_L,
        z_obj_L
    ) = student(
        tokens,
        lengths
    )

    action_pred = vla(
        z_sem_L,
        z_obj_L,
        teacher_outputs[
            "z_state"
        ]
    )

final_action_loss = F.mse_loss(
    action_pred,
    action
)


# ============================================================
# 19. Architecture Check
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "FINAL ARCHITECTURE CHECK"
)

print(
    "=" * 72
)

print(
    "Visual:",
    tuple(visual.shape)
)

print(
    "Teacher z_sem:",
    tuple(
        teacher_outputs[
            "z_sem"
        ].shape
    )
)

print(
    "Teacher z_obj:",
    tuple(
        teacher_outputs[
            "z_obj"
        ].shape
    )
)

print(
    "Teacher z_state:",
    tuple(
        teacher_outputs[
            "z_state"
        ].shape
    )
)

print(
    "Student z_sem:",
    tuple(z_sem_L.shape)
)

print(
    "Student z_obj:",
    tuple(z_obj_L.shape)
)

print(
    "Student z_state: NONE"
)

print(
    "Action:",
    tuple(action_pred.shape)
)


# ============================================================
# 20. Freeze Check
# ============================================================

teacher_trainable = sum(
    p.numel()
    for p in teacher.parameters()
    if p.requires_grad
)

student_trainable = sum(
    p.numel()
    for p in student.parameters()
    if p.requires_grad
)

vla_trainable = sum(
    p.numel()
    for p in vla.parameters()
    if p.requires_grad
)

print(
    "\nFreeze check:"
)

print(
    "Teacher trainable:",
    teacher_trainable
)

print(
    "Student trainable:",
    student_trainable
)

print(
    "VLA trainable:",
    vla_trainable
)


# ============================================================
# 21. Alignment
# ============================================================

with torch.no_grad():

    sem_alignment = F.mse_loss(
        z_sem_L,
        teacher_outputs[
            "z_sem"
        ]
    ).item()

    obj_alignment = F.mse_loss(
        z_obj_L,
        teacher_outputs[
            "z_obj"
        ]
    ).item()


# ============================================================
# 22. Same Language / Different State
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "TEST 1: SAME LANGUAGE / DIFFERENT STATE"
)

print(
    "=" * 72
)

target_instruction = \
    "push red cube right"

matches = [

    x
    for x in dataset.samples
    if x["instruction"]
    == target_instruction
]


if len(matches) >= 2:

    A = matches[0]
    B = matches[1]

    token_A = A[
        "tokens"
    ].unsqueeze(0).to(device)

    token_B = B[
        "tokens"
    ].unsqueeze(0).to(device)

    len_A = torch.tensor(
        [
            len(
                A["tokens"]
            )
        ],
        dtype=torch.long
    )

    len_B = torch.tensor(
        [
            len(
                B["tokens"]
            )
        ],
        dtype=torch.long
    )

    visual_A = A[
        "visual"
    ].unsqueeze(0).to(device)

    visual_B = B[
        "visual"
    ].unsqueeze(0).to(device)

    with torch.no_grad():

        sem_A, obj_A = student(
            token_A,
            len_A
        )

        sem_B, obj_B = student(
            token_B,
            len_B
        )

        teacher_A = teacher(
            visual_A
        )

        teacher_B = teacher(
            visual_B
        )

        action_A = vla(
            sem_A,
            obj_A,
            teacher_A[
                "z_state"
            ]
        )

        action_B = vla(
            sem_B,
            obj_B,
            teacher_B[
                "z_state"
            ]
        )

    language_sem_diff = F.mse_loss(
        sem_A,
        sem_B
    ).item()

    language_obj_diff = F.mse_loss(
        obj_A,
        obj_B
    ).item()

    state_diff = F.mse_loss(
        teacher_A[
            "z_state"
        ],
        teacher_B[
            "z_state"
        ]
    ).item()

    action_diff = F.mse_loss(
        action_A,
        action_B
    ).item()

    print(
        "\nInstruction:",
        target_instruction
    )

    print(
        "\nScene A red xyz:",
        A["positions"]["red"]
    )

    print(
        "Scene B red xyz:",
        B["positions"]["red"]
    )

    print(
        "\nLanguage Semantic difference:",
        f"{language_sem_diff:.8f}"
    )

    print(
        "Language Object difference:",
        f"{language_obj_diff:.8f}"
    )

    print(
        "Visual State difference:",
        f"{state_diff:.8f}"
    )

    print(
        "Action difference:",
        f"{action_diff:.8f}"
    )


# ============================================================
# 23. STATE SHUFFLE ABLATION
#
# Same language.
# Correct State vs wrong State.
#
# If factorization works:
#
#   Correct State -> correct action
#   Wrong State   -> shifted action
#
# Language remains identical.
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "TEST 2: STATE SHUFFLE ABLATION"
)

print(
    "=" * 72
)

if len(matches) >= 2:

    A = matches[0]
    B = matches[1]

    token_A = A[
        "tokens"
    ].unsqueeze(0).to(device)

    len_A = torch.tensor(
        [
            len(
                A["tokens"]
            )
        ],
        dtype=torch.long
    )

    visual_A = A[
        "visual"
    ].unsqueeze(0).to(device)

    visual_B = B[
        "visual"
    ].unsqueeze(0).to(device)

    with torch.no_grad():

        sem_A, obj_A = student(
            token_A,
            len_A
        )

        teacher_A = teacher(
            visual_A
        )

        teacher_B = teacher(
            visual_B
        )

        correct_action = vla(
            sem_A,
            obj_A,
            teacher_A[
                "z_state"
            ]
        )

        shuffled_action = vla(
            sem_A,
            obj_A,
            teacher_B[
                "z_state"
            ]
        )

    shuffle_difference = F.mse_loss(
        correct_action,
        shuffled_action
    ).item()

    print(
        "\nSame Language:",
        target_instruction
    )

    print(
        "Correct State:",
        A["positions"]["red"]
    )

    print(
        "Shuffled State:",
        B["positions"]["red"]
    )

    print(
        "\nCorrect-state action:"
    )

    print(
        correct_action[
            0
        ].cpu().numpy()
    )

    print(
        "\nShuffled-state action:"
    )

    print(
        shuffled_action[
            0
        ].cpu().numpy()
    )

    print(
        "\nAction difference after State Shuffle:",
        f"{shuffle_difference:.8f}"
    )

    print(
        "\nExpected:"
    )

    print(
        "  Language = unchanged"
    )

    print(
        "  State = changed"
    )

    print(
        "  Action = changed"
    )


# ============================================================
# 24. LANGUAGE SHUFFLE ABLATION
#
# Same visual scene.
# Different language.
#
# If factorization works:
#
#   State remains identical.
#   Action changes because semantic/object changes.
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "TEST 3: LANGUAGE SHUFFLE ABLATION"
)

print(
    "=" * 72
)

# ------------------------------------------------------------
# Find two instructions with different actions but same color.
# ------------------------------------------------------------

sample_A = None
sample_B = None

for x in dataset.samples:

    if x["color"] == "red":

        if x["action_name"] == "push_right":

            sample_A = x

        if x["action_name"] == "push_left":

            sample_B = x

    if (
        sample_A is not None
        and sample_B is not None
    ):
        break


if (
    sample_A is not None
    and sample_B is not None
):

    # Same visual scene from A.
    # Different language from B.

    visual_same = sample_A[
        "visual"
    ].unsqueeze(0).to(device)

    tokens_A = sample_A[
        "tokens"
    ].unsqueeze(0).to(device)

    tokens_B = sample_B[
        "tokens"
    ].unsqueeze(0).to(device)

    len_A = torch.tensor(
        [
            len(
                sample_A["tokens"]
            )
        ],
        dtype=torch.long
    )

    len_B = torch.tensor(
        [
            len(
                sample_B["tokens"]
            )
        ],
        dtype=torch.long
    )

    with torch.no_grad():

        sem_A, obj_A = student(
            tokens_A,
            len_A
        )

        sem_B, obj_B = student(
            tokens_B,
            len_B
        )

        teacher_same = teacher(
            visual_same
        )

        action_from_A = vla(
            sem_A,
            obj_A,
            teacher_same[
                "z_state"
            ]
        )

        action_from_B = vla(
            sem_B,
            obj_B,
            teacher_same[
                "z_state"
            ]
        )

    language_shuffle_action_diff = \
        F.mse_loss(
            action_from_A,
            action_from_B
        ).item()

    state_same_diff = 0.0

    print(
        "\nScene red xyz:",
        sample_A[
            "positions"
        ]["red"]
    )

    print(
        "\nLanguage A:",
        sample_A[
            "instruction"
        ]
    )

    print(
        "Language B:",
        sample_B[
            "instruction"
        ]
    )

    print(
        "\nAction A:"
    )

    print(
        action_from_A[
            0
        ].cpu().numpy()
    )

    print(
        "\nAction B:"
    )

    print(
        action_from_B[
            0
        ].cpu().numpy()
    )

    print(
        "\nAction difference after Language Shuffle:",
        f"{language_shuffle_action_diff:.8f}"
    )


# ============================================================
# 25. Final Report
# ============================================================

print(
    "\n" + "=" * 72
)

print(
    "FINAL REPORT"
)

print(
    "=" * 72
)

print(
    "\nTeacher Action:",
    f"{teacher_history[0][1]:.6f}",
    "->",
    f"{teacher_history[-1][1]:.6f}"
)

print(
    "Teacher Semantic:",
    f"{teacher_history[0][2]:.6f}",
    "->",
    f"{teacher_history[-1][2]:.6f}"
)

print(
    "Teacher Object:",
    f"{teacher_history[0][3]:.6f}",
    "->",
    f"{teacher_history[-1][3]:.6f}"
)

print(
    "Teacher State:",
    f"{teacher_history[0][4]:.6f}",
    "->",
    f"{teacher_history[-1][4]:.6f}"
)

print(
    "\nStudent Semantic:",
    f"{student_history[0][1]:.6f}",
    "->",
    f"{student_history[-1][1]:.6f}"
)

print(
    "Student Object:",
    f"{student_history[0][2]:.6f}",
    "->",
    f"{student_history[-1][2]:.6f}"
)

print(
    "\nFinal Semantic Alignment:",
    f"{sem_alignment:.6f}"
)

print(
    "Final Object Alignment:",
    f"{obj_alignment:.6f}"
)

print(
    "Final VLA Action Loss:",
    f"{final_action_loss.item():.6f}"
)

print(
    "\n" + "=" * 72
)

print(
    "Experiment 3 finished successfully."
)

print(
    "Factorization hypothesis tested."
)

print(
    "=" * 72
)


AntEncoder Experiment 3
Vision-Only Factorized Teacher
Device: cpu

Vocabulary size: 14
Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'blue': 2, 'cube': 3, 'down': 4, 'green': 5, 'left': 6, 'pick': 7, 'place': 8, 'push': 9, 'red': 10, 'right': 11, 'the': 12, 'up': 13}

Dataset size: 2400

Position-randomization check:
pick up the red cube -> [ 0.5479121  -0.12224312  0.865668  ]
pick up the red cube -> [0.41033074 0.56145805 0.48597   ]
pick up the red cube -> [ 0.10807229 -0.78284854  0.6886281 ]

Teacher parameters: 809297

PHASE 1: VISION-ONLY PHYSICAL TEACHER
Teacher Epoch 01/100 | Total=1.486984 | Action=0.121402 | Sem=1.392352 | Obj=1.102703 | State=0.236110
Teacher Epoch 02/100 | Total=1.435092 | Action=0.107848 | Sem=1.383586 | Obj=1.095380 | State=0.175522
Teacher Epoch 03/100 | Total=1.420181 | Action=0.103748 | Sem=1.375069 | Obj=1.089653 | State=0.168143
Teacher Epoch 04/100 | Total=1.411874 | Action=0.101902 | Sem=1.367931 | Obj=1.086597 | State=0.165416
Teacher Epoch 05/100 | Tota

In [ ]:
# ================================================================
# AntEncoder Experiment 4
# Separated Object Teacher + State Teacher + Language Student
# Google Colab single-cell implementation
# ================================================================

import random
import re
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ================================================================
# 0. Reproducibility / Device
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 72)
print("AntEncoder Experiment 4")
print("Separated Object Teacher + State Teacher + Language Student")
print("=" * 72)
print("Device:", device)


# ================================================================
# 1. Vocabulary
# ================================================================

instructions = [
    "pick up the red cube",
    "push red cube right",
    "push red cube left",
    "place red cube down",

    "pick up the blue cube",
    "push blue cube right",
    "push blue cube left",
    "place blue cube down",

    "pick up the green cube",
    "push green cube right",
    "push green cube left",
    "place green cube down",
]

special_tokens = ["<PAD>", "<UNK>"]

all_words = set()

for text in instructions:
    all_words.update(text.lower().split())

vocab_words = sorted(all_words)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for word in vocab_words:
    if word not in vocab:
        vocab[word] = len(vocab)

PAD_IDX = vocab["<PAD>"]
UNK_IDX = vocab["<UNK>"]

print("\nVocabulary size:", len(vocab))
print("Vocabulary:", vocab)


def tokenize(text, max_len=8):
    words = text.lower().split()

    ids = [
        vocab.get(word, UNK_IDX)
        for word in words
    ]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))

    return ids


# ================================================================
# 2. Dataset
#
# Visual input:
#
#   appearance:
#       red   = [1,0,0]
#       blue  = [0,0,1]
#       green = [0,1,0]
#
#   state:
#       xyz
#
# 3 objects × (RGB + XYZ) = 18 dimensions
#
# Then:
#   18D -> fixed random projection -> 512D
# ================================================================

class AntPhysicsDataset(Dataset):

    COLORS = ["red", "blue", "green"]

    COLOR_RGB = {
        "red":   np.array([1.0, 0.0, 0.0], dtype=np.float32),
        "blue":  np.array([0.0, 0.0, 1.0], dtype=np.float32),
        "green": np.array([0.0, 1.0, 0.0], dtype=np.float32),
    }

    ACTIONS = {
        "pick": 0,
        "right": 1,
        "left": 2,
        "down": 3,
    }

    def __init__(self, n_samples=2400, seed=42):

        self.n_samples = n_samples
        self.rng = np.random.default_rng(seed)

        # --------------------------------------------------------
        # Fixed physical projection:
        # 18D physical observation -> 512D visual representation
        # --------------------------------------------------------

        projection_rng = np.random.default_rng(12345)

        self.visual_projection = (
            projection_rng.normal(
                0.0,
                1.0 / np.sqrt(18),
                size=(18, 512)
            )
            .astype(np.float32)
        )

        self.samples = []

        for i in range(n_samples):

            color = self.COLORS[
                self.rng.integers(0, len(self.COLORS))
            ]

            action_name = [
                "pick",
                "right",
                "left",
                "down"
            ][self.rng.integers(0, 4)]

            if action_name == "pick":
                instruction = f"pick up the {color} cube"

            elif action_name == "right":
                instruction = f"push {color} cube right"

            elif action_name == "left":
                instruction = f"push {color} cube left"

            else:
                instruction = f"place {color} cube down"

            # ----------------------------------------------------
            # Randomize every object's 3D position independently.
            # This is critical for State randomization.
            # ----------------------------------------------------

            positions = {}

            for c in self.COLORS:

                positions[c] = self.rng.uniform(
                    low=[-1.0, -1.0, 0.35],
                    high=[1.0, 1.0, 1.0],
                    size=3
                ).astype(np.float32)

            # Target object's position
            target_xyz = positions[color].copy()

            # ----------------------------------------------------
            # 18D physical visual representation
            #
            # [red RGB, red XYZ,
            #  blue RGB, blue XYZ,
            #  green RGB, green XYZ]
            # ----------------------------------------------------

            physical_visual = []

            for c in self.COLORS:

                physical_visual.extend(
                    self.COLOR_RGB[c]
                )

                physical_visual.extend(
                    positions[c]
                )

            physical_visual = np.asarray(
                physical_visual,
                dtype=np.float32
            )

            # ----------------------------------------------------
            # Project 18D -> 512D
            # + Gaussian visual noise
            # ----------------------------------------------------

            visual = physical_visual @ self.visual_projection

            visual += self.rng.normal(
                0.0,
                0.02,
                size=512
            ).astype(np.float32)

            # ----------------------------------------------------
            # Action target
            #
            # Position is explicitly conditioned on target object.
            # Semantic command determines motion direction/gripper.
            # ----------------------------------------------------

            x, y, z = target_xyz

            roll = 0.0
            pitch = 0.0
            yaw = 0.0

            if action_name == "pick":

                action = [
                    x,
                    y,
                    z,
                    roll,
                    pitch,
                    yaw,
                    1.0
                ]

            elif action_name == "right":

                action = [
                    x + 0.35,
                    y,
                    z,
                    roll,
                    pitch,
                    yaw,
                    0.0
                ]

            elif action_name == "left":

                action = [
                    x - 0.35,
                    y,
                    z,
                    roll,
                    pitch,
                    yaw,
                    0.0
                ]

            else:

                action = [
                    x,
                    y,
                    max(0.20, z - 0.30),
                    roll,
                    pitch,
                    yaw,
                    0.0
                ]

            action = np.asarray(
                action,
                dtype=np.float32
            )

            # ----------------------------------------------------
            # Semantic label
            # ----------------------------------------------------

            semantic_label = self.ACTIONS[action_name]

            # ----------------------------------------------------
            # Object label
            # ----------------------------------------------------

            object_label = self.COLORS.index(color)

            self.samples.append({
                "instruction": instruction,
                "tokens": np.asarray(
                    tokenize(instruction),
                    dtype=np.int64
                ),
                "visual": visual.astype(np.float32),
                "physical_visual": physical_visual,
                "target_xyz": target_xyz,
                "semantic_label": semantic_label,
                "object_label": object_label,
                "action": action,
                "color": color,
                "action_name": action_name,
                "positions": positions,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        s = self.samples[idx]

        return {
            "instruction": s["instruction"],
            "tokens": torch.tensor(
                s["tokens"],
                dtype=torch.long
            ),
            "visual": torch.tensor(
                s["visual"],
                dtype=torch.float32
            ),
            "physical_visual": torch.tensor(
                s["physical_visual"],
                dtype=torch.float32
            ),
            "target_xyz": torch.tensor(
                s["target_xyz"],
                dtype=torch.float32
            ),
            "semantic_label": torch.tensor(
                s["semantic_label"],
                dtype=torch.long
            ),
            "object_label": torch.tensor(
                s["object_label"],
                dtype=torch.long
            ),
            "action": torch.tensor(
                s["action"],
                dtype=torch.float32
            ),
        }


dataset = AntPhysicsDataset(
    n_samples=2400,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

print("\nDataset size:", len(dataset))


# ================================================================
# 3. Position Randomization Check
# ================================================================

print("\nPosition randomization check:")

count = 0

for s in dataset.samples:

    if s["instruction"] == "pick up the red cube":

        print(
            s["instruction"],
            "->",
            s["positions"]["red"]
        )

        count += 1

        if count == 3:
            break


# ================================================================
# 4. Object Teacher
#
# Vision -> Object latent
#
# Object information:
#   red / blue / green
#
# Teacher DOES NOT generate semantic information.
# ================================================================

class ObjectTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        latent_dim=128,
        n_objects=3
    ):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),
            nn.Tanh()
        )

        self.classifier = nn.Linear(
            latent_dim,
            n_objects
        )

    def forward(self, visual):

        z_obj = self.encoder(visual)

        logits = self.classifier(z_obj)

        return z_obj, logits


# ================================================================
# 5. State Teacher
#
# Vision -> State latent
#
# State target = target object's XYZ
#
# No language input.
# ================================================================

class StateTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        latent_dim=128
    ):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),
            nn.Tanh()
        )

        self.state_decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, visual):

        z_state = self.encoder(visual)

        state_pred = self.state_decoder(
            z_state
        )

        return z_state, state_pred


# ================================================================
# 6. Physical Teacher
#
# Object Teacher + State Teacher
#
# The physical teacher learns:
#
#   Visual -> Object
#   Visual -> State
#
# It intentionally does NOT predict semantic task.
# ================================================================

object_teacher = ObjectTeacher().to(device)
state_teacher = StateTeacher().to(device)

teacher_parameters = (
    list(object_teacher.parameters())
    +
    list(state_teacher.parameters())
)

teacher_optimizer = torch.optim.AdamW(
    teacher_parameters,
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nTeacher parameters:",
    sum(
        p.numel()
        for p in teacher_parameters
    )
)


# ================================================================
# 7. PHASE 1
#    Physical Teacher Training
# ================================================================

print("\n" + "=" * 72)
print("PHASE 1: SEPARATED PHYSICAL TEACHERS")
print("=" * 72)

teacher_epochs = 80

for epoch in range(teacher_epochs):

    object_teacher.train()
    state_teacher.train()

    total_loss_sum = 0.0
    object_loss_sum = 0.0
    state_loss_sum = 0.0

    for batch in loader:

        visual = batch["visual"].to(device)
        target_xyz = batch["target_xyz"].to(device)
        object_label = batch["object_label"].to(device)

        teacher_optimizer.zero_grad()

        z_obj, object_logits = object_teacher(
            visual
        )

        z_state, state_pred = state_teacher(
            visual
        )

        object_loss = F.cross_entropy(
            object_logits,
            object_label
        )

        state_loss = F.mse_loss(
            state_pred,
            target_xyz
        )

        loss = (
            object_loss
            +
            state_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            teacher_parameters,
            1.0
        )

        teacher_optimizer.step()

        batch_size = visual.size(0)

        total_loss_sum += (
            loss.item() * batch_size
        )

        object_loss_sum += (
            object_loss.item() * batch_size
        )

        state_loss_sum += (
            state_loss.item() * batch_size
        )

    n = len(dataset)

    total_loss = total_loss_sum / n
    object_loss = object_loss_sum / n
    state_loss = state_loss_sum / n

    if (
        epoch < 5
        or (epoch + 1) % 5 == 0
        or epoch == teacher_epochs - 1
    ):

        print(
            f"Teacher Epoch {epoch+1:02d}/{teacher_epochs} | "
            f"Total={total_loss:.6f} | "
            f"Object={object_loss:.6f} | "
            f"State={state_loss:.6f}"
        )


# ================================================================
# Freeze Teacher
# ================================================================

for p in object_teacher.parameters():
    p.requires_grad = False

for p in state_teacher.parameters():
    p.requires_grad = False

object_teacher.eval()
state_teacher.eval()

print("\nTeacher FROZEN.")

print(
    "Trainable Object Teacher params:",
    sum(
        p.numel()
        for p in object_teacher.parameters()
        if p.requires_grad
    )
)

print(
    "Trainable State Teacher params:",
    sum(
        p.numel()
        for p in state_teacher.parameters()
        if p.requires_grad
    )
)


# ================================================================
# 8. Language Student
#
# Language -> Semantic + Object
#
# State branch intentionally DOES NOT EXIST.
# ================================================================

class LanguageStudent(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=96,
        hidden_dim=128,
        latent_dim=128,
        n_semantic=4,
        n_objects=3
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.semantic_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, latent_dim),
            nn.Tanh()
        )

        self.object_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, latent_dim),
            nn.Tanh()
        )

        self.semantic_classifier = nn.Linear(
            latent_dim,
            n_semantic
        )

        self.object_classifier = nn.Linear(
            latent_dim,
            n_objects
        )

    def forward(self, tokens):

        x = self.embedding(tokens)

        output, hidden = self.gru(x)

        # Last forward hidden state
        h_forward = hidden[-2]

        # Last backward hidden state
        h_backward = hidden[-1]

        h = torch.cat(
            [h_forward, h_backward],
            dim=-1
        )

        z_sem = self.semantic_head(h)

        z_obj = self.object_head(h)

        semantic_logits = self.semantic_classifier(
            z_sem
        )

        object_logits = self.object_classifier(
            z_obj
        )

        return (
            z_sem,
            z_obj,
            semantic_logits,
            object_logits
        )


student = LanguageStudent(
    vocab_size=len(vocab)
).to(device)

student_optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nStudent parameters:",
    sum(
        p.numel()
        for p in student.parameters()
    )
)


# ================================================================
# 9. PHASE 2
#    Language Student
#
# Student learns:
#
#   Semantic -> ground truth language/action label
#   Object   -> ground truth object
#              + Object Teacher distillation
#
# State is NEVER produced.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 2: LANGUAGE STUDENT")
print("=" * 72)

student_epochs = 20

for epoch in range(student_epochs):

    student.train()

    total_sum = 0.0
    semantic_sum = 0.0
    object_sum = 0.0
    object_ce_sum = 0.0

    for batch in loader:

        tokens = batch["tokens"].to(device)
        visual = batch["visual"].to(device)

        semantic_label = batch[
            "semantic_label"
        ].to(device)

        object_label = batch[
            "object_label"
        ].to(device)

        student_optimizer.zero_grad()

        (
            z_sem_L,
            z_obj_L,
            semantic_logits,
            object_logits
        ) = student(tokens)

        # --------------------------------------------------------
        # Teacher object representation
        # --------------------------------------------------------

        with torch.no_grad():

            z_obj_V, _ = object_teacher(
                visual
            )

        # --------------------------------------------------------
        # Semantic supervision
        # --------------------------------------------------------

        semantic_ce = F.cross_entropy(
            semantic_logits,
            semantic_label
        )

        # --------------------------------------------------------
        # Object classification supervision
        # --------------------------------------------------------

        object_ce = F.cross_entropy(
            object_logits,
            object_label
        )

        # --------------------------------------------------------
        # Object distillation
        # --------------------------------------------------------

        object_distill = F.mse_loss(
            z_obj_L,
            z_obj_V
        )

        loss = (
            semantic_ce
            +
            object_ce
            +
            object_distill
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            1.0
        )

        student_optimizer.step()

        bs = tokens.size(0)

        total_sum += loss.item() * bs
        semantic_sum += semantic_ce.item() * bs
        object_sum += object_distill.item() * bs
        object_ce_sum += object_ce.item() * bs

    n = len(dataset)

    total = total_sum / n
    semantic = semantic_sum / n
    object_distill = object_sum / n
    object_ce = object_ce_sum / n

    print(
        f"Student Epoch {epoch+1:02d}/{student_epochs} | "
        f"Total={total:.6f} | "
        f"SemanticCE={semantic:.6f} | "
        f"ObjectCE={object_ce:.6f} | "
        f"ObjectDistill={object_distill:.6f}"
    )


# ================================================================
# Freeze Student
# ================================================================

for p in student.parameters():
    p.requires_grad = False

student.eval()

print("\nStudent FROZEN.")

print(
    "Trainable Student params:",
    sum(
        p.numel()
        for p in student.parameters()
        if p.requires_grad
    )
)


# ================================================================
# 10. Factorized VLA Fusion
#
# Input:
#
#   z_sem_L
#   z_obj_L
#   z_state_V
#
# No language state branch.
# ================================================================

class FactorizedVLA(nn.Module):

    def __init__(
        self,
        latent_dim=128,
        action_dim=7
    ):

        super().__init__()

        self.fusion = nn.Sequential(

            nn.Linear(
                latent_dim * 3,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                action_dim
            )
        )

    def forward(
        self,
        z_sem_L,
        z_obj_L,
        z_state_V
    ):

        x = torch.cat(
            [
                z_sem_L,
                z_obj_L,
                z_state_V
            ],
            dim=-1
        )

        return self.fusion(x)


vla = FactorizedVLA().to(device)

vla_optimizer = torch.optim.AdamW(
    vla.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nVLA Fusion parameters:",
    sum(
        p.numel()
        for p in vla.parameters()
    )
)


# ================================================================
# 11. PHASE 3
#     Factorized VLA Action Learning
# ================================================================

print("\n" + "=" * 72)
print("PHASE 3: FACTORIZED VLA")
print("=" * 72)

vla_epochs = 20

for epoch in range(vla_epochs):

    vla.train()

    action_sum = 0.0

    for batch in loader:

        visual = batch["visual"].to(device)
        tokens = batch["tokens"].to(device)
        action = batch["action"].to(device)

        vla_optimizer.zero_grad()

        with torch.no_grad():

            (
                z_sem_L,
                z_obj_L,
                _,
                _
            ) = student(tokens)

            z_state_V, _ = state_teacher(
                visual
            )

        action_pred = vla(
            z_sem_L,
            z_obj_L,
            z_state_V
        )

        action_loss = F.mse_loss(
            action_pred,
            action
        )

        action_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            vla.parameters(),
            1.0
        )

        vla_optimizer.step()

        action_sum += (
            action_loss.item()
            *
            tokens.size(0)
        )

    action_epoch = (
        action_sum / len(dataset)
    )

    print(
        f"VLA Epoch {epoch+1:02d}/{vla_epochs} | "
        f"Action={action_epoch:.6f}"
    )


# ================================================================
# 12. Architecture Check
# ================================================================

print("\n" + "=" * 72)
print("FINAL ARCHITECTURE CHECK")
print("=" * 72)

batch = next(iter(loader))

visual = batch["visual"].to(device)
tokens = batch["tokens"].to(device)

with torch.no_grad():

    z_obj_V, _ = object_teacher(
        visual
    )

    z_state_V, _ = state_teacher(
        visual
    )

    (
        z_sem_L,
        z_obj_L,
        _,
        _
    ) = student(tokens)

    action_pred = vla(
        z_sem_L,
        z_obj_L,
        z_state_V
    )

print(
    "Visual input:",
    tuple(visual.shape)
)

print(
    "Object Teacher z_obj_V:",
    tuple(z_obj_V.shape)
)

print(
    "State Teacher z_state_V:",
    tuple(z_state_V.shape)
)

print(
    "Language Student z_sem_L:",
    tuple(z_sem_L.shape)
)

print(
    "Language Student z_obj_L:",
    tuple(z_obj_L.shape)
)

print(
    "Language Student State:",
    "NONE"
)

print(
    "Final Action:",
    tuple(action_pred.shape)
)

print("\nFreeze check:")

print(
    "Object Teacher trainable:",
    sum(
        p.numel()
        for p in object_teacher.parameters()
        if p.requires_grad
    )
)

print(
    "State Teacher trainable:",
    sum(
        p.numel()
        for p in state_teacher.parameters()
        if p.requires_grad
    )
)

print(
    "Student trainable:",
    sum(
        p.numel()
        for p in student.parameters()
        if p.requires_grad
    )
)

print(
    "VLA trainable:",
    sum(
        p.numel()
        for p in vla.parameters()
        if p.requires_grad
    )
)


# ================================================================
# 13. Helper functions for evaluation
# ================================================================

def find_sample(
    instruction,
    preferred_index=0
):

    candidates = []

    for i, s in enumerate(dataset.samples):

        if s["instruction"] == instruction:
            candidates.append(i)

    return candidates[
        preferred_index % len(candidates)
    ]


def encode_instruction(instruction):

    tokens = torch.tensor(
        [tokenize(instruction)],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        (
            z_sem,
            z_obj,
            _,
            _
        ) = student(tokens)

    return z_sem, z_obj


def encode_state_from_sample(index):

    visual = torch.tensor(
        dataset.samples[index]["visual"],
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        z_state, _ = state_teacher(
            visual
        )

    return z_state


def predict_from_factors(
    instruction,
    state_index
):

    z_sem, z_obj = encode_instruction(
        instruction
    )

    z_state = encode_state_from_sample(
        state_index
    )

    with torch.no_grad():

        action = vla(
            z_sem,
            z_obj,
            z_state
        )

    return (
        z_sem,
        z_obj,
        z_state,
        action
    )


# ================================================================
# 14. TEST 1
#     SAME LANGUAGE / DIFFERENT STATE
# ================================================================

print("\n" + "=" * 72)
print("TEST 1: SAME LANGUAGE / DIFFERENT STATE")
print("=" * 72)

instruction = "push red cube right"

idx_a = find_sample(
    instruction,
    0
)

idx_b = find_sample(
    instruction,
    1
)

sample_a = dataset.samples[idx_a]
sample_b = dataset.samples[idx_b]

print("\nInstruction:", instruction)

print(
    "\nScene A red xyz:",
    sample_a["positions"]["red"]
)

print(
    "Scene B red xyz:",
    sample_b["positions"]["red"]
)

(
    sem_a,
    obj_a,
    state_a,
    action_a
) = predict_from_factors(
    instruction,
    idx_a
)

(
    sem_b,
    obj_b,
    state_b,
    action_b
) = predict_from_factors(
    instruction,
    idx_b
)

semantic_difference = torch.mean(
    torch.abs(
        sem_a - sem_b
    )
).item()

object_difference = torch.mean(
    torch.abs(
        obj_a - obj_b
    )
).item()

state_difference = torch.mean(
    torch.abs(
        state_a - state_b
    )
).item()

action_difference = torch.mean(
    torch.abs(
        action_a - action_b
    )
).item()

print(
    f"\nLanguage Semantic difference: "
    f"{semantic_difference:.8f}"
)

print(
    f"Language Object difference: "
    f"{object_difference:.8f}"
)

print(
    f"Visual State difference: "
    f"{state_difference:.8f}"
)

print(
    f"Final Action difference: "
    f"{action_difference:.8f}"
)

print("\nExpected:")
print("  Language Semantic -> SMALL")
print("  Language Object   -> SMALL")
print("  Visual State      -> LARGE")
print("  Action            -> LARGE")


# ================================================================
# 15. TEST 2
#     STATE SHUFFLE
# ================================================================

print("\n" + "=" * 72)
print("TEST 2: STATE SHUFFLE ABLATION")
print("=" * 72)

print(
    "\nSame Language:",
    instruction
)

print(
    "Correct State:",
    sample_a["positions"]["red"]
)

print(
    "Shuffled State:",
    sample_b["positions"]["red"]
)

print("\nCorrect-state action:")

print(
    action_a.squeeze(0)
    .cpu()
    .numpy()
)

print("\nShuffled-state action:")

print(
    action_b.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nAction difference after State Shuffle:",
    f"{action_difference:.8f}"
)


# ================================================================
# 16. TEST 3
#     SEMANTIC SHUFFLE
# ================================================================

print("\n" + "=" * 72)
print("TEST 3: SEMANTIC SHUFFLE ABLATION")
print("=" * 72)

instruction_a = "push red cube right"
instruction_b = "push red cube left"

(
    sem_a,
    obj_a,
    state_fixed,
    action_sem_a
) = predict_from_factors(
    instruction_a,
    idx_a
)

(
    sem_b,
    obj_b,
    _,
    action_sem_b
) = predict_from_factors(
    instruction_b,
    idx_a
)

semantic_difference = torch.mean(
    torch.abs(
        sem_a - sem_b
    )
).item()

object_difference = torch.mean(
    torch.abs(
        obj_a - obj_b
    )
).item()

semantic_action_difference = torch.mean(
    torch.abs(
        action_sem_a - action_sem_b
    )
).item()

print(
    "\nScene red xyz:",
    sample_a["positions"]["red"]
)

print(
    "\nLanguage A:",
    instruction_a
)

print(
    "Language B:",
    instruction_b
)

print("\nAction A:")

print(
    action_sem_a.squeeze(0)
    .cpu()
    .numpy()
)

print("\nAction B:")

print(
    action_sem_b.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nSemantic latent difference:",
    f"{semantic_difference:.8f}"
)

print(
    "Object latent difference:",
    f"{object_difference:.8f}"
)

print(
    "Action difference after Semantic Shuffle:",
    f"{semantic_action_difference:.8f}"
)

print("\nExpected:")
print("  Semantic = changed")
print("  Object   = unchanged")
print("  State    = unchanged")
print("  Action   = changed")


# ================================================================
# 17. TEST 4
#     OBJECT SHUFFLE
#
# Same semantic + same spatial state,
# but language object changes.
# ================================================================

print("\n" + "=" * 72)
print("TEST 4: OBJECT SHUFFLE ABLATION")
print("=" * 72)

instruction_obj_a = "push red cube right"
instruction_obj_b = "push blue cube right"

idx_state = idx_a

(
    sem_obj_a,
    obj_lat_a,
    state_obj_a,
    action_obj_a
) = predict_from_factors(
    instruction_obj_a,
    idx_state
)

(
    sem_obj_b,
    obj_lat_b,
    state_obj_b,
    action_obj_b
) = predict_from_factors(
    instruction_obj_b,
    idx_state
)

object_latent_difference = torch.mean(
    torch.abs(
        obj_lat_a - obj_lat_b
    )
).item()

semantic_latent_difference = torch.mean(
    torch.abs(
        sem_obj_a - sem_obj_b
    )
).item()

object_action_difference = torch.mean(
    torch.abs(
        action_obj_a - action_obj_b
    )
).item()

print(
    "\nLanguage A:",
    instruction_obj_a
)

print(
    "Language B:",
    instruction_obj_b
)

print(
    "\nSemantic difference:",
    f"{semantic_latent_difference:.8f}"
)

print(
    "Object difference:",
    f"{object_latent_difference:.8f}"
)

print(
    "State difference:",
    f"{torch.mean(torch.abs(state_obj_a - state_obj_b)).item():.8f}"
)

print(
    "Action difference:",
    f"{object_action_difference:.8f}"
)

print("\nExpected:")
print("  Semantic -> SMALL")
print("  Object   -> LARGE")
print("  State    -> SMALL")
print("  Action   -> LARGE")


# ================================================================
# 18. TEST 5
#     CROSS-FACTOR RECOMBINATION
#
# Same semantic/object language representation,
# different visual state.
# ================================================================

print("\n" + "=" * 72)
print("TEST 5: CROSS-FACTOR RECOMBINATION")
print("=" * 72)

instruction_cross = "push red cube right"

idx_cross_a = find_sample(
    instruction_cross,
    2
)

idx_cross_b = find_sample(
    instruction_cross,
    3
)

(
    sem_cross,
    obj_cross,
    state_cross_a,
    action_cross_a
) = predict_from_factors(
    instruction_cross,
    idx_cross_a
)

_, _, state_cross_b, action_cross_b = (
    predict_from_factors(
        instruction_cross,
        idx_cross_b
    )
)

print(
    "\nInstruction:",
    instruction_cross
)

print(
    "\nState A:",
    dataset.samples[
        idx_cross_a
    ]["positions"]["red"]
)

print(
    "State B:",
    dataset.samples[
        idx_cross_b
    ]["positions"]["red"]
)

print(
    "\nAction using Language A + State A:"
)

print(
    action_cross_a.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nAction using SAME Language A + State B:"
)

print(
    action_cross_b.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nCross-factor action difference:",
    f"{torch.mean(torch.abs(action_cross_a - action_cross_b)).item():.8f}"
)


# ================================================================
# 19. FINAL REPORT
# ================================================================

print("\n" + "=" * 72)
print("FINAL REPORT")
print("=" * 72)

print(
    "\nTeacher design:"
)

print(
    "  Object Teacher : Vision -> z_obj_V"
)

print(
    "  State Teacher  : Vision -> z_state_V"
)

print(
    "  Semantic Teacher: NONE"
)

print(
    "\nStudent design:"
)

print(
    "  Semantic Student: Language -> z_sem_L"
)

print(
    "  Object Student  : Language -> z_obj_L"
)

print(
    "  State Student   : NONE"
)

print(
    "\nFinal VLA:"
)

print(
    "  z_sem_L + z_obj_L + z_state_V -> Action"
)

print(
    "\nAblation summary:"
)

print(
    f"  State Shuffle Action Δ    : "
    f"{action_difference:.8f}"
)

print(
    f"  Semantic Shuffle Action Δ : "
    f"{semantic_action_difference:.8f}"
)

print(
    f"  Object Shuffle Action Δ   : "
    f"{object_action_difference:.8f}"
)

print(
    f"  Cross-factor State Δ      : "
    f"{torch.mean(torch.abs(action_cross_a - action_cross_b)).item():.8f}"
)

print("\n" + "=" * 72)
print("Experiment 4 finished successfully.")
print("Separated Object/State Teacher hypothesis tested.")
print("=" * 72)


AntEncoder Experiment 4
Separated Object Teacher + State Teacher + Language Student
Device: cpu

Vocabulary size: 14
Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'blue': 2, 'cube': 3, 'down': 4, 'green': 5, 'left': 6, 'pick': 7, 'place': 8, 'push': 9, 'red': 10, 'right': 11, 'the': 12, 'up': 13}

Dataset size: 2400

Position randomization check:
pick up the red cube -> [-0.8748578  -0.78102535  0.41197973]
pick up the red cube -> [-0.9158671 -0.1905032  0.9068788]
pick up the red cube -> [-0.5272529   0.00121372  0.7013717 ]

Teacher parameters: 337286

PHASE 1: SEPARATED PHYSICAL TEACHERS
Teacher Epoch 01/80 | Total=1.303910 | Object=1.114242 | State=0.189669
Teacher Epoch 02/80 | Total=1.255261 | Object=1.095213 | State=0.160048
Teacher Epoch 03/80 | Total=1.250702 | Object=1.092848 | State=0.157854
Teacher Epoch 04/80 | Total=1.248385 | Object=1.092459 | State=0.155925
Teacher Epoch 05/80 | Total=1.245944 | Object=1.091418 | State=0.154525
Teacher Epoch 10/80 | Total=1.235558 | Object=1.083

In [ ]:
# ================================================================
# AntEncoder Experiment 5
# Direct Teacher Factorization / Latent Separability Test
#
# Uses the already-trained Experiment 4 models:
#
#   object_teacher : Vision -> z_obj_V
#   state_teacher  : Vision -> z_state_V
#
# Teacher parameters remain frozen.
#
# Main questions:
#
#   1. Does Object Teacher actually encode OBJECT?
#   2. Does State Teacher actually encode STATE?
#   3. Does Object latent ignore STATE?
#   4. Does State latent ignore OBJECT?
# ================================================================

print("\n")
print("=" * 72)
print("AntEncoder Experiment 5")
print("Direct Teacher Factorization / Latent Separability Test")
print("=" * 72)


# ================================================================
# 1. Teacher Accuracy
# ================================================================

object_teacher.eval()
state_teacher.eval()

correct = 0
total = 0

state_mse_sum = 0.0
state_mae_sum = 0.0

all_obj_latents = []
all_state_latents = []
all_object_labels = []
all_states = []

with torch.no_grad():

    for batch in loader:

        visual = batch["visual"].to(device)

        object_label = batch[
            "object_label"
        ].to(device)

        target_xyz = batch[
            "target_xyz"
        ].to(device)

        # --------------------------------------------------------
        # Object Teacher
        # --------------------------------------------------------

        z_obj, object_logits = object_teacher(
            visual
        )

        prediction = torch.argmax(
            object_logits,
            dim=-1
        )

        correct += (
            prediction == object_label
        ).sum().item()

        total += visual.size(0)

        # --------------------------------------------------------
        # State Teacher
        # --------------------------------------------------------

        z_state, state_pred = state_teacher(
            visual
        )

        state_mse_sum += (
            F.mse_loss(
                state_pred,
                target_xyz,
                reduction="sum"
            ).item()
        )

        state_mae_sum += (
            F.l1_loss(
                state_pred,
                target_xyz,
                reduction="sum"
            ).item()
        )

        all_obj_latents.append(
            z_obj.cpu()
        )

        all_state_latents.append(
            z_state.cpu()
        )

        all_object_labels.append(
            object_label.cpu()
        )

        all_states.append(
            target_xyz.cpu()
        )


object_accuracy = (
    correct / total
)

state_mse = (
    state_mse_sum
    /
    (total * 3)
)

state_mae = (
    state_mae_sum
    /
    (total * 3)
)

all_obj_latents = torch.cat(
    all_obj_latents,
    dim=0
)

all_state_latents = torch.cat(
    all_state_latents,
    dim=0
)

all_object_labels = torch.cat(
    all_object_labels,
    dim=0
)

all_states = torch.cat(
    all_states,
    dim=0
)


print("\n" + "-" * 72)
print("TEACHER PERFORMANCE")
print("-" * 72)

print(
    f"Object Teacher Accuracy : "
    f"{object_accuracy:.6f}"
)

print(
    f"State Teacher MSE       : "
    f"{state_mse:.6f}"
)

print(
    f"State Teacher MAE       : "
    f"{state_mae:.6f}"
)

print("\nReference:")
print(
    "Random 3-class Object Accuracy ~= 0.333"
)


# ================================================================
# 2. Latent Distance Helper
# ================================================================

def latent_distance(a, b):

    return torch.mean(
        torch.abs(a - b)
    ).item()


def euclidean_distance(a, b):

    return torch.mean(
        (a - b) ** 2
    ).sqrt().item()


# ================================================================
# 3. Build Controlled Counterfactual Visual Inputs
#
# We construct physical observations directly.
#
# Object factor:
#   RGB arrangement
#
# State factor:
#   XYZ positions
#
# Because the visual projection is fixed and known,
# these counterfactual scenes remain valid visual inputs.
# ================================================================

projection = torch.tensor(
    dataset.visual_projection,
    dtype=torch.float32,
    device=device
)


def make_physical_visual(
    red_xyz,
    blue_xyz,
    green_xyz
):

    colors = {
        "red": np.array(
            [1.0, 0.0, 0.0],
            dtype=np.float32
        ),

        "blue": np.array(
            [0.0, 0.0, 1.0],
            dtype=np.float32
        ),

        "green": np.array(
            [0.0, 1.0, 0.0],
            dtype=np.float32
        )
    }

    positions = {
        "red": np.asarray(
            red_xyz,
            dtype=np.float32
        ),

        "blue": np.asarray(
            blue_xyz,
            dtype=np.float32
        ),

        "green": np.asarray(
            green_xyz,
            dtype=np.float32
        )
    }

    physical = []

    for color in [
        "red",
        "blue",
        "green"
    ]:

        physical.extend(
            colors[color]
        )

        physical.extend(
            positions[color]
        )

    return np.asarray(
        physical,
        dtype=np.float32
    )


def make_visual(
    red_xyz,
    blue_xyz,
    green_xyz,
    noise=False
):

    physical = make_physical_visual(
        red_xyz,
        blue_xyz,
        green_xyz
    )

    physical_tensor = torch.tensor(
        physical,
        dtype=torch.float32,
        device=device
    )

    visual = (
        physical_tensor
        @ projection
    )

    if noise:

        visual = (
            visual
            +
            torch.randn_like(visual)
            * 0.02
        )

    return visual.unsqueeze(0)


# ================================================================
# 4. Controlled Scenes
# ================================================================

state_A = {
    "red":   np.array(
        [0.70, 0.20, 0.80],
        dtype=np.float32
    ),

    "blue":  np.array(
        [-0.40, 0.10, 0.60],
        dtype=np.float32
    ),

    "green": np.array(
        [0.20, -0.70, 0.50],
        dtype=np.float32
    )
}

state_B = {
    "red":   np.array(
        [-0.60, -0.50, 0.45],
        dtype=np.float32
    ),

    "blue":  state_A["blue"].copy(),

    "green": state_A["green"].copy()
}


# ---------------------------------------------------------------
# Scene 1:
#
# Same object arrangement
# Same state
# ---------------------------------------------------------------

visual_A = make_visual(
    state_A["red"],
    state_A["blue"],
    state_A["green"]
)


# ---------------------------------------------------------------
# Scene 2:
#
# Same objects
# Different RED state
# ---------------------------------------------------------------

visual_B = make_visual(
    state_B["red"],
    state_B["blue"],
    state_B["green"]
)


# ================================================================
# 5. SAME OBJECT / DIFFERENT STATE
#
# Desired:
#
#   Object latent -> SMALL change
#   State latent  -> LARGE change
# ================================================================

with torch.no_grad():

    obj_A, _ = object_teacher(
        visual_A
    )

    state_A_lat, _ = state_teacher(
        visual_A
    )

    obj_B, _ = object_teacher(
        visual_B
    )

    state_B_lat, _ = state_teacher(
        visual_B
    )


same_object_different_state_obj = latent_distance(
    obj_A,
    obj_B
)

same_object_different_state_state = latent_distance(
    state_A_lat,
    state_B_lat
)


print("\n" + "=" * 72)
print("TEST 1: SAME OBJECT / DIFFERENT STATE")
print("=" * 72)

print(
    "\nObject latent difference:",
    f"{same_object_different_state_obj:.8f}"
)

print(
    "State latent difference:",
    f"{same_object_different_state_state:.8f}"
)

print("\nExpected:")
print("  Object latent -> SMALL")
print("  State latent  -> LARGE")


# ================================================================
# 6. SAME STATE / DIFFERENT OBJECT
#
# We alter the RGB/object identity while preserving XYZ.
#
# This is a counterfactual scene:
#
#   original:
#       red   @ state_A[red]
#       blue  @ state_A[blue]
#
#   swapped:
#       blue  @ state_A[red]
#       red   @ state_A[blue]
#
# Physical positions remain unchanged.
# Object identities are exchanged.
# ================================================================

visual_object_swapped = make_visual(
    red_xyz=state_A["blue"],
    blue_xyz=state_A["red"],
    green_xyz=state_A["green"]
)

with torch.no_grad():

    obj_swap, _ = object_teacher(
        visual_object_swapped
    )

    state_swap, _ = state_teacher(
        visual_object_swapped
    )


different_object_same_state_obj = latent_distance(
    obj_A,
    obj_swap
)

different_object_same_state_state = latent_distance(
    state_A_lat,
    state_swap
)


print("\n" + "=" * 72)
print("TEST 2: DIFFERENT OBJECT / SAME SPATIAL STATE")
print("=" * 72)

print(
    "\nObject latent difference:",
    f"{different_object_same_state_obj:.8f}"
)

print(
    "State latent difference:",
    f"{different_object_same_state_state:.8f}"
)

print("\nExpected:")
print("  Object latent -> LARGE")
print("  State latent  -> SMALL")


# ================================================================
# 7. Object Teacher Classification on Controlled Scenes
# ================================================================

with torch.no_grad():

    _, logits_A = object_teacher(
        visual_A
    )

    _, logits_swap = object_teacher(
        visual_object_swapped
    )

    pred_A = torch.argmax(
        logits_A,
        dim=-1
    ).item()

    pred_swap = torch.argmax(
        logits_swap,
        dim=-1
    ).item()


color_names = [
    "red",
    "blue",
    "green"
]


print("\n" + "=" * 72)
print("TEST 3: OBJECT IDENTITY PREDICTION")
print("=" * 72)

print(
    "Scene A predicted object:",
    color_names[pred_A]
)

print(
    "Swapped scene predicted object:",
    color_names[pred_swap]
)


# ================================================================
# 8. State Teacher Prediction
#
# We inspect whether State Teacher tracks the RED target.
#
# IMPORTANT:
# The teacher was trained with target_xyz as supervision.
#
# Therefore the state representation is expected to follow
# the supervised target state.
# ================================================================

with torch.no_grad():

    _, pred_state_A = state_teacher(
        visual_A
    )

    _, pred_state_B = state_teacher(
        visual_B
    )

    _, pred_state_swap = state_teacher(
        visual_object_swapped
    )


print("\n" + "=" * 72)
print("TEST 4: STATE PREDICTION")
print("=" * 72)

print(
    "\nTarget state A:",
    state_A["red"]
)

print(
    "Predicted state A:",
    pred_state_A.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nTarget state B:",
    state_B["red"]
)

print(
    "Predicted state B:",
    pred_state_B.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nPredicted state after object swap:",
    pred_state_swap.squeeze(0)
    .cpu()
    .numpy()
)


# ================================================================
# 9. Per-Factor Ratio
#
# Useful because raw latent distances alone are difficult to
# interpret.
#
# If factorization is good:
#
#   Object sensitivity to object
#       >>
#   Object sensitivity to state
#
#   State sensitivity to state
#       >>
#   State sensitivity to object
# ================================================================

eps = 1e-8

object_selectivity_ratio = (
    different_object_same_state_obj
    /
    (
        same_object_different_state_obj
        + eps
    )
)

state_selectivity_ratio = (
    same_object_different_state_state
    /
    (
        different_object_same_state_state
        + eps
    )
)


print("\n" + "=" * 72)
print("TEST 5: FACTOR SELECTIVITY RATIOS")
print("=" * 72)

print(
    "\nObject selectivity ratio:",
    f"{object_selectivity_ratio:.6f}"
)

print(
    "State selectivity ratio:",
    f"{state_selectivity_ratio:.6f}"
)

print("\nInterpretation:")
print(
    "  Object ratio >> 1  => Object latent is more sensitive"
)
print(
    "                         to object than state."
)

print(
    "  State ratio >> 1   => State latent is more sensitive"
)

print(
    "                         to state than object."
)


# ================================================================
# 10. Latent Norms
#
# Prevent a misleading case where a latent difference appears
# small simply because the entire latent has collapsed.
# ================================================================

obj_norm = torch.mean(
    torch.abs(obj_A)
).item()

state_norm = torch.mean(
    torch.abs(state_A_lat)
).item()


print("\n" + "=" * 72)
print("TEST 6: LATENT MAGNITUDES")
print("=" * 72)

print(
    "Mean |Object latent|:",
    f"{obj_norm:.8f}"
)

print(
    "Mean |State latent| :",
    f"{state_norm:.8f}"
)


# ================================================================
# 11. FINAL EXPERIMENT 5 REPORT
# ================================================================

print("\n" + "=" * 72)
print("EXPERIMENT 5 FINAL REPORT")
print("=" * 72)

print("\nTeacher performance:")

print(
    f"  Object Accuracy : "
    f"{object_accuracy:.6f}"
)

print(
    f"  State MSE       : "
    f"{state_mse:.6f}"
)

print(
    f"  State MAE       : "
    f"{state_mae:.6f}"
)

print("\nLatent separation:")

print(
    f"  Same Object / Different State:"
)

print(
    f"    Object Δ = "
    f"{same_object_different_state_obj:.8f}"
)

print(
    f"    State Δ  = "
    f"{same_object_different_state_state:.8f}"
)

print(
    f"\n  Different Object / Same State:"
)

print(
    f"    Object Δ = "
    f"{different_object_same_state_obj:.8f}"
)

print(
    f"    State Δ  = "
    f"{different_object_same_state_state:.8f}"
)

print("\nSelectivity:")

print(
    f"  Object selectivity ratio = "
    f"{object_selectivity_ratio:.6f}"
)

print(
    f"  State selectivity ratio  = "
    f"{state_selectivity_ratio:.6f}"
)

print("\n" + "=" * 72)
print("Experiment 5 finished.")
print("Direct Object/State Teacher factorization tested.")
print("=" * 72)




AntEncoder Experiment 5
Direct Teacher Factorization / Latent Separability Test

------------------------------------------------------------------------
TEACHER PERFORMANCE
------------------------------------------------------------------------
Object Teacher Accuracy : 0.475833
State Teacher MSE       : 0.036331
State Teacher MAE       : 0.147148

Reference:
Random 3-class Object Accuracy ~= 0.333

TEST 1: SAME OBJECT / DIFFERENT STATE

Object latent difference: 0.11881666
State latent difference: 0.83360863

Expected:
  Object latent -> SMALL
  State latent  -> LARGE

TEST 2: DIFFERENT OBJECT / SAME SPATIAL STATE

Object latent difference: 0.07196171
State latent difference: 0.59479839

Expected:
  Object latent -> LARGE
  State latent  -> SMALL

TEST 3: OBJECT IDENTITY PREDICTION
Scene A predicted object: red
Swapped scene predicted object: red

TEST 4: STATE PREDICTION

Target state A: [0.7 0.2 0.8]
Predicted state A: [0.00332322 0.03903863 0.74439955]

Target state B: [-0.6  -

In [ ]:
# ================================================================
# AntEncoder Experiment 6A
# Two-Object Minimal Factorization Test
#
# Goal:
#   Object Teacher = WHAT
#   State Teacher  = WHERE
#
# IMPORTANT:
#   Object Teacher receives Vision only.
#   State Teacher receives Vision only.
#   Language is NOT used in Phase 1.
#
# The experiment deliberately uses TWO objects first,
# so failure can be diagnosed before increasing complexity.
# ================================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ================================================================
# 0. Reproducibility
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("AntEncoder Experiment 6A")
print("Two-Object Minimal Factorization Test")
print("=" * 72)
print("Device:", device)


# ================================================================
# 1. Object vocabulary
#
# Two objects:
#
#   red cube
#   blue elongated
#
# We intentionally separate COLOR and SHAPE.
# ================================================================

COLORS = ["red", "blue"]
SHAPES = ["cube", "elongated"]

COLOR_RGB = {
    "red": np.array(
        [1.0, 0.0, 0.0],
        dtype=np.float32
    ),

    "blue": np.array(
        [0.0, 0.0, 1.0],
        dtype=np.float32
    ),
}

SHAPE_CODE = {
    "cube": np.array(
        [1.0, 0.0],
        dtype=np.float32
    ),

    "elongated": np.array(
        [0.0, 1.0],
        dtype=np.float32
    ),
}


# ================================================================
# 2. Dataset
#
# Every scene contains TWO objects.
#
# Each object:
#
#   RGB  = appearance
#   shape = object attribute
#   XYZ  = spatial state
#
# Physical input per object:
#   3 RGB + 2 shape + 3 XYZ = 8
#
# Two objects:
#   16 dimensions
#
# Then fixed random projection -> 512D.
# ================================================================

class TwoObjectDataset(Dataset):

    def __init__(
        self,
        n_samples=4000,
        seed=42
    ):

        self.n_samples = n_samples

        self.rng = np.random.default_rng(seed)

        # --------------------------------------------------------
        # Fixed projection.
        #
        # The projection itself contains no trainable information.
        # --------------------------------------------------------

        projection_rng = np.random.default_rng(12345)

        self.projection = (
            projection_rng.normal(
                0.0,
                1.0 / np.sqrt(16),
                size=(16, 512)
            )
            .astype(np.float32)
        )

        self.samples = []

        for i in range(n_samples):

            # ----------------------------------------------------
            # Randomly choose one of four combinations.
            #
            # red cube
            # red elongated
            # blue cube
            # blue elongated
            #
            # This prevents object identity from being equivalent
            # to a single color.
            # ----------------------------------------------------

            object_a_color = COLORS[
                self.rng.integers(0, 2)
            ]

            object_a_shape = SHAPES[
                self.rng.integers(0, 2)
            ]

            # Force object B to be different in at least one
            # attribute.
            object_b_color = COLORS[
                self.rng.integers(0, 2)
            ]

            object_b_shape = SHAPES[
                self.rng.integers(0, 2)
            ]

            # ----------------------------------------------------
            # Random positions.
            #
            # Position is independent from object identity.
            # ----------------------------------------------------

            xyz_a = self.rng.uniform(
                low=[-1.0, -1.0, 0.35],
                high=[1.0, 1.0, 1.0],
                size=3
            ).astype(np.float32)

            xyz_b = self.rng.uniform(
                low=[-1.0, -1.0, 0.35],
                high=[1.0, 1.0, 1.0],
                size=3
            ).astype(np.float32)

            # ----------------------------------------------------
            # Physical representation.
            #
            # Object A:
            #   RGB + Shape + XYZ
            #
            # Object B:
            #   RGB + Shape + XYZ
            # ----------------------------------------------------

            object_a = np.concatenate([
                COLOR_RGB[object_a_color],
                SHAPE_CODE[object_a_shape],
                xyz_a
            ])

            object_b = np.concatenate([
                COLOR_RGB[object_b_color],
                SHAPE_CODE[object_b_shape],
                xyz_b
            ])

            physical = np.concatenate([
                object_a,
                object_b
            ]).astype(np.float32)

            # ----------------------------------------------------
            # Fixed projection + noise
            # ----------------------------------------------------

            visual = (
                physical @ self.projection
            )

            visual += self.rng.normal(
                0.0,
                0.01,
                size=512
            ).astype(np.float32)

            # ----------------------------------------------------
            # Object labels.
            #
            # We encode the full object attribute combination:
            #
            #   color × shape
            #
            # 0 = red cube
            # 1 = red elongated
            # 2 = blue cube
            # 3 = blue elongated
            # ----------------------------------------------------

            object_classes = {
                ("red", "cube"): 0,
                ("red", "elongated"): 1,
                ("blue", "cube"): 2,
                ("blue", "elongated"): 3,
            }

            label_a = object_classes[
                (object_a_color, object_a_shape)
            ]

            label_b = object_classes[
                (object_b_color, object_b_shape)
            ]

            # ----------------------------------------------------
            # Store
            # ----------------------------------------------------

            self.samples.append({

                "visual": visual.astype(
                    np.float32
                ),

                "physical": physical,

                "object_a_color": object_a_color,
                "object_a_shape": object_a_shape,

                "object_b_color": object_b_color,
                "object_b_shape": object_b_shape,

                "xyz_a": xyz_a,
                "xyz_b": xyz_b,

                "object_label_a": label_a,
                "object_label_b": label_b,

            })

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        s = self.samples[idx]

        return {

            "visual": torch.tensor(
                s["visual"],
                dtype=torch.float32
            ),

            "xyz_a": torch.tensor(
                s["xyz_a"],
                dtype=torch.float32
            ),

            "xyz_b": torch.tensor(
                s["xyz_b"],
                dtype=torch.float32
            ),

            "object_label_a": torch.tensor(
                s["object_label_a"],
                dtype=torch.long
            ),

            "object_label_b": torch.tensor(
                s["object_label_b"],
                dtype=torch.long
            ),
        }


dataset = TwoObjectDataset(
    n_samples=4000,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

print("\nDataset size:", len(dataset))


# ================================================================
# 3. Object Teacher
#
# Vision -> Object latent
#
# IMPORTANT:
#   State is NOT explicitly given to the object head.
#
# We want the latent to represent WHAT the objects are.
# ================================================================

class ObjectTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        latent_dim=128
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                latent_dim
            ),

            nn.Tanh()
        )

        self.classifier_a = nn.Linear(
            latent_dim,
            4
        )

        self.classifier_b = nn.Linear(
            latent_dim,
            4
        )

    def forward(self, visual):

        z_obj = self.encoder(
            visual
        )

        logits_a = self.classifier_a(
            z_obj
        )

        logits_b = self.classifier_b(
            z_obj
        )

        return (
            z_obj,
            logits_a,
            logits_b
        )


# ================================================================
# 4. State Teacher
#
# Vision -> State latent
#
# Predict BOTH object positions.
# ================================================================

class StateTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        latent_dim=128
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                latent_dim
            ),

            nn.Tanh()
        )

        self.decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                6
            )
        )

    def forward(self, visual):

        z_state = self.encoder(
            visual
        )

        state = self.decoder(
            z_state
        )

        return (
            z_state,
            state
        )


object_teacher = ObjectTeacher().to(
    device
)

state_teacher = StateTeacher().to(
    device
)

teacher_parameters = (
    list(object_teacher.parameters())
    +
    list(state_teacher.parameters())
)

optimizer = torch.optim.AdamW(
    teacher_parameters,
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nTeacher parameters:",
    sum(
        p.numel()
        for p in teacher_parameters
    )
)


# ================================================================
# 5. PHASE 1
# ================================================================

print("\n" + "=" * 72)
print("PHASE 1: OBJECT + STATE TEACHER")
print("=" * 72)

epochs = 100

for epoch in range(epochs):

    object_teacher.train()
    state_teacher.train()

    total_sum = 0.0
    object_sum = 0.0
    state_sum = 0.0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        xyz_a = batch[
            "xyz_a"
        ].to(device)

        xyz_b = batch[
            "xyz_b"
        ].to(device)

        label_a = batch[
            "object_label_a"
        ].to(device)

        label_b = batch[
            "object_label_b"
        ].to(device)

        optimizer.zero_grad()

        # --------------------------------------------------------
        # Object Teacher
        # --------------------------------------------------------

        (
            z_obj,
            logits_a,
            logits_b
        ) = object_teacher(
            visual
        )

        # --------------------------------------------------------
        # State Teacher
        # --------------------------------------------------------

        (
            z_state,
            state_pred
        ) = state_teacher(
            visual
        )

        state_target = torch.cat(
            [
                xyz_a,
                xyz_b
            ],
            dim=-1
        )

        # --------------------------------------------------------
        # Losses
        # --------------------------------------------------------

        object_loss = (
            F.cross_entropy(
                logits_a,
                label_a
            )
            +
            F.cross_entropy(
                logits_b,
                label_b
            )
        ) / 2.0

        state_loss = F.mse_loss(
            state_pred,
            state_target
        )

        loss = (
            object_loss
            +
            state_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            teacher_parameters,
            1.0
        )

        optimizer.step()

        bs = visual.size(0)

        total_sum += (
            loss.item() * bs
        )

        object_sum += (
            object_loss.item() * bs
        )

        state_sum += (
            state_loss.item() * bs
        )

    n = len(dataset)

    total = total_sum / n
    object_loss = object_sum / n
    state_loss = state_sum / n

    if (
        epoch < 5
        or (epoch + 1) % 10 == 0
        or epoch == epochs - 1
    ):

        print(
            f"Epoch {epoch+1:03d}/{epochs} | "
            f"Total={total:.6f} | "
            f"Object={object_loss:.6f} | "
            f"State={state_loss:.6f}"
        )


# ================================================================
# 6. Freeze
# ================================================================

for p in object_teacher.parameters():
    p.requires_grad = False

for p in state_teacher.parameters():
    p.requires_grad = False

object_teacher.eval()
state_teacher.eval()

print("\nTeachers FROZEN.")


# ================================================================
# 7. Helper
# ================================================================

def encode(index):

    visual = torch.tensor(
        dataset.samples[index]["visual"],
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        z_obj, logits_a, logits_b = (
            object_teacher(visual)
        )

        z_state, state_pred = (
            state_teacher(visual)
        )

    return (
        z_obj,
        z_state,
        logits_a,
        logits_b,
        state_pred
    )


# ================================================================
# 8. TEST 1
#
# SAME OBJECTS
# DIFFERENT POSITIONS
#
# Object should remain stable.
# State should change.
# ================================================================

print("\n" + "=" * 72)
print("TEST 1: SAME OBJECTS / DIFFERENT STATE")
print("=" * 72)

base = dataset.samples[0]

# ---------------------------------------------------------------
# Construct a paired scene manually.
# Same objects, new positions.
# ---------------------------------------------------------------

def make_scene(
    object_a_color,
    object_a_shape,
    object_b_color,
    object_b_shape,
    xyz_a,
    xyz_b,
    noise_seed=999
):

    physical = np.concatenate([

        COLOR_RGB[object_a_color],
        SHAPE_CODE[object_a_shape],
        xyz_a,

        COLOR_RGB[object_b_color],
        SHAPE_CODE[object_b_shape],
        xyz_b,

    ]).astype(np.float32)

    visual = (
        physical @ dataset.projection
    )

    rng = np.random.default_rng(
        noise_seed
    )

    visual += rng.normal(
        0.0,
        0.01,
        size=512
    ).astype(np.float32)

    return visual.astype(
        np.float32
    )


visual_A = make_scene(
    "red",
    "cube",
    "blue",
    "elongated",
    np.array(
        [0.7, 0.2, 0.8],
        dtype=np.float32
    ),
    np.array(
        [-0.6, -0.5, 0.45],
        dtype=np.float32
    ),
    noise_seed=1
)

visual_B = make_scene(
    "red",
    "cube",
    "blue",
    "elongated",
    np.array(
        [-0.4, 0.8, 0.5],
        dtype=np.float32
    ),
    np.array(
        [0.8, -0.2, 0.9],
        dtype=np.float32
    ),
    noise_seed=2
)

with torch.no_grad():

    z_obj_A, _, _ = object_teacher(
        torch.tensor(
            visual_A,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )

    z_obj_B, _, _ = object_teacher(
        torch.tensor(
            visual_B,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )

    z_state_A, _ = state_teacher(
        torch.tensor(
            visual_A,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )

    z_state_B, _ = state_teacher(
        torch.tensor(
            visual_B,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )

same_object_object_delta = torch.mean(
    torch.abs(
        z_obj_A - z_obj_B
    )
).item()

same_object_state_delta = torch.mean(
    torch.abs(
        z_state_A - z_state_B
    )
).item()

print(
    "Object latent difference:",
    f"{same_object_object_delta:.8f}"
)

print(
    "State latent difference:",
    f"{same_object_state_delta:.8f}"
)

print("\nExpected:")
print("  Object -> SMALL")
print("  State  -> LARGE")


# ================================================================
# 9. TEST 2
#
# SAME POSITIONS
# DIFFERENT OBJECTS
#
# Object should change.
# State should remain stable.
# ================================================================

print("\n" + "=" * 72)
print("TEST 2: DIFFERENT OBJECTS / SAME STATE")
print("=" * 72)

visual_C = make_scene(
    "red",
    "cube",
    "blue",
    "elongated",
    np.array(
        [0.4, 0.2, 0.7],
        dtype=np.float32
    ),
    np.array(
        [-0.5, -0.4, 0.5],
        dtype=np.float32
    ),
    noise_seed=3
)

visual_D = make_scene(
    "blue",
    "elongated",
    "red",
    "cube",
    np.array(
        [0.4, 0.2, 0.7],
        dtype=np.float32
    ),
    np.array(
        [-0.5, -0.4, 0.5],
        dtype=np.float32
    ),
    noise_seed=4
)

with torch.no_grad():

    z_obj_C, _, _ = object_teacher(
        torch.tensor(
            visual_C,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )

    z_obj_D, _, _ = object_teacher(
        torch.tensor(
            visual_D,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )

    z_state_C, _ = state_teacher(
        torch.tensor(
            visual_C,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )

    z_state_D, _ = state_teacher(
        torch.tensor(
            visual_D,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )

different_object_object_delta = torch.mean(
    torch.abs(
        z_obj_C - z_obj_D
    )
).item()

different_object_state_delta = torch.mean(
    torch.abs(
        z_state_C - z_state_D
    )
).item()

print(
    "Object latent difference:",
    f"{different_object_object_delta:.8f}"
)

print(
    "State latent difference:",
    f"{different_object_state_delta:.8f}"
)

print("\nExpected:")
print("  Object -> LARGE")
print("  State  -> SMALL")


# ================================================================
# 10. TEST 3
#
# Explicit object classification
# ================================================================

print("\n" + "=" * 72)
print("TEST 3: OBJECT IDENTITY")
print("=" * 72)

with torch.no_grad():

    visual_tensor = torch.tensor(
        visual_A,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    (
        _,
        logits_a,
        logits_b
    ) = object_teacher(
        visual_tensor
    )

pred_a = torch.argmax(
    logits_a,
    dim=-1
).item()

pred_b = torch.argmax(
    logits_b,
    dim=-1
).item()

names = [
    "red cube",
    "red elongated",
    "blue cube",
    "blue elongated"
]

print(
    "Object A predicted:",
    names[pred_a]
)

print(
    "Object B predicted:",
    names[pred_b]
)


# ================================================================
# 11. TEST 4
#
# State prediction
# ================================================================

print("\n" + "=" * 72)
print("TEST 4: STATE PREDICTION")
print("=" * 72)

with torch.no_grad():

    state_pred_A = state_teacher(
        torch.tensor(
            visual_A,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )[1]

    state_pred_B = state_teacher(
        torch.tensor(
            visual_B,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)
    )[1]

print(
    "\nTarget A:"
)

print(
    np.concatenate([
        [0.7, 0.2, 0.8],
        [-0.6, -0.5, 0.45]
    ])
)

print(
    "Predicted A:"
)

print(
    state_pred_A.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nTarget B:"
)

print(
    np.concatenate([
        [-0.4, 0.8, 0.5],
        [0.8, -0.2, 0.9]
    ])
)

print(
    "Predicted B:"
)

print(
    state_pred_B.squeeze(0)
    .cpu()
    .numpy()
)


# ================================================================
# 12. TEST 5
#
# Selectivity ratios
#
# Object ratio:
#
#   different object / same object
#
# State ratio:
#
#   different state / same state
# ================================================================

print("\n" + "=" * 72)
print("TEST 5: SELECTIVITY")
print("=" * 72)

object_ratio = (
    different_object_object_delta
    /
    (same_object_object_delta + 1e-8)
)

state_ratio = (
    same_object_state_delta
    /
    (different_object_state_delta + 1e-8)
)

print(
    "Object selectivity ratio:",
    f"{object_ratio:.8f}"
)

print(
    "State selectivity ratio:",
    f"{state_ratio:.8f}"
)

print("\nInterpretation:")

print(
    "  Object ratio > 1"
    "  => Object latent reacts more to object"
)

print(
    "  State ratio > 1"
    "  => State latent reacts more to state"
)


# ================================================================
# 13. FINAL REPORT
# ================================================================

print("\n" + "=" * 72)
print("EXPERIMENT 6A FINAL REPORT")
print("=" * 72)

print("\nCore hypothesis:")

print(
    "  Object Teacher = WHAT"
)

print(
    "  State Teacher  = WHERE"
)

print(
    "\nSame Object / Different State:"
)

print(
    f"  Object Δ = "
    f"{same_object_object_delta:.8f}"
)

print(
    f"  State Δ  = "
    f"{same_object_state_delta:.8f}"
)

print(
    "\nDifferent Object / Same State:"
)

print(
    f"  Object Δ = "
    f"{different_object_object_delta:.8f}"
)

print(
    f"  State Δ  = "
    f"{different_object_state_delta:.8f}"
)

print(
    "\nSelectivity:"
)

print(
    f"  Object ratio = "
    f"{object_ratio:.8f}"
)

print(
    f"  State ratio  = "
    f"{state_ratio:.8f}"
)

print("\n" + "=" * 72)
print("Experiment 6A finished.")
print("Two-object minimal factorization tested.")
print("=" * 72)


AntEncoder Experiment 6A
Two-Object Minimal Factorization Test
Device: cpu

Dataset size: 4000

Teacher parameters: 338126

PHASE 1: OBJECT + STATE TEACHER
Epoch 001/100 | Total=0.261672 | Object=0.198207 | State=0.063464
Epoch 002/100 | Total=0.013861 | Object=0.005326 | State=0.008535
Epoch 003/100 | Total=0.005149 | Object=0.002807 | State=0.002342
Epoch 004/100 | Total=0.002651 | Object=0.001908 | State=0.000743
Epoch 005/100 | Total=0.001983 | Object=0.001402 | State=0.000581
Epoch 010/100 | Total=0.001004 | Object=0.000502 | State=0.000502
Epoch 020/100 | Total=0.000710 | Object=0.000170 | State=0.000540
Epoch 030/100 | Total=0.000291 | Object=0.000088 | State=0.000203
Epoch 040/100 | Total=0.000374 | Object=0.000054 | State=0.000320
Epoch 050/100 | Total=0.000257 | Object=0.000036 | State=0.000221
Epoch 060/100 | Total=0.000329 | Object=0.000025 | State=0.000303
Epoch 070/100 | Total=0.000175 | Object=0.000019 | State=0.000156
Epoch 080/100 | Total=0.000233 | Object=0.000014 | S

In [ ]:
# ================================================================
# AntEncoder Experiment 7
# Language-Conditioned Partial Object Reconstruction
#
# Hypothesis:
#
#   Object Teacher:
#       Vision -> complete object attributes
#
#   Language Student:
#       Language -> ONLY attributes recoverable from language
#
#   State Teacher:
#       Vision -> state
#
# Language is NEVER given to the Teachers.
# State is NEVER produced by the Language Student.
# ================================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ================================================================
# 0. Reproducibility
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("AntEncoder Experiment 7")
print("Language-Conditioned Partial Object Reconstruction")
print("=" * 72)
print("Device:", device)


# ================================================================
# 1. Object / Attribute Definitions
# ================================================================

COLORS = ["red", "blue"]
SHAPES = ["cube", "elongated"]

COLOR_RGB = {
    "red": np.array(
        [1.0, 0.0, 0.0],
        dtype=np.float32
    ),
    "blue": np.array(
        [0.0, 0.0, 1.0],
        dtype=np.float32
    ),
}

SHAPE_CODE = {
    "cube": 0.0,
    "elongated": 1.0,
}

# ---------------------------------------------------------------
# Language templates
#
# Crucial:
# Some instructions mention only color.
# Some mention only shape.
# Some mention both.
# Some mention neither.
# ---------------------------------------------------------------

templates = [
    ("color",   "push {color} object"),
    ("shape",   "push {shape} object"),
    ("both",    "push {color} {shape}"),
    ("none",    "push the object"),
]

# Vocabulary

all_texts = []

for _, template in templates:

    for color in COLORS:
        for shape in SHAPES:

            text = template.format(
                color=color,
                shape=shape
            )

            all_texts.append(text)

special_tokens = ["<PAD>", "<UNK>"]

words = set()

for text in all_texts:
    words.update(text.lower().split())

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for word in sorted(words):

    if word not in vocab:
        vocab[word] = len(vocab)

PAD_IDX = vocab["<PAD>"]
UNK_IDX = vocab["<UNK>"]

print("\nVocabulary size:", len(vocab))
print("Vocabulary:", vocab)


def tokenize(text, max_len=8):

    ids = [
        vocab.get(
            word,
            UNK_IDX
        )
        for word in text.lower().split()
    ]

    ids = ids[:max_len]

    if len(ids) < max_len:

        ids += [
            PAD_IDX
        ] * (
            max_len - len(ids)
        )

    return ids


# ================================================================
# 2. Dataset
#
# Two objects only.
#
# Each scene contains:
#
#   Object A
#       color
#       shape
#       xyz
#
#   Object B
#       color
#       shape
#       xyz
#
# The language identifies an object through attributes.
#
# Teacher target:
#
#   complete attributes of the referenced object
#
# Student target:
#
#   only attributes explicitly recoverable from language.
# ================================================================

class PartialObjectDataset(Dataset):

    def __init__(
        self,
        n_samples=5000,
        seed=42
    ):

        self.n_samples = n_samples

        self.rng = np.random.default_rng(
            seed
        )

        # --------------------------------------------------------
        # Fixed projection
        #
        # Each object:
        #   RGB 3
        #   Shape 1
        #   XYZ 3
        #
        # Two objects = 14 dimensions.
        # --------------------------------------------------------

        projection_rng = np.random.default_rng(
            12345
        )

        self.visual_projection = (
            projection_rng.normal(
                0.0,
                1.0 / np.sqrt(14),
                size=(14, 512)
            )
            .astype(np.float32)
        )

        self.samples = []

        for _ in range(n_samples):

            # ----------------------------------------------------
            # Random object identities
            #
            # Force different colors/shapes so identity is useful.
            # ----------------------------------------------------

            color_a = COLORS[
                self.rng.integers(
                    0,
                    len(COLORS)
                )
            ]

            color_b = (
                COLORS[
                    1
                    -
                    COLORS.index(color_a)
                ]
            )

            shape_a = SHAPES[
                self.rng.integers(
                    0,
                    len(SHAPES)
                )
            ]

            shape_b = (
                SHAPES[
                    1
                    -
                    SHAPES.index(shape_a)
                ]
            )

            # ----------------------------------------------------
            # Random states
            # ----------------------------------------------------

            xyz_a = self.rng.uniform(
                low=[
                    -1.0,
                    -1.0,
                    0.35
                ],
                high=[
                    1.0,
                    1.0,
                    1.0
                ],
                size=3
            ).astype(np.float32)

            xyz_b = self.rng.uniform(
                low=[
                    -1.0,
                    -1.0,
                    0.35
                ],
                high=[
                    1.0,
                    1.0,
                    1.0
                ],
                size=3
            ).astype(np.float32)

            objects = [
                {
                    "color": color_a,
                    "shape": shape_a,
                    "xyz": xyz_a,
                },
                {
                    "color": color_b,
                    "shape": shape_b,
                    "xyz": xyz_b,
                },
            ]

            # ----------------------------------------------------
            # Randomly select target object.
            # ----------------------------------------------------

            target_idx = self.rng.integers(
                0,
                2
            )

            target = objects[
                target_idx
            ]

            # ----------------------------------------------------
            # Randomly select language information content.
            # ----------------------------------------------------

            mode, template = templates[
                self.rng.integers(
                    0,
                    len(templates)
                )
            ]

            instruction = template.format(
                color=target["color"],
                shape=target["shape"]
            )

            # ----------------------------------------------------
            # Complete Teacher target
            #
            # [red, blue color one-hot]
            # [cube / elongated]
            # [x,y,z]
            #
            # 2 + 1 + 3 = 6
            # ----------------------------------------------------

            color_target = np.array(
                [
                    1.0
                    if target["color"] == "red"
                    else 0.0,

                    1.0
                    if target["color"] == "blue"
                    else 0.0,
                ],
                dtype=np.float32
            )

            shape_target = np.array(
                [
                    SHAPE_CODE[
                        target["shape"]
                    ]
                ],
                dtype=np.float32
            )

            full_object_target = np.concatenate(
                [
                    color_target,
                    shape_target,
                    target["xyz"],
                ]
            ).astype(np.float32)

            # ----------------------------------------------------
            # Language recoverability mask
            #
            # color mode:
            #   color = 1
            #   shape = 0
            #   state = 0
            #
            # shape mode:
            #   color = 0
            #   shape = 1
            #
            # both:
            #   color = 1
            #   shape = 1
            #
            # none:
            #   all = 0
            #
            # NOTE:
            # state is NEVER language-recoverable.
            # ----------------------------------------------------

            if mode == "color":

                mask = np.array(
                    [
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        0.0,
                        0.0,
                    ],
                    dtype=np.float32
                )

            elif mode == "shape":

                mask = np.array(
                    [
                        0.0,
                        0.0,
                        1.0,
                        0.0,
                        0.0,
                        0.0,
                    ],
                    dtype=np.float32
                )

            elif mode == "both":

                mask = np.array(
                    [
                        1.0,
                        1.0,
                        1.0,
                        0.0,
                        0.0,
                        0.0,
                    ],
                    dtype=np.float32
                )

            else:

                mask = np.zeros(
                    6,
                    dtype=np.float32
                )

            # ----------------------------------------------------
            # Physical visual
            #
            # Object A:
            #   RGB + Shape + XYZ
            #
            # Object B:
            #   RGB + Shape + XYZ
            # ----------------------------------------------------

            physical_visual = []

            for obj in objects:

                physical_visual.extend(
                    COLOR_RGB[
                        obj["color"]
                    ]
                )

                physical_visual.append(
                    SHAPE_CODE[
                        obj["shape"]
                    ]
                )

                physical_visual.extend(
                    obj["xyz"]
                )

            physical_visual = np.asarray(
                physical_visual,
                dtype=np.float32
            )

            visual = (
                physical_visual
                @
                self.visual_projection
            )

            visual += self.rng.normal(
                0.0,
                0.01,
                size=512
            ).astype(np.float32)

            self.samples.append({

                "instruction": instruction,

                "mode": mode,

                "tokens": np.asarray(
                    tokenize(
                        instruction
                    ),
                    dtype=np.int64
                ),

                "visual": visual.astype(
                    np.float32
                ),

                "physical_visual":
                    physical_visual,

                "object_target":
                    full_object_target,

                "language_mask":
                    mask,

                "target_idx":
                    target_idx,

                "target_color":
                    target["color"],

                "target_shape":
                    target["shape"],

                "target_xyz":
                    target["xyz"],
            })

    def __len__(self):
        return len(
            self.samples
        )

    def __getitem__(self, idx):

        s = self.samples[idx]

        return {

            "instruction":
                s["instruction"],

            "mode":
                s["mode"],

            "tokens":
                torch.tensor(
                    s["tokens"],
                    dtype=torch.long
                ),

            "visual":
                torch.tensor(
                    s["visual"],
                    dtype=torch.float32
                ),

            "physical_visual":
                torch.tensor(
                    s["physical_visual"],
                    dtype=torch.float32
                ),

            "object_target":
                torch.tensor(
                    s["object_target"],
                    dtype=torch.float32
                ),

            "language_mask":
                torch.tensor(
                    s["language_mask"],
                    dtype=torch.float32
                ),

            "target_xyz":
                torch.tensor(
                    s["target_xyz"],
                    dtype=torch.float32
                ),

            "target_idx":
                torch.tensor(
                    s["target_idx"],
                    dtype=torch.long
                ),
        }


dataset = PartialObjectDataset(
    n_samples=5000,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

print(
    "\nDataset size:",
    len(dataset)
)


# ================================================================
# 3. Object Teacher
#
# Vision -> complete object representation
#
# IMPORTANT:
# No Language.
# ================================================================

class ObjectTeacher(
    nn.Module
):

    def __init__(
        self,
        input_dim=512,
        latent_dim=128
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                latent_dim
            ),

            nn.Tanh()
        )

        self.decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                6
            )
        )

    def forward(
        self,
        visual
    ):

        z_obj = self.encoder(
            visual
        )

        object_pred = self.decoder(
            z_obj
        )

        return (
            z_obj,
            object_pred
        )


object_teacher = ObjectTeacher().to(
    device
)

teacher_optimizer = torch.optim.AdamW(
    object_teacher.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nObject Teacher parameters:",
    sum(
        p.numel()
        for p in object_teacher.parameters()
    )
)


# ================================================================
# 4. State Teacher
#
# Vision -> target XYZ
#
# IMPORTANT:
# No Language.
# ================================================================

class StateTeacher(
    nn.Module
):

    def __init__(
        self,
        input_dim=512,
        latent_dim=128
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                latent_dim
            ),

            nn.Tanh()
        )

        self.decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                3
            )
        )

    def forward(
        self,
        visual
    ):

        z_state = self.encoder(
            visual
        )

        state_pred = self.decoder(
            z_state
        )

        return (
            z_state,
            state_pred
        )


state_teacher = StateTeacher().to(
    device
)

state_optimizer = torch.optim.AdamW(
    state_teacher.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)


# ================================================================
# 5. PHASE 1
#    Train Teachers
# ================================================================

print("\n" + "=" * 72)
print("PHASE 1: OBJECT + STATE TEACHERS")
print("=" * 72)

teacher_epochs = 80

for epoch in range(
    teacher_epochs
):

    object_teacher.train()
    state_teacher.train()

    total_sum = 0.0
    object_sum = 0.0
    state_sum = 0.0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        object_target = batch[
            "object_target"
        ].to(device)

        state_target = batch[
            "target_xyz"
        ].to(device)

        # --------------------------------------------------------
        # Object Teacher
        # --------------------------------------------------------

        teacher_optimizer.zero_grad()

        _, object_pred = (
            object_teacher(
                visual
            )
        )

        object_loss = F.mse_loss(
            object_pred,
            object_target
        )

        object_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            object_teacher.parameters(),
            1.0
        )

        teacher_optimizer.step()

        # --------------------------------------------------------
        # State Teacher
        # --------------------------------------------------------

        state_optimizer.zero_grad()

        _, state_pred = (
            state_teacher(
                visual
            )
        )

        state_loss = F.mse_loss(
            state_pred,
            state_target
        )

        state_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            state_teacher.parameters(),
            1.0
        )

        state_optimizer.step()

        bs = visual.size(0)

        total_sum += (
            object_loss.item()
            +
            state_loss.item()
        ) * bs

        object_sum += (
            object_loss.item()
            * bs
        )

        state_sum += (
            state_loss.item()
            * bs
        )

    n = len(dataset)

    if (
        epoch < 5
        or (epoch + 1) % 10 == 0
        or epoch == teacher_epochs - 1
    ):

        print(
            f"Epoch {epoch+1:03d}/{teacher_epochs} | "
            f"Total={total_sum/n:.6f} | "
            f"Object={object_sum/n:.6f} | "
            f"State={state_sum/n:.6f}"
        )


# ================================================================
# Freeze Teachers
# ================================================================

for p in object_teacher.parameters():
    p.requires_grad = False

for p in state_teacher.parameters():
    p.requires_grad = False

object_teacher.eval()
state_teacher.eval()

print(
    "\nTeachers FROZEN."
)


# ================================================================
# 6. Language Student
#
# Language -> partial object concept
#
# The Student produces:
#
#   z_obj_L
#
# and a 6D attribute reconstruction.
#
# IMPORTANT:
# The loss is MASKED.
#
# The Student is only penalized for attributes
# that are recoverable from language.
# ================================================================

class LanguageStudent(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_dim=96,
        latent_dim=128
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.object_head = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                latent_dim
            ),

            nn.Tanh()
        )

        self.attribute_decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                6
            )
        )

    def forward(
        self,
        tokens
    ):

        x = self.embedding(
            tokens
        )

        _, hidden = self.gru(x)

        h_forward = hidden[-2]
        h_backward = hidden[-1]

        h = torch.cat(
            [
                h_forward,
                h_backward
            ],
            dim=-1
        )

        z_obj = self.object_head(
            h
        )

        attributes = (
            self.attribute_decoder(
                z_obj
            )
        )

        return (
            z_obj,
            attributes
        )


student = LanguageStudent(
    vocab_size=len(vocab)
).to(device)

student_optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nStudent parameters:",
    sum(
        p.numel()
        for p in student.parameters()
    )
)


# ================================================================
# 7. PHASE 2
#    Partial Reconstruction
# ================================================================

print("\n" + "=" * 72)
print("PHASE 2: LANGUAGE PARTIAL RECONSTRUCTION")
print("=" * 72)

student_epochs = 40

for epoch in range(
    student_epochs
):

    student.train()

    loss_sum = 0.0
    color_loss_sum = 0.0
    shape_loss_sum = 0.0
    hidden_loss_sum = 0.0

    for batch in loader:

        tokens = batch[
            "tokens"
        ].to(device)

        object_target = batch[
            "object_target"
        ].to(device)

        mask = batch[
            "language_mask"
        ].to(device)

        student_optimizer.zero_grad()

        z_obj_L, pred = student(
            tokens
        )

        # --------------------------------------------------------
        # Masked reconstruction
        #
        # Color dimensions = [0,1]
        # Shape dimension = [2]
        # State dimensions = [3,4,5]
        #
        # State is NEVER supervised from language.
        # --------------------------------------------------------

        masked_error = (
            torch.abs(
                pred - object_target
            )
            * mask
        )

        denominator = (
            mask.sum()
            .clamp_min(1.0)
        )

        reconstruction_loss = (
            masked_error.sum()
            /
            denominator
        )

        # --------------------------------------------------------
        # Explicitly measure hidden-state leakage.
        #
        # We do NOT train on these dimensions.
        # --------------------------------------------------------

        hidden_mask = 1.0 - mask

        hidden_error = (
            torch.abs(
                pred - object_target
            )
            * hidden_mask
        )

        hidden_denominator = (
            hidden_mask.sum()
            .clamp_min(1.0)
        )

        hidden_loss = (
            hidden_error.sum()
            /
            hidden_denominator
        )

        # --------------------------------------------------------
        # Only reconstruction_loss is optimized.
        # --------------------------------------------------------

        loss = reconstruction_loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            1.0
        )

        student_optimizer.step()

        bs = tokens.size(0)

        loss_sum += (
            reconstruction_loss.item()
            * bs
        )

        # Color error
        color_mask = mask[:, :2]

        color_denominator = (
            color_mask.sum()
            .clamp_min(1.0)
        )

        color_error = (
            torch.abs(
                pred[:, :2]
                -
                object_target[:, :2]
            )
            *
            color_mask
        ).sum() / color_denominator

        # Shape error
        shape_mask = mask[:, 2:3]

        shape_denominator = (
            shape_mask.sum()
            .clamp_min(1.0)
        )

        shape_error = (
            torch.abs(
                pred[:, 2:3]
                -
                object_target[:, 2:3]
            )
            *
            shape_mask
        ).sum() / shape_denominator

        color_loss_sum += (
            color_error.item()
            * bs
        )

        shape_loss_sum += (
            shape_error.item()
            * bs
        )

        hidden_loss_sum += (
            hidden_loss.item()
            * bs
        )

    n = len(dataset)

    print(
        f"Student Epoch {epoch+1:02d}/{student_epochs} | "
        f"Masked={loss_sum/n:.6f} | "
        f"Color={color_loss_sum/n:.6f} | "
        f"Shape={shape_loss_sum/n:.6f} | "
        f"Unsupervised={hidden_loss_sum/n:.6f}"
    )


# ================================================================
# Freeze Student
# ================================================================

for p in student.parameters():
    p.requires_grad = False

student.eval()

print(
    "\nStudent FROZEN."
)


# ================================================================
# 8. Evaluation Helpers
# ================================================================

def student_encode(
    instruction
):

    tokens = torch.tensor(
        [
            tokenize(
                instruction
            )
        ],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        z, attributes = student(
            tokens
        )

    return (
        z,
        attributes
    )


def teacher_encode(
    index
):

    visual = torch.tensor(
        dataset.samples[index][
            "visual"
        ],
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        z_obj, obj_pred = (
            object_teacher(
                visual
            )
        )

        z_state, state_pred = (
            state_teacher(
                visual
            )
        )

    return (
        z_obj,
        obj_pred,
        z_state,
        state_pred
    )


# ================================================================
# 9. TEST 1
#    COLOR ONLY
# ================================================================

print("\n" + "=" * 72)
print("TEST 1: COLOR-ONLY LANGUAGE")
print("=" * 72)

instruction = (
    "push red object"
)

z_color, pred_color = (
    student_encode(
        instruction
    )
)

print(
    "\nLanguage:",
    instruction
)

print(
    "Student prediction:",
    pred_color.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nExpected:"
)

print(
    "  Color -> reconstructable"
)

print(
    "  Shape -> NOT specified"
)

print(
    "  XYZ   -> NOT specified"
)


# ================================================================
# 10. TEST 2
#     SHAPE ONLY
# ================================================================

print("\n" + "=" * 72)
print("TEST 2: SHAPE-ONLY LANGUAGE")
print("=" * 72)

instruction = (
    "push elongated object"
)

z_shape, pred_shape = (
    student_encode(
        instruction
    )
)

print(
    "\nLanguage:",
    instruction
)

print(
    "Student prediction:",
    pred_shape.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nExpected:"
)

print(
    "  Color -> NOT specified"
)

print(
    "  Shape -> reconstructable"
)

print(
    "  XYZ   -> NOT specified"
)


# ================================================================
# 11. TEST 3
#     COLOR + SHAPE
# ================================================================

print("\n" + "=" * 72)
print("TEST 3: COLOR + SHAPE LANGUAGE")
print("=" * 72)

instruction = (
    "push red elongated"
)

z_both, pred_both = (
    student_encode(
        instruction
    )
)

print(
    "\nLanguage:",
    instruction
)

print(
    "Student prediction:",
    pred_both.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nExpected:"
)

print(
    "  Color -> reconstructable"
)

print(
    "  Shape -> reconstructable"
)

print(
    "  XYZ   -> NOT specified"
)


# ================================================================
# 12. TEST 4
#     NO OBJECT ATTRIBUTE
# ================================================================

print("\n" + "=" * 72)
print("TEST 4: NO ATTRIBUTE LANGUAGE")
print("=" * 72)

instruction = (
    "push the object"
)

z_none, pred_none = (
    student_encode(
        instruction
    )
)

print(
    "\nLanguage:",
    instruction
)

print(
    "Student prediction:",
    pred_none.squeeze(0)
    .cpu()
    .numpy()
)

print(
    "\nExpected:"
)

print(
    "  Color -> unspecified"
)

print(
    "  Shape -> unspecified"
)

print(
    "  XYZ   -> unspecified"
)


# ================================================================
# 13. TEST 5
#     SAME LANGUAGE / DIFFERENT WORLD
#
# The Student should remain identical because
# it only sees language.
# ================================================================

print("\n" + "=" * 72)
print("TEST 5: SAME LANGUAGE / DIFFERENT WORLD")
print("=" * 72)

instruction = (
    "push red object"
)

z1, pred1 = student_encode(
    instruction
)

z2, pred2 = student_encode(
    instruction
)

student_difference = torch.mean(
    torch.abs(
        z1 - z2
    )
).item()

print(
    "\nLanguage:",
    instruction
)

print(
    "Student latent difference:",
    f"{student_difference:.10f}"
)

print(
    "\nExpected:"
)

print(
    "  Student is language-only."
)

print(
    "  Therefore same language -> same representation."
)


# ================================================================
# 14. TEST 6
#     LANGUAGE ATTRIBUTE SELECTIVITY
# ================================================================

print("\n" + "=" * 72)
print("TEST 6: LANGUAGE ATTRIBUTE SELECTIVITY")
print("=" * 72)

_, red_pred = student_encode(
    "push red object"
)

_, blue_pred = student_encode(
    "push blue object"
)

_, cube_pred = student_encode(
    "push cube object"
)

_, elongated_pred = student_encode(
    "push elongated object"
)

color_difference = torch.mean(
    torch.abs(
        red_pred[:, :2]
        -
        blue_pred[:, :2]
    )
).item()

shape_difference = torch.mean(
    torch.abs(
        cube_pred[:, 2]
        -
        elongated_pred[:, 2]
    )
).item()

cross_color_shape = torch.mean(
    torch.abs(
        red_pred[:, 2]
        -
        blue_pred[:, 2]
    )
).item()

print(
    "\nColor language difference:",
    f"{color_difference:.8f}"
)

print(
    "Shape language difference:",
    f"{shape_difference:.8f}"
)

print(
    "Cross-attribute shape difference:",
    f"{cross_color_shape:.8f}"
)

print(
    "\nExpected:"
)

print(
    "  Color wording changes -> color representation"
)

print(
    "  Shape wording changes -> shape representation"
)

print(
    "  Changing color should NOT strongly change shape."
)


# ================================================================
# 15. TEST 7
#     LANGUAGE MUST NOT RECOVER STATE
# ================================================================

print("\n" + "=" * 72)
print("TEST 7: LANGUAGE STATE NON-RECOVERABILITY")
print("=" * 72)

# Find two scenes with the same language
# but different target states.

indices = []

for i, s in enumerate(
    dataset.samples
):

    if (
        s["instruction"]
        ==
        "push red object"
    ):

        indices.append(i)

    if len(indices) >= 2:
        break

idx_a = indices[0]
idx_b = indices[1]

(
    _,
    _,
    state_z_a,
    state_pred_a
) = teacher_encode(
    idx_a
)

(
    _,
    _,
    state_z_b,
    state_pred_b
) = teacher_encode(
    idx_b
)

state_difference = torch.mean(
    torch.abs(
        state_z_a
        -
        state_z_b
    )
).item()

student_state_difference = torch.mean(
    torch.abs(
        pred1[:, 3:]
        -
        pred2[:, 3:]
    )
).item()

print(
    "\nTeacher state difference:",
    f"{state_difference:.8f}"
)

print(
    "Student state output difference:",
    f"{student_state_difference:.8f}"
)

print(
    "\nExpected:"
)

print(
    "  Teacher state -> LARGE"
)

print(
    "  Student language-derived state -> NONE / meaningless"
)


# ================================================================
# 16. FINAL REPORT
# ================================================================

print("\n" + "=" * 72)
print("EXPERIMENT 7 FINAL REPORT")
print("=" * 72)

print(
    "\nCore hypothesis:"
)

print(
    "  Object Teacher = complete world-side WHAT"
)

print(
    "  Language Student = language-recoverable WHAT only"
)

print(
    "  State Teacher = WHERE"
)

print(
    "\nInformation asymmetry:"
)

print(
    "  Teacher sees Vision only."
)

print(
    "  Student sees Language only."
)

print(
    "  Teacher knows complete object attributes."
)

print(
    "  Student is supervised only on language-recoverable attributes."
)

print(
    "\nImportant:"
)

print(
    "  Language never enters Object Teacher."
)

print(
    "  Language never enters State Teacher."
)

print(
    "  State never enters Language Student."
)

print(
    "\nExperiment 7 finished."
)

print(
    "Language-conditioned partial object reconstruction tested."
)

print("=" * 72)


AntEncoder Experiment 7
Language-Conditioned Partial Object Reconstruction
Device: cpu

Vocabulary size: 9
Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'blue': 2, 'cube': 3, 'elongated': 4, 'object': 5, 'push': 6, 'red': 7, 'the': 8}

Dataset size: 5000

Object Teacher parameters: 172870

PHASE 1: OBJECT + STATE TEACHERS
Epoch 001/80 | Total=0.360594 | Object=0.209289 | State=0.151305
Epoch 002/80 | Total=0.311660 | Object=0.188231 | State=0.123430
Epoch 003/80 | Total=0.307743 | Object=0.186946 | State=0.120798
Epoch 004/80 | Total=0.305198 | Object=0.185988 | State=0.119210
Epoch 005/80 | Total=0.306251 | Object=0.186270 | State=0.119981
Epoch 010/80 | Total=0.299316 | Object=0.183167 | State=0.116149
Epoch 020/80 | Total=0.294811 | Object=0.181088 | State=0.113723
Epoch 030/80 | Total=0.290992 | Object=0.178873 | State=0.112119
Epoch 040/80 | Total=0.275536 | Object=0.171727 | State=0.103810
Epoch 050/80 | Total=0.257495 | Object=0.162612 | State=0.094883
Epoch 060/80 | Total=0.233132 | Obj

In [ ]:
# ================================================================
# AntEncoder Experiment 8
# Masked Partial Object Reconstruction
#
# Goal:
#   Object Teacher = complete WHAT
#   Language Student = only language-recoverable WHAT
#   Unknown attributes are explicitly represented by masks.
#
# Student outputs:
#   color latent + color mask
#   shape latent + shape mask
#
# No state is given to Student.
# ================================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ================================================================
# 0. Reproducibility
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("AntEncoder Experiment 8")
print("Masked Partial Object Reconstruction")
print("=" * 72)

print("Device:", device)


# ================================================================
# 1. Object vocabulary
# ================================================================

COLORS = [
    "red",
    "blue",
]

SHAPES = [
    "cube",
    "elongated",
]

# Four possible complete objects
OBJECTS = [
    ("red", "cube"),
    ("red", "elongated"),
    ("blue", "cube"),
    ("blue", "elongated"),
]

COLOR_ID = {
    "red": 0,
    "blue": 1,
}

SHAPE_ID = {
    "cube": 0,
    "elongated": 1,
}


# ================================================================
# 2. Language patterns
#
# Each sentence intentionally specifies only some attributes.
# ================================================================

language_patterns = [
    ("push red object",       "red",       None),
    ("push blue object",      "blue",      None),

    ("push cube object",      None,        "cube"),
    ("push elongated object", None,        "elongated"),

    ("push red cube",          "red",       "cube"),
    ("push red elongated",     "red",       "elongated"),
    ("push blue cube",         "blue",      "cube"),
    ("push blue elongated",    "blue",      "elongated"),

    ("push the object",        None,        None),
]


# ================================================================
# 3. Vocabulary
# ================================================================

special_tokens = [
    "<PAD>",
    "<UNK>",
]

words = set()

for text, _, _ in language_patterns:
    words.update(text.split())

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for word in sorted(words):
    if word not in vocab:
        vocab[word] = len(vocab)

PAD_IDX = vocab["<PAD>"]
UNK_IDX = vocab["<UNK>"]

print("\nVocabulary size:", len(vocab))
print("Vocabulary:", vocab)


def tokenize(text, max_len=6):

    ids = [
        vocab.get(w, UNK_IDX)
        for w in text.lower().split()
    ]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [
            PAD_IDX
        ] * (
            max_len - len(ids)
        )

    return ids


# ================================================================
# 4. Dataset
#
# Vision contains:
#
#   color
#   shape
#   xyz
#
# Object Teacher must learn complete WHAT.
#
# Language Student only receives language.
# ================================================================

class ObjectWorldDataset(Dataset):

    def __init__(
        self,
        n_samples=6000,
        seed=42
    ):

        self.rng = np.random.default_rng(seed)

        projection_rng = np.random.default_rng(
            12345
        )

        # 2 objects ×
        # (RGB 3 + shape encoding 2 + XYZ 3)
        #
        # = 16 physical dimensions

        self.projection = (
            projection_rng.normal(
                0.0,
                1.0 / np.sqrt(16),
                size=(16, 512)
            )
            .astype(np.float32)
        )

        self.samples = []

        for _ in range(n_samples):

            # ----------------------------------------------------
            # Select complete world object
            # ----------------------------------------------------

            color, shape = OBJECTS[
                self.rng.integers(
                    0,
                    len(OBJECTS)
                )
            ]

            # ----------------------------------------------------
            # Random target position
            # ----------------------------------------------------

            target_xyz = self.rng.uniform(
                [-1.0, -1.0, 0.35],
                [1.0, 1.0, 1.0]
            ).astype(np.float32)

            # ----------------------------------------------------
            # Another distractor object
            # ----------------------------------------------------

            other_candidates = [
                obj
                for obj in OBJECTS
                if obj != (color, shape)
            ]

            other_color, other_shape = (
                other_candidates[
                    self.rng.integers(
                        0,
                        len(other_candidates)
                    )
                ]
            )

            other_xyz = self.rng.uniform(
                [-1.0, -1.0, 0.35],
                [1.0, 1.0, 1.0]
            ).astype(np.float32)

            # ----------------------------------------------------
            # Physical representation
            #
            # Object 1:
            #   RGB + shape one-hot + XYZ
            #
            # Object 2:
            #   RGB + shape one-hot + XYZ
            # ----------------------------------------------------

            def encode_object(
                c,
                s,
                xyz
            ):

                rgb = {
                    "red": [1., 0., 0.],
                    "blue": [0., 0., 1.]
                }[c]

                shape_vec = {
                    "cube": [1., 0.],
                    "elongated": [0., 1.]
                }[s]

                return (
                    rgb
                    + shape_vec
                    + list(xyz)
                )

            physical = (
                encode_object(
                    color,
                    shape,
                    target_xyz
                )
                +
                encode_object(
                    other_color,
                    other_shape,
                    other_xyz
                )
            )

            physical = np.asarray(
                physical,
                dtype=np.float32
            )

            visual = (
                physical @ self.projection
            )

            visual += self.rng.normal(
                0.0,
                0.02,
                size=512
            ).astype(np.float32)

            # ----------------------------------------------------
            # Language
            #
            # Randomly choose how much information
            # language actually specifies.
            # ----------------------------------------------------

            instruction, lang_color, lang_shape = (
                language_patterns[
                    self.rng.integers(
                        0,
                        len(language_patterns)
                    )
                ]
            )

            # ----------------------------------------------------
            # Language masks
            #
            # 1 = attribute specified
            # 0 = attribute unspecified
            # ----------------------------------------------------

            color_mask = (
                1.0
                if lang_color is not None
                else 0.0
            )

            shape_mask = (
                1.0
                if lang_shape is not None
                else 0.0
            )

            # ----------------------------------------------------
            # Labels only exist where language specifies them.
            # ----------------------------------------------------

            color_label = (
                COLOR_ID[lang_color]
                if lang_color is not None
                else -1
            )

            shape_label = (
                SHAPE_ID[lang_shape]
                if lang_shape is not None
                else -1
            )

            self.samples.append({
                "instruction": instruction,
                "tokens": np.asarray(
                    tokenize(instruction),
                    dtype=np.int64
                ),

                "visual": visual,

                "color": color,
                "shape": shape,

                "target_xyz": target_xyz,

                "color_label": color_label,
                "shape_label": shape_label,

                "color_mask": color_mask,
                "shape_mask": shape_mask,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        s = self.samples[idx]

        return {
            "instruction": s["instruction"],

            "tokens": torch.tensor(
                s["tokens"],
                dtype=torch.long
            ),

            "visual": torch.tensor(
                s["visual"],
                dtype=torch.float32
            ),

            "color_label": torch.tensor(
                s["color_label"],
                dtype=torch.long
            ),

            "shape_label": torch.tensor(
                s["shape_label"],
                dtype=torch.long
            ),

            "color_mask": torch.tensor(
                s["color_mask"],
                dtype=torch.float32
            ),

            "shape_mask": torch.tensor(
                s["shape_mask"],
                dtype=torch.float32
            ),
        }


dataset = ObjectWorldDataset(
    n_samples=6000,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

print("\nDataset size:", len(dataset))


# ================================================================
# 5. Object Teacher
#
# Vision -> complete WHAT
#
# Teacher knows:
#   color
#   shape
#
# State/XYZ deliberately excluded from object latent target.
# ================================================================

class ObjectTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        latent_dim=128
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                latent_dim
            ),

            nn.Tanh()
        )

        self.color_head = nn.Linear(
            latent_dim,
            2
        )

        self.shape_head = nn.Linear(
            latent_dim,
            2
        )

    def forward(self, visual):

        z = self.encoder(visual)

        color_logits = (
            self.color_head(z)
        )

        shape_logits = (
            self.shape_head(z)
        )

        return (
            z,
            color_logits,
            shape_logits
        )


teacher = ObjectTeacher().to(device)

optimizer = torch.optim.AdamW(
    teacher.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nObject Teacher parameters:",
    sum(
        p.numel()
        for p in teacher.parameters()
    )
)


# ================================================================
# 6. Teacher training
#
# IMPORTANT:
# Teacher always learns COMPLETE object identity.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 1: COMPLETE OBJECT TEACHER")
print("=" * 72)

teacher_epochs = 80

for epoch in range(
    teacher_epochs
):

    teacher.train()

    total_sum = 0.0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        # Complete world-side labels
        colors = []
        shapes = []

        for i in range(
            len(batch["instruction"])
        ):

            sample = dataset.samples[
                i
            ]

        # Reconstruct labels directly
        # from dataset batch using language-compatible
        # target information is insufficient here, so
        # create complete labels from original samples
        #
        # Instead use the fact that each language example
        # specifies an object attribute when available.
        #
        # For teacher training we therefore recover the
        # actual object identity from the physical sample
        # stored by dataset.

        # --------------------------------------------------------
        # To avoid ambiguity from the language,
        # build complete labels by matching each visual sample.
        # --------------------------------------------------------

        # We attach labels dynamically below.
        #
        # The clean implementation is to use the stored
        # complete labels from the dataset.

        # This block is replaced by explicit tensors generated
        # from dataset sample indices through a custom collate.
        pass


AntEncoder Experiment 8
Masked Partial Object Reconstruction
Device: cpu

Vocabulary size: 9
Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'blue': 2, 'cube': 3, 'elongated': 4, 'object': 5, 'push': 6, 'red': 7, 'the': 8}

Dataset size: 6000

Object Teacher parameters: 164740

PHASE 1: COMPLETE OBJECT TEACHER


In [ ]:
# ================================================================
# AntEncoder Experiment 8B
# Attribute-wise Object Teacher + Partial Language Student
#
# Hypothesis:
#
#   Vision Teacher:
#       Vision -> Color latent
#       Vision -> Shape latent
#       Vision -> State latent
#
#   Language Student:
#       Language -> Color latent
#       Language -> Shape latent
#
#   BUT:
#       Student is supervised ONLY on attributes explicitly
#       specified by language.
#
#   Unspecified attributes receive NO reconstruction loss.
#
# Goal:
#   Test whether language can recover only the WHAT information
#   that is actually expressed in language.
# ================================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ================================================================
# 0. Reproducibility / Device
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("AntEncoder Experiment 8B")
print("Attribute-wise Object Teacher + Partial Language Student")
print("=" * 72)
print("Device:", device)


# ================================================================
# 1. Vocabulary
# ================================================================

instructions = [
    "push red object",
    "push blue object",
    "push cube",
    "push elongated object",
    "push red cube",
    "push blue cube",
    "push red elongated",
    "push blue elongated",
    "push the object",
]

all_words = set()

for text in instructions:
    all_words.update(text.split())

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for word in sorted(all_words):
    if word not in vocab:
        vocab[word] = len(vocab)

PAD_IDX = vocab["<PAD>"]
UNK_IDX = vocab["<UNK>"]

print("\nVocabulary size:", len(vocab))
print("Vocabulary:", vocab)


def tokenize(text, max_len=6):

    ids = [
        vocab.get(w, UNK_IDX)
        for w in text.lower().split()
    ]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [PAD_IDX] * (
            max_len - len(ids)
        )

    return ids


# ================================================================
# 2. Dataset
#
# Two object types:
#
#   red cube
#   blue elongated
#
# Position is independently randomized.
#
# IMPORTANT:
# The language does NOT necessarily specify all object attributes.
# ================================================================

class AttributeDataset(Dataset):

    COLORS = [
        "red",
        "blue"
    ]

    SHAPES = [
        "cube",
        "elongated"
    ]

    COLOR_RGB = {
        "red": np.array(
            [1.0, 0.0, 0.0],
            dtype=np.float32
        ),
        "blue": np.array(
            [0.0, 0.0, 1.0],
            dtype=np.float32
        ),
    }

    SHAPE_VEC = {
        "cube": np.array(
            [1.0, 0.0],
            dtype=np.float32
        ),
        "elongated": np.array(
            [0.0, 1.0],
            dtype=np.float32
        ),
    }

    def __init__(
        self,
        n_samples=6000,
        seed=42
    ):

        self.rng = np.random.default_rng(seed)

        projection_rng = np.random.default_rng(
            12345
        )

        # Physical representation:
        #
        # RGB 3
        # shape 2
        # XYZ 3
        #
        # total = 8D

        self.projection = (
            projection_rng.normal(
                0.0,
                1.0 / np.sqrt(8),
                size=(8, 512)
            )
            .astype(np.float32)
        )

        self.samples = []

        for _ in range(n_samples):

            color = self.COLORS[
                self.rng.integers(0, 2)
            ]

            shape = self.SHAPES[
                self.rng.integers(0, 2)
            ]

            xyz = self.rng.uniform(
                low=[-1.0, -1.0, 0.35],
                high=[1.0, 1.0, 1.0],
                size=3
            ).astype(np.float32)

            # ----------------------------------------------------
            # Language template
            #
            # Each language specifies a subset of attributes.
            # ----------------------------------------------------

            template_id = self.rng.integers(
                0,
                len(instructions)
            )

            template = instructions[
                template_id
            ]

            # Only allow language to describe
            # attributes that are actually true.
            #
            # If "red cube" is requested, object must be red cube.
            # If "red object", only color is constrained.
            # If "cube", only shape is constrained.
            # If "object", nothing is constrained.

            valid = False

            while not valid:

                if template == "push red object":

                    valid = (
                        color == "red"
                    )

                elif template == "push blue object":

                    valid = (
                        color == "blue"
                    )

                elif template == "push cube":

                    valid = (
                        shape == "cube"
                    )

                elif template == "push elongated object":

                    valid = (
                        shape == "elongated"
                    )

                elif template == "push red cube":

                    valid = (
                        color == "red"
                        and
                        shape == "cube"
                    )

                elif template == "push blue cube":

                    valid = (
                        color == "blue"
                        and
                        shape == "cube"
                    )

                elif template == "push red elongated":

                    valid = (
                        color == "red"
                        and
                        shape == "elongated"
                    )

                elif template == "push blue elongated":

                    valid = (
                        color == "blue"
                        and
                        shape == "elongated"
                    )

                elif template == "push the object":

                    valid = True

                if not valid:

                    color = self.COLORS[
                        self.rng.integers(0, 2)
                    ]

                    shape = self.SHAPES[
                        self.rng.integers(0, 2)
                    ]

            # ----------------------------------------------------
            # Attribute specification masks
            #
            # 1 = explicitly specified by language
            # 0 = unspecified
            # ----------------------------------------------------

            color_specified = int(
                "red" in template
                or
                "blue" in template
            )

            shape_specified = int(
                "cube" in template
                or
                "elongated" in template
            )

            # ----------------------------------------------------
            # Physical visual representation
            # ----------------------------------------------------

            physical = np.concatenate([
                self.COLOR_RGB[color],
                self.SHAPE_VEC[shape],
                xyz
            ]).astype(np.float32)

            visual = (
                physical @ self.projection
            )

            visual += self.rng.normal(
                0.0,
                0.02,
                size=512
            ).astype(np.float32)

            self.samples.append({

                "instruction": template,

                "tokens": np.asarray(
                    tokenize(template),
                    dtype=np.int64
                ),

                "visual": visual.astype(
                    np.float32
                ),

                "physical": physical,

                "xyz": xyz,

                "color": color,

                "shape": shape,

                "color_label": (
                    0 if color == "red"
                    else 1
                ),

                "shape_label": (
                    0 if shape == "cube"
                    else 1
                ),

                "color_specified":
                    color_specified,

                "shape_specified":
                    shape_specified,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        s = self.samples[idx]

        return {

            "instruction":
                s["instruction"],

            "tokens":
                torch.tensor(
                    s["tokens"],
                    dtype=torch.long
                ),

            "visual":
                torch.tensor(
                    s["visual"],
                    dtype=torch.float32
                ),

            "color_label":
                torch.tensor(
                    s["color_label"],
                    dtype=torch.long
                ),

            "shape_label":
                torch.tensor(
                    s["shape_label"],
                    dtype=torch.long
                ),

            "color_specified":
                torch.tensor(
                    s["color_specified"],
                    dtype=torch.float32
                ),

            "shape_specified":
                torch.tensor(
                    s["shape_specified"],
                    dtype=torch.float32
                ),

            "xyz":
                torch.tensor(
                    s["xyz"],
                    dtype=torch.float32
                ),
        }


dataset = AttributeDataset(
    n_samples=6000,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

print(
    "\nDataset size:",
    len(dataset)
)


# ================================================================
# 3. Attribute-wise Object Teacher
#
# Vision -> independent Color / Shape representations
#
# This is the critical difference from Experiment 7.
# ================================================================

class AttributeObjectTeacher(
    nn.Module
):

    def __init__(
        self,
        input_dim=512,
        latent_dim=64
    ):

        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(
                input_dim,
                256
            ),
            nn.ReLU()
        )

        self.color_encoder = nn.Sequential(
            nn.Linear(
                256,
                latent_dim
            ),
            nn.Tanh()
        )

        self.shape_encoder = nn.Sequential(
            nn.Linear(
                256,
                latent_dim
            ),
            nn.Tanh()
        )

        self.color_classifier = nn.Linear(
            latent_dim,
            2
        )

        self.shape_classifier = nn.Linear(
            latent_dim,
            2
        )

    def forward(self, visual):

        h = self.shared(visual)

        z_color = self.color_encoder(h)

        z_shape = self.shape_encoder(h)

        color_logits = (
            self.color_classifier(
                z_color
            )
        )

        shape_logits = (
            self.shape_classifier(
                z_shape
            )
        )

        return (
            z_color,
            z_shape,
            color_logits,
            shape_logits
        )


teacher = AttributeObjectTeacher().to(
    device
)

teacher_optimizer = torch.optim.AdamW(
    teacher.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nTeacher parameters:",
    sum(
        p.numel()
        for p in teacher.parameters()
    )
)


# ================================================================
# 4. Teacher Training
# ================================================================

print("\n" + "=" * 72)
print("PHASE 1: ATTRIBUTE-WISE OBJECT TEACHER")
print("=" * 72)

teacher_epochs = 80

for epoch in range(
    teacher_epochs
):

    teacher.train()

    total_sum = 0.0
    color_sum = 0.0
    shape_sum = 0.0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        color_label = batch[
            "color_label"
        ].to(device)

        shape_label = batch[
            "shape_label"
        ].to(device)

        teacher_optimizer.zero_grad()

        (
            z_color,
            z_shape,
            color_logits,
            shape_logits
        ) = teacher(
            visual
        )

        color_loss = F.cross_entropy(
            color_logits,
            color_label
        )

        shape_loss = F.cross_entropy(
            shape_logits,
            shape_label
        )

        loss = (
            color_loss
            +
            shape_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            teacher.parameters(),
            1.0
        )

        teacher_optimizer.step()

        bs = visual.size(0)

        total_sum += (
            loss.item() * bs
        )

        color_sum += (
            color_loss.item() * bs
        )

        shape_sum += (
            shape_loss.item() * bs
        )

    n = len(dataset)

    if (
        epoch < 5
        or
        (epoch + 1) % 10 == 0
        or
        epoch == teacher_epochs - 1
    ):

        print(
            f"Epoch {epoch+1:03d}/{teacher_epochs} | "
            f"Total={total_sum/n:.6f} | "
            f"Color={color_sum/n:.6f} | "
            f"Shape={shape_sum/n:.6f}"
        )


# ================================================================
# 5. Freeze Teacher
# ================================================================

for p in teacher.parameters():
    p.requires_grad = False

teacher.eval()

print("\nTeacher FROZEN.")


# ================================================================
# 6. Language Student
#
# Language -> Color latent
# Language -> Shape latent
#
# No State branch.
# ================================================================

class PartialLanguageStudent(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_dim=96,
        latent_dim=64
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.color_head = nn.Sequential(
            nn.Linear(
                hidden_dim * 2,
                latent_dim
            ),
            nn.Tanh()
        )

        self.shape_head = nn.Sequential(
            nn.Linear(
                hidden_dim * 2,
                latent_dim
            ),
            nn.Tanh()
        )

    def forward(self, tokens):

        x = self.embedding(tokens)

        _, hidden = self.gru(x)

        h = torch.cat([
            hidden[-2],
            hidden[-1]
        ], dim=-1)

        z_color = self.color_head(h)

        z_shape = self.shape_head(h)

        return (
            z_color,
            z_shape
        )


student = PartialLanguageStudent(
    vocab_size=len(vocab)
).to(device)

student_optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nStudent parameters:",
    sum(
        p.numel()
        for p in student.parameters()
    )
)


# ================================================================
# 7. PHASE 2
#
# Partial reconstruction.
#
# IMPORTANT:
#
# Color loss only when color is specified.
# Shape loss only when shape is specified.
#
# No loss whatsoever for unspecified attributes.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 2: MASKED PARTIAL LANGUAGE RECONSTRUCTION")
print("=" * 72)

student_epochs = 50

for epoch in range(
    student_epochs
):

    student.train()

    total_sum = 0.0
    color_sum = 0.0
    shape_sum = 0.0

    for batch in loader:

        tokens = batch[
            "tokens"
        ].to(device)

        visual = batch[
            "visual"
        ].to(device)

        color_label = batch[
            "color_label"
        ].to(device)

        shape_label = batch[
            "shape_label"
        ].to(device)

        color_mask = batch[
            "color_specified"
        ].to(device)

        shape_mask = batch[
            "shape_specified"
        ].to(device)

        student_optimizer.zero_grad()

        (
            z_color_L,
            z_shape_L
        ) = student(tokens)

        with torch.no_grad():

            (
                z_color_V,
                z_shape_V,
                _,
                _
            ) = teacher(visual)

        # --------------------------------------------------------
        # Attribute classifiers are used ONLY to train/check
        # semantic content.
        # --------------------------------------------------------

        # Convert teacher latent targets into detached
        # attribute classification targets via teacher logits.
        #
        # We use latent distillation only for specified attributes.
        # --------------------------------------------------------

        color_distill = (
            torch.mean(
                (
                    z_color_L
                    -
                    z_color_V
                ) ** 2,
                dim=1
            )
        )

        shape_distill = (
            torch.mean(
                (
                    z_shape_L
                    -
                    z_shape_V
                ) ** 2,
                dim=1
            )
        )

        # --------------------------------------------------------
        # MASKING
        # --------------------------------------------------------

        color_den = (
            color_mask.sum()
            + 1e-8
        )

        shape_den = (
            shape_mask.sum()
            + 1e-8
        )

        color_loss = (
            (
                color_distill
                *
                color_mask
            ).sum()
            /
            color_den
        )

        shape_loss = (
            (
                shape_distill
                *
                shape_mask
            ).sum()
            /
            shape_den
        )

        loss = (
            color_loss
            +
            shape_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            1.0
        )

        student_optimizer.step()

        bs = tokens.size(0)

        total_sum += (
            loss.item() * bs
        )

        color_sum += (
            color_loss.item() * bs
        )

        shape_sum += (
            shape_loss.item() * bs
        )

    n = len(dataset)

    if (
        epoch < 5
        or
        (epoch + 1) % 5 == 0
        or
        epoch == student_epochs - 1
    ):

        print(
            f"Student Epoch "
            f"{epoch+1:02d}/{student_epochs} | "
            f"Masked={total_sum/n:.6f} | "
            f"Color={color_sum/n:.6f} | "
            f"Shape={shape_sum/n:.6f}"
        )


# ================================================================
# 8. Freeze Student
# ================================================================

for p in student.parameters():
    p.requires_grad = False

student.eval()

print("\nStudent FROZEN.")


# ================================================================
# 9. Evaluation Helpers
# ================================================================

def encode_language(text):

    tokens = torch.tensor(
        [tokenize(text)],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        z_color, z_shape = student(
            tokens
        )

    return z_color, z_shape


def teacher_attribute_from_sample(
    sample_index
):

    visual = torch.tensor(
        dataset.samples[
            sample_index
        ]["visual"],
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        (
            z_color,
            z_shape,
            color_logits,
            shape_logits
        ) = teacher(visual)

    return (
        z_color,
        z_shape,
        color_logits,
        shape_logits
    )


def mean_abs(a, b):

    return torch.mean(
        torch.abs(a - b)
    ).item()


# ================================================================
# 10. TEST 1
#     COLOR ONLY
# ================================================================

print("\n" + "=" * 72)
print("TEST 1: COLOR-ONLY LANGUAGE")
print("=" * 72)

lang_red = "push red object"
lang_blue = "push blue object"

z_red_color, z_red_shape = (
    encode_language(lang_red)
)

z_blue_color, z_blue_shape = (
    encode_language(lang_blue)
)

color_change = mean_abs(
    z_red_color,
    z_blue_color
)

shape_change = mean_abs(
    z_red_shape,
    z_blue_shape
)

print("\nLanguage A:", lang_red)
print("Language B:", lang_blue)

print(
    "\nColor latent difference:",
    f"{color_change:.8f}"
)

print(
    "Shape latent difference:",
    f"{shape_change:.8f}"
)

print("\nExpected:")
print("  Color -> LARGE")
print("  Shape -> SMALL")


# ================================================================
# 11. TEST 2
#     SHAPE ONLY
# ================================================================

print("\n" + "=" * 72)
print("TEST 2: SHAPE-ONLY LANGUAGE")
print("=" * 72)

lang_cube = "push cube"
lang_elongated = "push elongated object"

z_cube_color, z_cube_shape = (
    encode_language(lang_cube)
)

z_long_color, z_long_shape = (
    encode_language(lang_elongated)
)

color_change = mean_abs(
    z_cube_color,
    z_long_color
)

shape_change = mean_abs(
    z_cube_shape,
    z_long_shape
)

print("\nLanguage A:", lang_cube)
print("Language B:", lang_elongated)

print(
    "\nColor latent difference:",
    f"{color_change:.8f}"
)

print(
    "Shape latent difference:",
    f"{shape_change:.8f}"
)

print("\nExpected:")
print("  Color -> SMALL")
print("  Shape -> LARGE")


# ================================================================
# 12. TEST 3
#     COLOR + SHAPE
# ================================================================

print("\n" + "=" * 72)
print("TEST 3: COLOR + SHAPE")
print("=" * 72)

z_a_color, z_a_shape = (
    encode_language(
        "push red cube"
    )
)

z_b_color, z_b_shape = (
    encode_language(
        "push blue elongated"
    )
)

print(
    "\nColor difference:",
    f"{mean_abs(z_a_color, z_b_color):.8f}"
)

print(
    "Shape difference:",
    f"{mean_abs(z_a_shape, z_b_shape):.8f}"
)

print("\nExpected:")
print("  Both attributes -> changed")


# ================================================================
# 13. TEST 4
#     UNSPECIFIED ATTRIBUTE STABILITY
#
# Compare:
#
#   red object
#   red object
#
# across different worlds.
#
# Language representation MUST be identical.
# ================================================================

print("\n" + "=" * 72)
print("TEST 4: SAME LANGUAGE / DIFFERENT WORLD")
print("=" * 72)

z1_color, z1_shape = encode_language(
    "push red object"
)

z2_color, z2_shape = encode_language(
    "push red object"
)

print(
    "\nSame-language color difference:",
    f"{mean_abs(z1_color, z2_color):.10f}"
)

print(
    "Same-language shape difference:",
    f"{mean_abs(z1_shape, z2_shape):.10f}"
)

print("\nExpected:")
print("  Both -> 0")


# ================================================================
# 14. TEST 5
#     ATTRIBUTE CROSS-LEAKAGE
#
# Color change should NOT strongly change shape.
# Shape change should NOT strongly change color.
# ================================================================

print("\n" + "=" * 72)
print("TEST 5: ATTRIBUTE CROSS-LEAKAGE")
print("=" * 72)

# color-only change
c1, s1 = encode_language(
    "push red object"
)

c2, s2 = encode_language(
    "push blue object"
)

color_signal = mean_abs(c1, c2)
shape_leakage = mean_abs(s1, s2)

# shape-only change
c3, s3 = encode_language(
    "push cube"
)

c4, s4 = encode_language(
    "push elongated object"
)

shape_signal = mean_abs(s3, s4)
color_leakage = mean_abs(c3, c4)

print(
    "\nColor signal:",
    f"{color_signal:.8f}"
)

print(
    "Shape leakage under color change:",
    f"{shape_leakage:.8f}"
)

print(
    "\nShape signal:",
    f"{shape_signal:.8f}"
)

print(
    "Color leakage under shape change:",
    f"{color_leakage:.8f}"
)

print("\nExpected:")
print("  Color signal >> Shape leakage")
print("  Shape signal >> Color leakage")


# ================================================================
# 15. TEST 6
#     UNSPECIFIED ATTRIBUTES
# ================================================================

print("\n" + "=" * 72)
print("TEST 6: UNSPECIFIED ATTRIBUTE OUTPUT")
print("=" * 72)

z_red_obj_color, z_red_obj_shape = (
    encode_language(
        "push red object"
    )
)

z_blue_obj_color, z_blue_obj_shape = (
    encode_language(
        "push blue object"
    )
)

z_cube_color, z_cube_shape = (
    encode_language(
        "push cube"
    )

)

z_long_color, z_long_shape = (
    encode_language(
        "push elongated object"
    )
)

print(
    "\nColor-only language:"
)

print(
    "  Color latent magnitude:",
    f"{torch.mean(torch.abs(z_red_obj_color)).item():.8f}"
)

print(
    "  Shape latent magnitude:",
    f"{torch.mean(torch.abs(z_red_obj_shape)).item():.8f}"
)

print(
    "\nShape-only language:"
)

print(
    "  Color latent magnitude:",
    f"{torch.mean(torch.abs(z_cube_color)).item():.8f}"
)

print(
    "  Shape latent magnitude:",
    f"{torch.mean(torch.abs(z_cube_shape)).item():.8f}"
)

print(
    "\nIMPORTANT:"
)

print(
    "Magnitude alone is NOT interpreted as "
    "'specified' or 'unspecified'."
)

print(
    "The real criterion is cross-language selectivity."
)


# ================================================================
# 16. TEST 7
#     COMPOSITIONALITY
# ================================================================

print("\n" + "=" * 72)
print("TEST 7: ATTRIBUTE COMPOSITIONALITY")
print("=" * 72)

red_only_color, red_only_shape = (
    encode_language(
        "push red object"
    )
)

cube_only_color, cube_only_shape = (
    encode_language(
        "push cube"
    )
)

red_cube_color, red_cube_shape = (
    encode_language(
        "push red cube"
    )
)

print(
    "\nRed-only -> Red+Cube color difference:",
    f"{mean_abs(red_only_color, red_cube_color):.8f}"
)

print(
    "Cube-only -> Red+Cube shape difference:",
    f"{mean_abs(cube_only_shape, red_cube_shape):.8f}"
)

print("\nExpected:")
print(
    "  Adding cube should mainly affect shape"
)

print(
    "  Adding red should mainly affect color"
)


# ================================================================
# 17. FINAL REPORT
# ================================================================

print("\n" + "=" * 72)
print("EXPERIMENT 8B FINAL REPORT")
print("=" * 72)

print("""
Core hypothesis:

  Object Teacher:
      Vision -> Color latent
      Vision -> Shape latent

  Language Student:
      Language -> Color latent
      Language -> Shape latent

  Training:
      Only explicitly specified attributes receive loss.

Therefore:

  "push red object"
      -> recover RED
      -> do not require SHAPE

  "push elongated object"
      -> recover ELONGATED
      -> do not require COLOR

  "push red cube"
      -> recover RED + CUBE

  "push the object"
      -> recover neither

Critical tests:

  1. Color selectivity
  2. Shape selectivity
  3. Cross-attribute leakage
  4. Same-language invariance
  5. Attribute compositionality

Interpretation:

  SUCCESS:
      Language representation changes selectively
      according to the attribute expressed in language.

  FAILURE:
      Color and shape remain entangled,
      or unspecified attributes systematically vary
      with unrelated language changes.

Next step if SUCCESS:

      Partial WHAT (Language)
              +
      WHERE (Visual State)
              ↓
             VLA

      Language supplies only recoverable WHAT.
      Vision supplies spatial WHERE.
""")

print("=" * 72)
print("Experiment 8B finished.")
print("Attribute-wise partial object reconstruction tested.")
print("=" * 72)


AntEncoder Experiment 8B
Attribute-wise Object Teacher + Partial Language Student
Device: cpu

Vocabulary size: 9
Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'blue': 2, 'cube': 3, 'elongated': 4, 'object': 5, 'push': 6, 'red': 7, 'the': 8}

Dataset size: 6000

Teacher parameters: 164484

PHASE 1: ATTRIBUTE-WISE OBJECT TEACHER
Epoch 001/80 | Total=0.057573 | Color=0.028641 | Shape=0.028932
Epoch 002/80 | Total=0.000280 | Color=0.000136 | Shape=0.000144
Epoch 003/80 | Total=0.000195 | Color=0.000095 | Shape=0.000100
Epoch 004/80 | Total=0.000143 | Color=0.000069 | Shape=0.000074
Epoch 005/80 | Total=0.000109 | Color=0.000053 | Shape=0.000056
Epoch 010/80 | Total=0.000042 | Color=0.000020 | Shape=0.000022
Epoch 020/80 | Total=0.000014 | Color=0.000007 | Shape=0.000007
Epoch 030/80 | Total=0.000007 | Color=0.000004 | Shape=0.000004
Epoch 040/80 | Total=0.000004 | Color=0.000002 | Shape=0.000002
Epoch 050/80 | Total=0.000003 | Color=0.000001 | Shape=0.000001
Epoch 060/80 | Total=0.000002 | Color=0

In [ ]:
# ================================================================
# AntEncoder Experiment 9B
# Partial WHAT + Object-wise Visual WHERE -> Action
#
# Core hypothesis:
#
#   Language -> Partial WHAT
#       color
#       shape
#
#   Vision -> object-wise WHERE
#       object A XYZ
#       object B XYZ
#
#   Partial WHAT + WHERE -> Action
#
# ------------------------------------------------
# Critical tests
# ------------------------------------------------
#
# TEST 1
# Same language / different world
#
#   WHAT same
#   WHERE different
#   ACTION different
#
# TEST 2
# Same world / different object language
#
#   WHERE candidates same
#   WHAT changes
#   selected target changes
#   ACTION changes
#
# TEST 3
# Language swap
#
#   same visual world
#   red cube -> blue elongated
#   action should move to different object
#
# TEST 4
# WHERE swap
#
#   same language
#   swap object positions
#   action should follow the target object's new position
#
# ================================================================

import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ================================================================
# 0. Reproducibility
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("AntEncoder Experiment 9B")
print("Partial WHAT + Object-wise Visual WHERE -> Action")
print("=" * 72)

print("Device:", device)


# ================================================================
# 1. Vocabulary
# ================================================================

instructions = [

    "push red object",
    "push blue object",

    "push cube",
    "push elongated object",

    "push red cube",
    "push red elongated",
    "push blue cube",
    "push blue elongated",

    "push the object",
]

all_words = set()

for text in instructions:
    all_words.update(text.lower().split())

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for word in sorted(all_words):
    if word not in vocab:
        vocab[word] = len(vocab)

PAD_IDX = vocab["<PAD>"]
UNK_IDX = vocab["<UNK>"]

print("\nVocabulary size:", len(vocab))
print("Vocabulary:", vocab)


def tokenize(text, max_len=6):

    words = text.lower().split()

    ids = [
        vocab.get(word, UNK_IDX)
        for word in words
    ]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [
            PAD_IDX
        ] * (
            max_len - len(ids)
        )

    return ids


# ================================================================
# 2. Dataset
#
# Two objects.
#
# Each object has:
#
#   color
#   shape
#   xyz
#
# Vision representation:
#
#   object A physical 7D
#   object B physical 7D
#
# Total = 14D
#
# IMPORTANT:
# We preserve object-wise structure in the teacher target.
#
# ================================================================

class TwoObjectDataset(Dataset):

    COLORS = [
        "red",
        "blue"
    ]

    SHAPES = [
        "cube",
        "elongated"
    ]

    COLOR_VEC = {

        "red":
            np.array(
                [1.0, 0.0],
                dtype=np.float32
            ),

        "blue":
            np.array(
                [0.0, 1.0],
                dtype=np.float32
            )
    }

    SHAPE_VEC = {

        "cube":
            np.array(
                [1.0, 0.0],
                dtype=np.float32
            ),

        "elongated":
            np.array(
                [0.0, 1.0],
                dtype=np.float32
            )
    }

    def __init__(
        self,
        n_samples=8000,
        seed=42
    ):

        self.n_samples = n_samples

        self.rng = np.random.default_rng(
            seed
        )

        projection_rng = np.random.default_rng(
            12345
        )

        self.visual_projection = None

        self.samples = []

        combinations = [

            ("red", "cube"),
            ("red", "elongated"),
            ("blue", "cube"),
            ("blue", "elongated")
        ]

        for _ in range(n_samples):

            # ----------------------------------------------------
            # Select two different objects
            # ----------------------------------------------------

            pair = self.rng.choice(
                len(combinations),
                size=2,
                replace=False
            )

            objects = []

            for p in pair:

                color, shape = combinations[p]

                xyz = self.rng.uniform(

                    low=[
                        -1.0,
                        -1.0,
                        0.35
                    ],

                    high=[
                        1.0,
                        1.0,
                        1.0
                    ]

                ).astype(
                    np.float32
                )

                objects.append({

                    "color": color,
                    "shape": shape,
                    "xyz": xyz
                })

            # ----------------------------------------------------
            # Target object
            # ----------------------------------------------------

            target_idx = int(
                self.rng.integers(
                    0,
                    2
                )
            )

            target = objects[
                target_idx
            ]

            target_color = target["color"]
            target_shape = target["shape"]
            target_xyz = target["xyz"].copy()

            # ----------------------------------------------------
            # Language abstraction
            # ----------------------------------------------------

            language_type = int(
                self.rng.integers(
                    0,
                    4
                )
            )

            if language_type == 0:

                instruction = (
                    f"push {target_color} object"
                )

                color_specified = True
                shape_specified = False

            elif language_type == 1:

                instruction = (
                    f"push {target_shape} object"
                )

                color_specified = False
                shape_specified = True

            elif language_type == 2:

                instruction = (
                    f"push {target_color} {target_shape}"
                )

                color_specified = True
                shape_specified = True

            else:

                instruction = (
                    "push the object"
                )

                color_specified = False
                shape_specified = False

            # ----------------------------------------------------
            # Action
            #
            # Direction is determined by language-independent
            # random action direction.
            #
            # The important part is:
            #
            # ACTION depends on target XYZ.
            #
            # ----------------------------------------------------

            direction = int(
                self.rng.integers(
                    0,
                    2
                )
            )

            if direction == 0:

                action_name = "right"

                action = np.array(
                    [
                        target_xyz[0] + 0.35,
                        target_xyz[1],
                        target_xyz[2]
                    ],
                    dtype=np.float32
                )

            else:

                action_name = "left"

                action = np.array(
                    [
                        target_xyz[0] - 0.35,
                        target_xyz[1],
                        target_xyz[2]
                    ],
                    dtype=np.float32
                )

            # ----------------------------------------------------
            # Attribute targets
            # ----------------------------------------------------

            color_target = self.COLOR_VEC[
                target_color
            ]

            shape_target = self.SHAPE_VEC[
                target_shape
            ]

            # ----------------------------------------------------
            # Physical representation
            #
            # 7D per object:
            #
            #   color 2
            #   shape 2
            #   xyz   3
            #
            # total 14D
            # ----------------------------------------------------

            physical = []

            for obj in objects:

                physical.extend(
                    self.COLOR_VEC[
                        obj["color"]
                    ]
                )

                physical.extend(
                    self.SHAPE_VEC[
                        obj["shape"]
                    ]
                )

                physical.extend(
                    obj["xyz"]
                )

            physical = np.asarray(
                physical,
                dtype=np.float32
            )

            physical_dim = physical.shape[0]

            if self.visual_projection is None:

                self.visual_projection = (
                    projection_rng.normal(

                        0.0,

                        1.0 /
                        np.sqrt(
                            physical_dim
                        ),

                        size=(
                            physical_dim,
                            512
                        )
                    ).astype(
                        np.float32
                    )
                )

            # ----------------------------------------------------
            # Physical -> visual
            # ----------------------------------------------------

            visual = (
                physical
                @ self.visual_projection
            )

            visual += self.rng.normal(

                0.0,
                0.02,

                size=512

            ).astype(
                np.float32
            )

            self.samples.append({

                "instruction":
                    instruction,

                "tokens":
                    np.asarray(
                        tokenize(
                            instruction
                        ),
                        dtype=np.int64
                    ),

                "visual":
                    visual.astype(
                        np.float32
                    ),

                "physical":
                    physical,

                "objects":
                    objects,

                "target_idx":
                    target_idx,

                "target_color":
                    target_color,

                "target_shape":
                    target_shape,

                "target_xyz":
                    target_xyz,

                "color_target":
                    color_target,

                "shape_target":
                    shape_target,

                "color_specified":
                    color_specified,

                "shape_specified":
                    shape_specified,

                "action":
                    action,

                "action_name":
                    action_name
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        s = self.samples[idx]

        return {

            "tokens":
                torch.tensor(
                    s["tokens"],
                    dtype=torch.long
                ),

            "visual":
                torch.tensor(
                    s["visual"],
                    dtype=torch.float32
                ),

            "physical":
                torch.tensor(
                    s["physical"],
                    dtype=torch.float32
                ),

            "target_idx":
                torch.tensor(
                    s["target_idx"],
                    dtype=torch.long
                ),

            "color_target":
                torch.tensor(
                    s["color_target"],
                    dtype=torch.float32
                ),

            "shape_target":
                torch.tensor(
                    s["shape_target"],
                    dtype=torch.float32
                ),

            "target_xyz":
                torch.tensor(
                    s["target_xyz"],
                    dtype=torch.float32
                ),

            "color_specified":
                torch.tensor(
                    s["color_specified"],
                    dtype=torch.bool
                ),

            "shape_specified":
                torch.tensor(
                    s["shape_specified"],
                    dtype=torch.bool
                ),

            "action":
                torch.tensor(
                    s["action"],
                    dtype=torch.float32
                )
        }


# ================================================================
# Dataset
# ================================================================

dataset = TwoObjectDataset(
    n_samples=8000,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

print("\nDataset size:", len(dataset))
print(
    "Physical dimension:",
    dataset.samples[0]["physical"].shape[0]
)
print(
    "Visual dimension:",
    dataset.samples[0]["visual"].shape[0]
)


# ================================================================
# 3. Object Teacher
#
# Vision -> WHAT
#
# Separate color / shape latents.
# ================================================================

class ObjectTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        latent_dim=64
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU()
        )

        self.color_head = nn.Sequential(

            nn.Linear(
                128,
                latent_dim
            ),

            nn.Tanh()
        )

        self.shape_head = nn.Sequential(

            nn.Linear(
                128,
                latent_dim
            ),

            nn.Tanh()
        )

        self.color_decoder = nn.Linear(
            latent_dim,
            2
        )

        self.shape_decoder = nn.Linear(
            latent_dim,
            2
        )

    def forward(self, visual):

        h = self.encoder(
            visual
        )

        z_color = self.color_head(
            h
        )

        z_shape = self.shape_head(
            h
        )

        color_logits = self.color_decoder(
            z_color
        )

        shape_logits = self.shape_decoder(
            z_shape
        )

        return (
            z_color,
            z_shape,
            color_logits,
            shape_logits
        )


# ================================================================
# 4. Object-wise State Teacher
#
# Vision -> WHERE_A + WHERE_B
#
# IMPORTANT:
#
# State teacher outputs BOTH object locations.
#
# No language input.
#
# ================================================================

class ObjectWiseStateTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        state_dim=64
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU()
        )

        self.state_head = nn.Sequential(

            nn.Linear(
                128,
                state_dim * 2
            ),

            nn.Tanh()
        )

        self.decoder = nn.Sequential(

            nn.Linear(
                state_dim * 2,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                6
            )
        )

    def forward(self, visual):

        h = self.encoder(
            visual
        )

        z_state = self.state_head(
            h
        )

        xyz = self.decoder(
            z_state
        )

        return (
            z_state,
            xyz
        )


object_teacher = ObjectTeacher().to(
    device
)

state_teacher = ObjectWiseStateTeacher().to(
    device
)

teacher_parameters = (
    list(object_teacher.parameters())
    +
    list(state_teacher.parameters())
)

teacher_optimizer = torch.optim.AdamW(

    teacher_parameters,

    lr=2e-3,

    weight_decay=1e-4
)

print(
    "\nTeacher parameters:",
    sum(
        p.numel()
        for p in teacher_parameters
    )
)


# ================================================================
# 5. PHASE 1
# ================================================================

print("\n" + "=" * 72)
print("PHASE 1: OBJECT + OBJECT-WISE WHERE TEACHERS")
print("=" * 72)

teacher_epochs = 80

for epoch in range(
    teacher_epochs
):

    object_teacher.train()
    state_teacher.train()

    total_sum = 0.0
    color_sum = 0.0
    shape_sum = 0.0
    state_sum = 0.0

    for batch in loader:

        visual = batch["visual"].to(device)

        physical = batch["physical"].to(device)

        color_target = batch[
            "color_target"
        ].to(device)

        shape_target = batch[
            "shape_target"
        ].to(device)

        teacher_optimizer.zero_grad()

        (
            z_color,
            z_shape,
            color_logits,
            shape_logits
        ) = object_teacher(
            visual
        )

        (
            z_state,
            state_pred
        ) = state_teacher(
            visual
        )

        # --------------------------------------------------------
        # WHAT
        # --------------------------------------------------------

        color_loss = F.mse_loss(
            color_logits,
            color_target
        )

        shape_loss = F.mse_loss(
            shape_logits,
            shape_target
        )

        # --------------------------------------------------------
        # WHERE
        #
        # physical:
        #
        # [A_color,
        #  A_shape,
        #  A_xyz,
        #  B_color,
        #  B_shape,
        #  B_xyz]
        #
        # XYZ positions:
        #
        # indices 4:7
        # indices 11:14
        # --------------------------------------------------------

        target_xyz_all = torch.cat(

            [
                physical[:, 4:7],
                physical[:, 11:14]
            ],

            dim=-1
        )

        state_loss = F.mse_loss(
            state_pred,
            target_xyz_all
        )

        loss = (
            color_loss
            +
            shape_loss
            +
            state_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            teacher_parameters,
            1.0
        )

        teacher_optimizer.step()

        bs = visual.size(0)

        total_sum += (
            loss.item() * bs
        )

        color_sum += (
            color_loss.item() * bs
        )

        shape_sum += (
            shape_loss.item() * bs
        )

        state_sum += (
            state_loss.item() * bs
        )

    n = len(dataset)

    if (
        epoch < 5
        or (epoch + 1) % 10 == 0
        or epoch == teacher_epochs - 1
    ):

        print(

            f"Epoch {epoch+1:03d}/"
            f"{teacher_epochs} | "

            f"Total="
            f"{total_sum/n:.6f} | "

            f"Color="
            f"{color_sum/n:.6f} | "

            f"Shape="
            f"{shape_sum/n:.6f} | "

            f"WHERE="
            f"{state_sum/n:.6f}"
        )


# ================================================================
# Freeze teachers
# ================================================================

for p in object_teacher.parameters():
    p.requires_grad = False

for p in state_teacher.parameters():
    p.requires_grad = False

object_teacher.eval()
state_teacher.eval()

print("\nTeachers FROZEN.")


# ================================================================
# 6. Language Student
# ================================================================

class PartialLanguageStudent(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=96,
        hidden_dim=128,
        latent_dim=64
    ):

        super().__init__()

        self.embedding = nn.Embedding(

            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.gru = nn.GRU(

            embedding_dim,
            hidden_dim,

            batch_first=True,

            bidirectional=True
        )

        self.color_head = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                latent_dim
            ),

            nn.Tanh()
        )

        self.shape_head = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                latent_dim
            ),

            nn.Tanh()
        )

    def forward(self, tokens):

        x = self.embedding(
            tokens
        )

        _, hidden = self.gru(
            x
        )

        h = torch.cat(

            [
                hidden[-2],
                hidden[-1]
            ],

            dim=-1
        )

        z_color = self.color_head(
            h
        )

        z_shape = self.shape_head(
            h
        )

        return (
            z_color,
            z_shape
        )


student = PartialLanguageStudent(
    vocab_size=len(vocab)
).to(device)

student_optimizer = torch.optim.AdamW(

    student.parameters(),

    lr=2e-3,

    weight_decay=1e-4
)

print(
    "\nStudent parameters:",
    sum(
        p.numel()
        for p in student.parameters()
    )
)


# ================================================================
# 7. PHASE 2
#
# Partial WHAT
#
# Only specified attributes receive distillation.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 2: PARTIAL LANGUAGE WHAT")
print("=" * 72)

student_epochs = 50

for epoch in range(
    student_epochs
):

    student.train()

    total_sum = 0.0
    color_sum = 0.0
    shape_sum = 0.0

    for batch in loader:

        tokens = batch[
            "tokens"
        ].to(device)

        visual = batch[
            "visual"
        ].to(device)

        color_mask = batch[
            "color_specified"
        ].to(device)

        shape_mask = batch[
            "shape_specified"
        ].to(device)

        student_optimizer.zero_grad()

        (
            z_color_L,
            z_shape_L
        ) = student(
            tokens
        )

        with torch.no_grad():

            (
                z_color_V,
                z_shape_V,
                _,
                _
            ) = object_teacher(
                visual
            )

        color_distill = (

            z_color_L
            -
            z_color_V

        ).pow(2).mean(
            dim=-1
        )

        shape_distill = (

            z_shape_L
            -
            z_shape_V

        ).pow(2).mean(
            dim=-1
        )

        if color_mask.any():

            color_loss = (
                color_distill[
                    color_mask
                ].mean()
            )

        else:

            color_loss = torch.tensor(
                0.0,
                device=device
            )

        if shape_mask.any():

            shape_loss = (
                shape_distill[
                    shape_mask
                ].mean()
            )

        else:

            shape_loss = torch.tensor(
                0.0,
                device=device
            )

        loss = (
            color_loss
            +
            shape_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            1.0
        )

        student_optimizer.step()

        bs = tokens.size(0)

        total_sum += (
            loss.item() * bs
        )

        color_sum += (
            color_loss.item() * bs
        )

        shape_sum += (
            shape_loss.item() * bs
        )

    n = len(dataset)

    if (
        epoch < 5
        or (epoch + 1) % 10 == 0
        or epoch == student_epochs - 1
    ):

        print(

            f"Student Epoch "
            f"{epoch+1:02d}/"
            f"{student_epochs} | "

            f"Total="
            f"{total_sum/n:.6f} | "

            f"Color="
            f"{color_sum/n:.6f} | "

            f"Shape="
            f"{shape_sum/n:.6f}"
        )


# ================================================================
# Freeze student
# ================================================================

for p in student.parameters():
    p.requires_grad = False

student.eval()

print("\nStudent FROZEN.")


# ================================================================
# 8. Factorized VLA
#
# Input:
#
#   Language WHAT
#       color latent
#       shape latent
#
#   Vision WHERE
#       object A XYZ
#       object B XYZ
#
# Output:
#       action XYZ
#
# ================================================================

class FactorizedVLA(nn.Module):

    def __init__(
        self,
        what_dim=64,
        where_dim=128,
        action_dim=3
    ):

        super().__init__()

        input_dim = (
            what_dim
            + what_dim
            + where_dim
        )

        print(
            f"VLA input_dim = "
            f"{what_dim} + {what_dim} + {where_dim}"
            f" = {input_dim}"
        )

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                action_dim
            )
        )

    def forward(
        self,
        z_color,
        z_shape,
        z_where
    ):

        x = torch.cat(
            [
                z_color,
                z_shape,
                z_where
            ],
            dim=-1
        )

        return self.net(x)

vla = FactorizedVLA(
    what_dim=64,
    where_dim=128,
    action_dim=3
).to(device)


vla_optimizer = torch.optim.AdamW(

    vla.parameters(),

    lr=2e-3,

    weight_decay=1e-4
)

print(
    "\nVLA parameters:",
    sum(
        p.numel()
        for p in vla.parameters()
    )
)


# ================================================================
# 9. PHASE 3
# ================================================================

print("\n" + "=" * 72)
print("PHASE 3: PARTIAL WHAT + OBJECT-WISE WHERE -> ACTION")
print("=" * 72)

vla_epochs = 40

for epoch in range(
    vla_epochs
):

    vla.train()

    action_sum = 0.0

    for batch in loader:

        tokens = batch[
            "tokens"
        ].to(device)

        visual = batch[
            "visual"
        ].to(device)

        action = batch[
            "action"
        ].to(device)

        vla_optimizer.zero_grad()

        with torch.no_grad():

            (
                z_color_L,
                z_shape_L
            ) = student(
                tokens
            )

            (
                z_where,
                _
            ) = state_teacher(
                visual
            )

        action_pred = vla(

            z_color_L,
            z_shape_L,
            z_where
        )

        action_loss = F.mse_loss(

            action_pred,
            action
        )

        action_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            vla.parameters(),
            1.0
        )

        vla_optimizer.step()

        action_sum += (
            action_loss.item()
            *
            tokens.size(0)
        )

    epoch_loss = (
        action_sum
        /
        len(dataset)
    )

    print(

        f"VLA Epoch "
        f"{epoch+1:02d}/"
        f"{vla_epochs} | "

        f"Action="
        f"{epoch_loss:.6f}"
    )


# ================================================================
# 10. Helpers
# ================================================================

def encode_language(
    instruction
):

    tokens = torch.tensor(

        [
            tokenize(
                instruction
            )
        ],

        dtype=torch.long,

        device=device
    )

    with torch.no_grad():

        z_color, z_shape = student(
            tokens
        )

    return (
        z_color,
        z_shape
    )


def encode_where(
    sample_index
):

    visual = torch.tensor(

        dataset.samples[
            sample_index
        ]["visual"],

        dtype=torch.float32,

        device=device

    ).unsqueeze(0)

    with torch.no_grad():

        z_where, xyz = state_teacher(
            visual
        )

    return (
        z_where,
        xyz
    )


def predict(
    instruction,
    sample_index
):

    z_color, z_shape = (
        encode_language(
            instruction
        )
    )

    z_where, xyz = (
        encode_where(
            sample_index
        )
    )

    with torch.no_grad():

        action = vla(

            z_color,
            z_shape,
            z_where
        )

    return (
        z_color,
        z_shape,
        z_where,
        xyz,
        action
    )


# ================================================================
# 11. TEST 1
#
# Same language / different world
# ================================================================

print("\n" + "=" * 72)
print("TEST 1: SAME LANGUAGE / DIFFERENT WORLD")
print("=" * 72)

instruction = "push red cube"

candidates = [

    i

    for i, s in enumerate(
        dataset.samples
    )

    if s["instruction"] == instruction
]

idx_a = candidates[0]
idx_b = candidates[1]

(
    color_a,
    shape_a,
    where_a,
    xyz_a,
    action_a
) = predict(
    instruction,
    idx_a
)

(
    color_b,
    shape_b,
    where_b,
    xyz_b,
    action_b
) = predict(
    instruction,
    idx_b
)

print(
    "\nWorld A objects:"
)

for i, obj in enumerate(
    dataset.samples[idx_a]["objects"]
):

    print(
        i,
        obj["color"],
        obj["shape"],
        obj["xyz"]
    )

print(
    "\nWorld B objects:"
)

for i, obj in enumerate(
    dataset.samples[idx_b]["objects"]
):

    print(
        i,
        obj["color"],
        obj["shape"],
        obj["xyz"]
    )

print(
    "\nLanguage color difference:",
    torch.mean(
        torch.abs(
            color_a - color_b
        )
    ).item()
)

print(
    "Language shape difference:",
    torch.mean(
        torch.abs(
            shape_a - shape_b
        )
    ).item()
)

print(
    "WHERE difference:",
    torch.mean(
        torch.abs(
            where_a - where_b
        )
    ).item()
)

print(
    "Action difference:",
    torch.mean(
        torch.abs(
            action_a - action_b
        )
    ).item()
)

print("\nExpected:")
print("  WHAT  -> SMALL")
print("  WHERE -> LARGE")
print("  Action -> LARGE")


# ================================================================
# 12. TEST 2
#
# Same world / different object language
#
# Find world containing:
#
#   red cube
#   blue elongated
# ================================================================

print("\n" + "=" * 72)
print("TEST 2: SAME WORLD / DIFFERENT OBJECT LANGUAGE")
print("=" * 72)

world_idx = None

for i, s in enumerate(
    dataset.samples
):

    combos = [

        (
            obj["color"],
            obj["shape"]
        )

        for obj in s["objects"]
    ]

    if (

        ("red", "cube") in combos

        and

        ("blue", "elongated") in combos

    ):

        world_idx = i
        break


if world_idx is None:

    raise RuntimeError(
        "No suitable world found."
    )


instruction_a = "push red cube"
instruction_b = "push blue elongated"

(
    color_a,
    shape_a,
    where_a,
    xyz_a,
    action_a
) = predict(
    instruction_a,
    world_idx
)

(
    color_b,
    shape_b,
    where_b,
    xyz_b,
    action_b
) = predict(
    instruction_b,
    world_idx
)

world = dataset.samples[
    world_idx
]

print("\nWorld:")

for i, obj in enumerate(
    world["objects"]
):

    print(
        i,
        obj["color"],
        obj["shape"],
        obj["xyz"]
    )

print(
    "\nLanguage A:",
    instruction_a
)

print(
    "Language B:",
    instruction_b
)

print(
    "\nWHERE difference:",
    torch.mean(
        torch.abs(
            where_a - where_b
        )
    ).item()
)

print(
    "Color WHAT difference:",
    torch.mean(
        torch.abs(
            color_a - color_b
        )
    ).item()
)

print(
    "Shape WHAT difference:",
    torch.mean(
        torch.abs(
            shape_a - shape_b
        )
    ).item()
)

print(
    "Action difference:",
    torch.mean(
        torch.abs(
            action_a - action_b
        )
    ).item()
)

print("\nExpected:")
print("  WHERE -> SMALL / identical")
print("  WHAT  -> LARGE")
print("  Action -> LARGE")


# ================================================================
# 13. TEST 3
#
# Language swap
#
# Same visual world.
#
# Compare action against actual target XYZ.
# ================================================================

print("\n" + "=" * 72)
print("TEST 3: LANGUAGE SWAP / TARGET SELECTION")
print("=" * 72)

objects = world["objects"]

red_idx = None
blue_idx = None

for i, obj in enumerate(objects):

    if (
        obj["color"] == "red"
        and
        obj["shape"] == "cube"
    ):

        red_idx = i

    if (
        obj["color"] == "blue"
        and
        obj["shape"] == "elongated"
    ):

        blue_idx = i


red_xyz = objects[
    red_idx
]["xyz"]

blue_xyz = objects[
    blue_idx
]["xyz"]

print(
    "\nRed cube XYZ:",
    red_xyz
)

print(
    "Blue elongated XYZ:",
    blue_xyz
)

print(
    "\nPredicted action for red cube:",
    action_a.detach().cpu().numpy()
)

print(
    "Predicted action for blue elongated:",
    action_b.detach().cpu().numpy()
)

print(
    "\nDistance between target XYZs:",
    np.mean(
        np.abs(
            red_xyz
            -
            blue_xyz
        )
    )
)

print(
    "Distance between actions:",
    torch.mean(
        torch.abs(
            action_a
            -
            action_b
        )
    ).item()
)

print("\nExpected:")
print(
    "  Different language -> different selected object"
)

print(
    "  Different target position -> different action"
)


# ================================================================
# 14. TEST 4
#
# WHERE swap
#
# Construct an artificial visual world where the SAME objects
# exchange positions.
#
# Language remains identical.
#
# The action should follow the object's new position.
#
# We create modified physical representations and project them
# through the same visual projection.
# ================================================================

print("\n" + "=" * 72)
print("TEST 4: WHERE SWAP")
print("=" * 72)

base_sample = world

physical_original = (
    base_sample["physical"]
    .copy()
)

physical_swapped = (
    physical_original.copy()
)

# swap XYZ only
xyz_a_orig = physical_original[
    4:7
].copy()

xyz_b_orig = physical_original[
    11:14
].copy()

physical_swapped[
    4:7
] = xyz_b_orig

physical_swapped[
    11:14
] = xyz_a_orig

rng = np.random.default_rng(
    999
)

visual_swapped = (

    physical_swapped
    @ dataset.visual_projection

)

visual_swapped += rng.normal(
    0.0,
    0.02,
    size=512
).astype(
    np.float32
)

visual_original_tensor = torch.tensor(

    base_sample["visual"],

    dtype=torch.float32,

    device=device

).unsqueeze(0)

visual_swapped_tensor = torch.tensor(

    visual_swapped,

    dtype=torch.float32,

    device=device

).unsqueeze(0)

with torch.no_grad():

    (
        where_original,
        _
    ) = state_teacher(
        visual_original_tensor
    )

    (
        where_swapped,
        _
    ) = state_teacher(
        visual_swapped_tensor
    )

    z_color, z_shape = (
        student(
            torch.tensor(
                [
                    tokenize(
                        "push red cube"
                    )
                ],
                dtype=torch.long,
                device=device
            )
        )
    )

    action_original = vla(
        z_color,
        z_shape,
        where_original
    )

    action_swapped = vla(
        z_color,
        z_shape,
        where_swapped
    )

print(
    "\nOriginal red cube XYZ:",
    xyz_a_orig
)

print(
    "Original blue elongated XYZ:",
    xyz_b_orig
)

print(
    "\nSwapped red cube XYZ:",
    xyz_b_orig
)

print(
    "Swapped blue elongated XYZ:",
    xyz_a_orig
)

print(
    "\nAction original:",
    action_original.detach().cpu().numpy()
)

print(
    "Action after WHERE swap:",
    action_swapped.detach().cpu().numpy()
)

print(
    "\nAction difference:",
    torch.mean(
        torch.abs(
            action_original
            -
            action_swapped
        )
    ).item()
)

print("\nExpected:")
print(
    "  Language -> SAME"
)

print(
    "  Object identity -> SAME"
)

print(
    "  WHERE -> changes"
)

print(
    "  Action -> follows changed WHERE"
)


# ================================================================
# 15. TEST 5
#
# WHAT / WHERE ablation
# ================================================================

print("\n" + "=" * 72)
print("TEST 5: WHAT / WHERE ABLATION")
print("=" * 72)

instruction = "push red cube"

z_color, z_shape = (
    encode_language(
        instruction
    )
)

z_where, _ = (
    encode_where(
        world_idx
    )
)

with torch.no_grad():

    full_action = vla(
        z_color,
        z_shape,
        z_where
    )

    zero_what = vla(
        torch.zeros_like(
            z_color
        ),
        torch.zeros_like(
            z_shape
        ),
        z_where
    )

    zero_where = vla(
        z_color,
        z_shape,
        torch.zeros_like(
            z_where
        )
    )

print(
    "\nFull WHAT + WHERE:",
    full_action.detach().cpu().numpy()
)

print(
    "Zero WHAT:",
    zero_what.detach().cpu().numpy()
)

print(
    "Zero WHERE:",
    zero_where.detach().cpu().numpy()
)

print("\nInterpretation:")
print(
    "  Removing WHAT should impair object selection."
)

print(
    "  Removing WHERE should impair spatially correct action."
)


# ================================================================
# 16. FINAL REPORT
# ================================================================

print("\n" + "=" * 72)
print("EXPERIMENT 9B FINAL REPORT")
print("=" * 72)

print(
    """
Core hypothesis:

    Language
        -> Partial WHAT

    Vision
        -> Object-wise WHERE

    Partial WHAT + WHERE
        -> Action


Desired factorization:

    WHAT:
        Which object / attributes?

    WHERE:
        Where are the candidate objects?

    ACTION:
        Move toward / act on the object selected by WHAT
        using its WHERE information.


Critical evidence:

    TEST 1
        Same language
        Different world

        WHAT should remain invariant.
        WHERE should change.
        Action should change.


    TEST 2
        Same world
        Different object language

        WHERE should remain invariant.
        WHAT should change.
        Action should change.


    TEST 3
        Language swap

        Different language should select
        different object.


    TEST 4
        WHERE swap

        Same language should follow
        the object's new spatial position.


    TEST 5
        WHAT / WHERE ablation

        WHAT is required for object selection.
        WHERE is required for spatial grounding.


Interpretation:

    SUCCESS:

        Language provides only Partial WHAT.

        Vision provides object-wise WHERE.

        VLA combines them to select and act on
        the correct object.


    This is the intended:

        Partial WHAT
             +
        Visual WHERE
             ↓
           Action


Next step:

        Partial WHAT + WHERE
                    ↓
                  VLA
                    ↓
                ACTION

    This is the bridge from the factorization experiments
    toward the actual AntEncoder VLA formulation.
"""
)

print("=" * 72)
print("Experiment 9B finished.")
print("=" * 72)


AntEncoder Experiment 9B
Partial WHAT + Object-wise Visual WHERE -> Action
Device: cpu

Vocabulary size: 9
Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'blue': 2, 'cube': 3, 'elongated': 4, 'object': 5, 'push': 6, 'red': 7, 'the': 8}

Dataset size: 8000
Physical dimension: 14
Visual dimension: 512

Teacher parameters: 379018

PHASE 1: OBJECT + OBJECT-WISE WHERE TEACHERS
Epoch 001/80 | Total=0.420593 | Color=0.187342 | Shape=0.187799 | WHERE=0.045452
Epoch 002/80 | Total=0.343480 | Color=0.171266 | Shape=0.170092 | WHERE=0.002122
Epoch 003/80 | Total=0.339315 | Color=0.169021 | Shape=0.169512 | WHERE=0.000782
Epoch 004/80 | Total=0.338101 | Color=0.168991 | Shape=0.168405 | WHERE=0.000705
Epoch 005/80 | Total=0.336472 | Color=0.169245 | Shape=0.166587 | WHERE=0.000640
Epoch 010/80 | Total=0.334368 | Color=0.167782 | Shape=0.166220 | WHERE=0.000366
Epoch 020/80 | Total=0.330096 | Color=0.165451 | Shape=0.164298 | WHERE=0.000346
Epoch 030/80 | Total=0.324350 | Color=0.162566 | Shape=0.161520 | WH

In [ ]:
# ================================================================
# AntEncoder Experiment 10
# WORLD-COMPLETE WHAT + PARTIAL LANGUAGE WHAT
# Object Matching -> WHERE -> ACTION
#
# IMPORTANT DESIGN
#
#   Teacher = WORLD
#   Student = LANGUAGE
#
# Teacher:
#   Vision -> ALL OBJECTS' WHAT
#   Vision -> ALL OBJECTS' WHERE
#
# Student:
#   Language -> PARTIAL WHAT
#
# VLA:
#   Partial WHAT
#       +
#   World-complete WHAT
#       ↓
#   Object matching
#       ↓
#   Selected WHERE
#       ↓
#   ACTION
#
# Teacher NEVER receives target_idx.
# ================================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ================================================================
# 0. Reproducibility
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("AntEncoder Experiment 10")
print("WORLD-COMPLETE WHAT + PARTIAL LANGUAGE WHAT")
print("Object Matching -> WHERE -> ACTION")
print("=" * 72)
print("Device:", device)


# ================================================================
# 1. Vocabulary
# ================================================================

instructions = [
    "push red object",
    "push blue object",
    "push cube",
    "push elongated object",
    "push red cube",
    "push red elongated",
    "push blue cube",
    "push blue elongated",
    "push the object",
]

all_words = set()

for text in instructions:
    all_words.update(text.lower().split())

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for word in sorted(all_words):
    if word not in vocab:
        vocab[word] = len(vocab)

PAD_IDX = vocab["<PAD>"]
UNK_IDX = vocab["<UNK>"]

print("\nVocabulary size:", len(vocab))
print("Vocabulary:", vocab)


def tokenize(text, max_len=6):

    words = text.lower().split()

    ids = [
        vocab.get(word, UNK_IDX)
        for word in words
    ]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [PAD_IDX] * (
            max_len - len(ids)
        )

    return ids


# ================================================================
# 2. Dataset
#
# Two objects per world.
#
# IMPORTANT:
# The visual input contains the COMPLETE WORLD.
#
# Object attributes:
#   color = 2D one-hot
#   shape = 2D one-hot
#   xyz   = 3D
#
# Each object = 7D
# Two objects = 14D
# ================================================================

class WorldObjectDataset(Dataset):

    COLORS = ["red", "blue"]
    SHAPES = ["cube", "elongated"]

    COLOR_VEC = {
        "red": np.array(
            [1.0, 0.0],
            dtype=np.float32
        ),
        "blue": np.array(
            [0.0, 1.0],
            dtype=np.float32
        ),
    }

    SHAPE_VEC = {
        "cube": np.array(
            [1.0, 0.0],
            dtype=np.float32
        ),
        "elongated": np.array(
            [0.0, 1.0],
            dtype=np.float32
        ),
    }

    def __init__(
        self,
        n_samples=8000,
        seed=42
    ):

        self.rng = np.random.default_rng(seed)

        self.samples = []

        combinations = [
            ("red", "cube"),
            ("red", "elongated"),
            ("blue", "cube"),
            ("blue", "elongated"),
        ]

        projection_rng = np.random.default_rng(12345)

        self.visual_projection = None

        for _ in range(n_samples):

            # ----------------------------------------------------
            # Choose two different objects
            # ----------------------------------------------------

            pair = self.rng.choice(
                len(combinations),
                size=2,
                replace=False
            )

            objects = []

            for p in pair:

                color, shape = combinations[p]

                xyz = self.rng.uniform(
                    low=[
                        -1.0,
                        -1.0,
                        0.35
                    ],
                    high=[
                        1.0,
                        1.0,
                        1.0
                    ]
                ).astype(np.float32)

                objects.append({
                    "color": color,
                    "shape": shape,
                    "xyz": xyz,
                })

            # ----------------------------------------------------
            # Target object
            #
            # This is ONLY used to generate language/action.
            #
            # It is NOT given to Teacher.
            # ----------------------------------------------------

            target_idx = int(
                self.rng.integers(0, 2)
            )

            target = objects[target_idx]

            color = target["color"]
            shape = target["shape"]

            # ----------------------------------------------------
            # Language abstraction
            # ----------------------------------------------------

            language_type = int(
                self.rng.integers(0, 4)
            )

            if language_type == 0:

                instruction = (
                    f"push {color} object"
                )

                color_specified = True
                shape_specified = False

            elif language_type == 1:

                instruction = (
                    f"push {shape} object"
                )

                color_specified = False
                shape_specified = True

            elif language_type == 2:

                instruction = (
                    f"push {color} {shape}"
                )

                color_specified = True
                shape_specified = True

            else:

                instruction = (
                    "push the object"
                )

                color_specified = False
                shape_specified = False

            # ----------------------------------------------------
            # Action
            #
            # Direction is language-independent.
            # Target location determines spatial part.
            # ----------------------------------------------------

            direction = int(
                self.rng.integers(0, 2)
            )

            if direction == 0:

                action = target["xyz"].copy()
                action[0] += 0.35

            else:

                action = target["xyz"].copy()
                action[0] -= 0.35

            action = action.astype(
                np.float32
            )

            # ----------------------------------------------------
            # Complete world representation
            # ----------------------------------------------------

            physical_objects = []

            for obj in objects:

                physical_objects.append(
                    np.concatenate([
                        self.COLOR_VEC[
                            obj["color"]
                        ],

                        self.SHAPE_VEC[
                            obj["shape"]
                        ],

                        obj["xyz"],
                    ])
                )

            physical_objects = np.asarray(
                physical_objects,
                dtype=np.float32
            )

            # [2, 7]

            physical_flat = (
                physical_objects
                .reshape(-1)
            )

            physical_dim = (
                physical_flat.shape[0]
            )

            # ----------------------------------------------------
            # Fixed visual projection
            # ----------------------------------------------------

            if self.visual_projection is None:

                self.visual_projection = (
                    projection_rng.normal(
                        0.0,
                        1.0 / np.sqrt(
                            physical_dim
                        ),
                        size=(
                            physical_dim,
                            512
                        )
                    ).astype(
                        np.float32
                    )
                )

            visual = (
                physical_flat
                @ self.visual_projection
            )

            visual += self.rng.normal(
                0.0,
                0.02,
                size=512
            ).astype(np.float32)

            # ----------------------------------------------------
            # World targets
            # ----------------------------------------------------

            world_color = np.asarray([
                self.COLOR_VEC[
                    obj["color"]
                ]
                for obj in objects
            ], dtype=np.float32)

            world_shape = np.asarray([
                self.SHAPE_VEC[
                    obj["shape"]
                ]
                for obj in objects
            ], dtype=np.float32)

            world_xyz = np.asarray([
                obj["xyz"]
                for obj in objects
            ], dtype=np.float32)

            self.samples.append({

                "instruction": instruction,

                "tokens": np.asarray(
                    tokenize(instruction),
                    dtype=np.int64
                ),

                "visual": visual.astype(
                    np.float32
                ),

                "objects": objects,

                "target_idx": target_idx,

                "world_color":
                    world_color,

                "world_shape":
                    world_shape,

                "world_xyz":
                    world_xyz,

                "color_specified":
                    color_specified,

                "shape_specified":
                    shape_specified,

                "action": action,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        s = self.samples[idx]

        return {

            "tokens": torch.tensor(
                s["tokens"],
                dtype=torch.long
            ),

            "visual": torch.tensor(
                s["visual"],
                dtype=torch.float32
            ),

            "world_color": torch.tensor(
                s["world_color"],
                dtype=torch.float32
            ),

            "world_shape": torch.tensor(
                s["world_shape"],
                dtype=torch.float32
            ),

            "world_xyz": torch.tensor(
                s["world_xyz"],
                dtype=torch.float32
            ),

            "target_idx": torch.tensor(
                s["target_idx"],
                dtype=torch.long
            ),

            "color_specified": torch.tensor(
                s["color_specified"],
                dtype=torch.bool
            ),

            "shape_specified": torch.tensor(
                s["shape_specified"],
                dtype=torch.bool
            ),

            "action": torch.tensor(
                s["action"],
                dtype=torch.float32
            ),
        }


dataset = WorldObjectDataset(
    n_samples=8000,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

print("\nDataset size:", len(dataset))
print(
    "Physical dimension:",
    dataset.samples[0]["world_color"].shape[0] * 2
    + dataset.samples[0]["world_shape"].shape[0] * 2
    + dataset.samples[0]["world_xyz"].shape[0] * 2
)
print("Visual dimension: 512")


# ================================================================
# 3. WORLD WHAT TEACHER
#
# Vision -> ALL OBJECTS' WHAT
#
# Output:
#
#   z_color_world [B, 2, D]
#   z_shape_world [B, 2, D]
#
# Teacher DOES NOT receive target_idx.
# ================================================================

class WorldWHATTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        what_dim=64,
        num_objects=2
    ):

        super().__init__()

        self.num_objects = num_objects

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU()
        )

        self.color_head = nn.Sequential(

            nn.Linear(
                128,
                num_objects * what_dim
            ),

            nn.Tanh()
        )

        self.shape_head = nn.Sequential(

            nn.Linear(
                128,
                num_objects * what_dim
            ),

            nn.Tanh()
        )

        self.color_decoder = nn.Linear(
            what_dim,
            2
        )

        self.shape_decoder = nn.Linear(
            what_dim,
            2
        )

    def forward(self, visual):

        h = self.encoder(visual)

        z_color = self.color_head(h)

        z_shape = self.shape_head(h)

        z_color = z_color.view(
            -1,
            self.num_objects,
            64
        )

        z_shape = z_shape.view(
            -1,
            self.num_objects,
            64
        )

        return (
            z_color,
            z_shape
        )


# ================================================================
# 4. OBJECT-WISE WHERE TEACHER
#
# Vision -> WHERE for ALL objects
#
# Output:
#
#   z_where [B, 2, 128]
# ================================================================

class WorldWHERETeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        where_dim=128,
        num_objects=2
    ):

        super().__init__()

        self.num_objects = num_objects

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                num_objects * where_dim
            ),

            nn.Tanh()
        )

        self.decoder = nn.Linear(
            where_dim,
            3
        )

    def forward(self, visual):

        z_where = self.encoder(
            visual
        )

        z_where = z_where.view(
            -1,
            self.num_objects,
            128
        )

        return z_where


what_teacher = WorldWHATTeacher().to(device)
where_teacher = WorldWHERETeacher().to(device)

teacher_parameters = (
    list(what_teacher.parameters())
    +
    list(where_teacher.parameters())
)

teacher_optimizer = torch.optim.AdamW(
    teacher_parameters,
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nTeacher parameters:",
    sum(
        p.numel()
        for p in teacher_parameters
    )
)


# ================================================================
# 5. PHASE 1
#
# Train Teacher on COMPLETE WORLD
#
# No target_idx is used here.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 1: WORLD-COMPLETE WHAT + OBJECT-WISE WHERE")
print("=" * 72)

teacher_epochs = 80

for epoch in range(
    teacher_epochs
):

    what_teacher.train()
    where_teacher.train()

    total_sum = 0.0
    color_sum = 0.0
    shape_sum = 0.0
    where_sum = 0.0

    for batch in loader:

        visual = batch[
            "visual"
        ].to(device)

        world_color = batch[
            "world_color"
        ].to(device)

        world_shape = batch[
            "world_shape"
        ].to(device)

        world_xyz = batch[
            "world_xyz"
        ].to(device)

        teacher_optimizer.zero_grad()

        (
            zc,
            zs
        ) = what_teacher(
            visual
        )

        zw = where_teacher(
            visual
        )

        # --------------------------------------------------------
        # Decode WHAT
        # --------------------------------------------------------

        color_logits = (
            what_teacher.color_decoder(
                zc
            )
        )

        shape_logits = (
            what_teacher.shape_decoder(
                zs
            )
        )

        color_loss = F.mse_loss(
            color_logits,
            world_color
        )

        shape_loss = F.mse_loss(
            shape_logits,
            world_shape
        )

        # --------------------------------------------------------
        # Decode WHERE
        # --------------------------------------------------------

        xyz_pred = (
            where_teacher.decoder(
                zw
            )
        )

        where_loss = F.mse_loss(
            xyz_pred,
            world_xyz
        )

        loss = (
            color_loss
            +
            shape_loss
            +
            where_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            teacher_parameters,
            1.0
        )

        teacher_optimizer.step()

        bs = visual.size(0)

        total_sum += (
            loss.item() * bs
        )

        color_sum += (
            color_loss.item() * bs
        )

        shape_sum += (
            shape_loss.item() * bs
        )

        where_sum += (
            where_loss.item() * bs
        )

    n = len(dataset)

    total = total_sum / n
    color = color_sum / n
    shape = shape_sum / n
    where = where_sum / n

    if (
        epoch < 5
        or (epoch + 1) % 5 == 0
        or epoch == teacher_epochs - 1
    ):

        print(
            f"Epoch {epoch+1:03d}/"
            f"{teacher_epochs} | "
            f"Total={total:.6f} | "
            f"Color={color:.6f} | "
            f"Shape={shape:.6f} | "
            f"WHERE={where:.6f}"
        )


# ================================================================
# Freeze Teachers
# ================================================================

for p in what_teacher.parameters():
    p.requires_grad = False

for p in where_teacher.parameters():
    p.requires_grad = False

what_teacher.eval()
where_teacher.eval()

print("\nTeachers FROZEN.")


# ================================================================
# 6. LANGUAGE STUDENT
#
# Language -> PARTIAL WHAT
#
# It does NOT see the world.
# ================================================================

class PartialLanguageStudent(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=96,
        hidden_dim=128,
        what_dim=64
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.color_head = nn.Sequential(
            nn.Linear(
                hidden_dim * 2,
                what_dim
            ),
            nn.Tanh()
        )

        self.shape_head = nn.Sequential(
            nn.Linear(
                hidden_dim * 2,
                what_dim
            ),
            nn.Tanh()
        )

    def forward(self, tokens):

        x = self.embedding(tokens)

        _, hidden = self.gru(x)

        h_forward = hidden[-2]
        h_backward = hidden[-1]

        h = torch.cat(
            [
                h_forward,
                h_backward
            ],
            dim=-1
        )

        z_color = self.color_head(h)
        z_shape = self.shape_head(h)

        return (
            z_color,
            z_shape
        )


student = PartialLanguageStudent(
    vocab_size=len(vocab)
).to(device)

student_optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nStudent parameters:",
    sum(
        p.numel()
        for p in student.parameters()
    )
)


# ================================================================
# 7. PHASE 2
#
# Partial WHAT distillation
#
# Teacher provides WORLD object representations.
#
# For each sample, the target language refers to target_idx.
#
# Only the specified attributes are distilled.
#
# IMPORTANT:
# Teacher itself still represents ALL objects.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 2: PARTIAL LANGUAGE WHAT")
print("=" * 72)

student_epochs = 50

for epoch in range(
    student_epochs
):

    student.train()

    total_sum = 0.0
    color_sum = 0.0
    shape_sum = 0.0

    for batch in loader:

        tokens = batch[
            "tokens"
        ].to(device)

        visual = batch[
            "visual"
        ].to(device)

        target_idx = batch[
            "target_idx"
        ].to(device)

        color_mask = batch[
            "color_specified"
        ].to(device)

        shape_mask = batch[
            "shape_specified"
        ].to(device)

        student_optimizer.zero_grad()

        (
            z_color_L,
            z_shape_L
        ) = student(tokens)

        with torch.no_grad():

            (
                z_color_W,
                z_shape_W
            ) = what_teacher(
                visual
            )

        # --------------------------------------------------------
        # Select only the target object for language supervision.
        #
        # This is NOT passed to Teacher.
        # It is only the supervision relation:
        #
        # Language -> target object's WHAT
        # --------------------------------------------------------

        batch_indices = torch.arange(
            tokens.size(0),
            device=device
        )

        target_color_W = (
            z_color_W[
                batch_indices,
                target_idx
            ]
        )

        target_shape_W = (
            z_shape_W[
                batch_indices,
                target_idx
            ]
        )

        color_dist = (
            z_color_L
            -
            target_color_W
        ).pow(2).mean(dim=-1)

        shape_dist = (
            z_shape_L
            -
            target_shape_W
        ).pow(2).mean(dim=-1)

        if color_mask.any():

            color_loss = (
                color_dist[color_mask].mean()
            )

        else:

            color_loss = torch.tensor(
                0.0,
                device=device
            )

        if shape_mask.any():

            shape_loss = (
                shape_dist[shape_mask].mean()
            )

        else:

            shape_loss = torch.tensor(
                0.0,
                device=device
            )

        loss = (
            color_loss
            +
            shape_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            1.0
        )

        student_optimizer.step()

        bs = tokens.size(0)

        total_sum += (
            loss.item() * bs
        )

        color_sum += (
            color_loss.item() * bs
        )

        shape_sum += (
            shape_loss.item() * bs
        )

    n = len(dataset)

    total = total_sum / n
    color = color_sum / n
    shape = shape_sum / n

    if (
        epoch < 5
        or (epoch + 1) % 5 == 0
        or epoch == student_epochs - 1
    ):

        print(
            f"Student Epoch "
            f"{epoch+1:02d}/"
            f"{student_epochs} | "
            f"Total={total:.6f} | "
            f"Color={color:.6f} | "
            f"Shape={shape:.6f}"
        )


for p in student.parameters():
    p.requires_grad = False

student.eval()

print("\nStudent FROZEN.")


# ================================================================
# 8. Object Matching VLA
#
# Input:
#
#   Language WHAT
#   World WHAT for every object
#   World WHERE for every object
#
# First calculate object compatibility.
#
#   score_i =
#       color similarity
#       + shape similarity
#
# Then soft-select WHERE.
#
# This explicitly implements:
#
#   LANGUAGE WHAT -> OBJECT
#                   -> WHERE
# ================================================================

class ObjectMatchingVLA(nn.Module):

    def __init__(
        self,
        what_dim=64,
        where_dim=128,
        hidden_dim=128
    ):

        super().__init__()

        # --------------------------------------------------------
        # Match language WHAT against one world object's WHAT.
        # --------------------------------------------------------

        self.match_net = nn.Sequential(

            nn.Linear(
                what_dim * 4,
                hidden_dim
            ),

            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                1
            )
        )

        # --------------------------------------------------------
        # Action head receives:
        #
        #   selected WHERE
        #   language WHAT
        # --------------------------------------------------------

        self.action_net = nn.Sequential(

            nn.Linear(
                where_dim
                +
                what_dim
                +
                what_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                3
            )
        )

    def forward(
        self,
        z_color_L,
        z_shape_L,
        z_color_W,
        z_shape_W,
        z_where_W
    ):

        B, N, D = z_color_W.shape

        language_color = (
            z_color_L
            .unsqueeze(1)
            .expand(-1, N, -1)
        )

        language_shape = (
            z_shape_L
            .unsqueeze(1)
            .expand(-1, N, -1)
        )

        # --------------------------------------------------------
        # Pairwise WHAT representation
        # --------------------------------------------------------

        pair = torch.cat(
            [
                language_color,
                language_shape,
                z_color_W,
                z_shape_W
            ],
            dim=-1
        )

        scores = self.match_net(
            pair
        ).squeeze(-1)

        attention = F.softmax(
            scores,
            dim=-1
        )

        # --------------------------------------------------------
        # Selected WHERE
        # --------------------------------------------------------

        selected_where = (
            attention.unsqueeze(-1)
            *
            z_where_W
        ).sum(dim=1)

        # --------------------------------------------------------
        # Action
        # --------------------------------------------------------

        action_input = torch.cat(
            [
                z_color_L,
                z_shape_L,
                selected_where
            ],
            dim=-1
        )

        action = self.action_net(
            action_input
        )

        return (
            action,
            scores,
            attention,
            selected_where
        )


vla = ObjectMatchingVLA().to(device)

vla_optimizer = torch.optim.AdamW(
    vla.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

print(
    "\nVLA parameters:",
    sum(
        p.numel()
        for p in vla.parameters()
    )
)


# ================================================================
# 9. PHASE 3
#
# Object Matching + WHERE -> Action
#
# We train BOTH:
#
#   action loss
#   object selection loss
#
# This is critical.
#
# Otherwise action regression can learn shortcuts.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 3: PARTIAL WHAT -> OBJECT -> WHERE -> ACTION")
print("=" * 72)

vla_epochs = 60

for epoch in range(
    vla_epochs
):

    vla.train()

    action_sum = 0.0
    match_sum = 0.0
    total_sum = 0.0

    for batch in loader:

        tokens = batch[
            "tokens"
        ].to(device)

        visual = batch[
            "visual"
        ].to(device)

        target_idx = batch[
            "target_idx"
        ].to(device)

        action = batch[
            "action"
        ].to(device)

        with torch.no_grad():

            z_color_L, z_shape_L = (
                student(tokens)
            )

            z_color_W, z_shape_W = (
                what_teacher(visual)
            )

            z_where_W = (
                where_teacher(visual)
            )

        # --------------------------------------------------------
        # NOTE:
        #
        # We do not backprop through Teacher or Student.
        # --------------------------------------------------------

        vla_optimizer.zero_grad()

        (
            action_pred,
            scores,
            attention,
            selected_where
        ) = vla(

            z_color_L,
            z_shape_L,

            z_color_W,
            z_shape_W,

            z_where_W
        )

        # --------------------------------------------------------
        # Action loss
        # --------------------------------------------------------

        action_loss = F.mse_loss(
            action_pred,
            action
        )

        # --------------------------------------------------------
        # Object selection loss
        # --------------------------------------------------------

        match_loss = F.cross_entropy(
            scores,
            target_idx
        )

        # --------------------------------------------------------
        # Combined objective
        # --------------------------------------------------------

        loss = (
            action_loss
            +
            0.5 * match_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            vla.parameters(),
            1.0
        )

        vla_optimizer.step()

        bs = tokens.size(0)

        action_sum += (
            action_loss.item()
            * bs
        )

        match_sum += (
            match_loss.item()
            * bs
        )

        total_sum += (
            loss.item()
            * bs
        )

    n = len(dataset)

    action_epoch = action_sum / n
    match_epoch = match_sum / n
    total_epoch = total_sum / n

    print(
        f"VLA Epoch "
        f"{epoch+1:02d}/"
        f"{vla_epochs} | "
        f"Total={total_epoch:.6f} | "
        f"Action={action_epoch:.6f} | "
        f"Match={match_epoch:.6f}"
    )


# ================================================================
# 10. Helpers
# ================================================================

def language_encode(
    instruction
):

    tokens = torch.tensor(
        [
            tokenize(instruction)
        ],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        return student(tokens)


def world_encode(
    sample_index
):

    visual = torch.tensor(
        dataset.samples[
            sample_index
        ]["visual"],
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        zc, zs = what_teacher(
            visual
        )

        zw = where_teacher(
            visual
        )

    return (
        zc,
        zs,
        zw
    )


def run_vla(
    instruction,
    sample_index
):

    zc_L, zs_L = language_encode(
        instruction
    )

    zc_W, zs_W, zw_W = world_encode(
        sample_index
    )

    with torch.no_grad():

        (
            action,
            scores,
            attention,
            selected_where
        ) = vla(
            zc_L,
            zs_L,
            zc_W,
            zs_W,
            zw_W
        )

    return (
        action,
        scores,
        attention,
        selected_where
    )


# ================================================================
# 11. TEST 1
#
# SAME WORLD
# DIFFERENT OBJECT LANGUAGE
#
# This is the central experiment.
# ================================================================

print("\n" + "=" * 72)
print("TEST 1: SAME WORLD / LANGUAGE SWAP")
print("=" * 72)

world_idx = None

for i, s in enumerate(
    dataset.samples
):

    combos = [
        (
            obj["color"],
            obj["shape"]
        )
        for obj in s["objects"]
    ]

    if (
        ("red", "cube") in combos
        and
        ("blue", "elongated") in combos
    ):

        world_idx = i
        break

if world_idx is None:
    raise RuntimeError(
        "Suitable world not found."
    )

world = dataset.samples[
    world_idx
]

print("\nWorld:")

for i, obj in enumerate(
    world["objects"]
):

    print(
        i,
        obj["color"],
        obj["shape"],
        obj["xyz"]
    )


instruction_a = "push red cube"
instruction_b = "push blue elongated"

(
    action_a,
    scores_a,
    attention_a,
    selected_where_a
) = run_vla(
    instruction_a,
    world_idx
)

(
    action_b,
    scores_b,
    attention_b,
    selected_where_b
) = run_vla(
    instruction_b,
    world_idx
)

print(
    "\nLanguage A:",
    instruction_a
)

print(
    "Matching scores:",
    scores_a.cpu().numpy()
)

print(
    "Attention:",
    attention_a.cpu().numpy()
)

print(
    "Action:",
    action_a.cpu().numpy()
)

print(
    "\nLanguage B:",
    instruction_b
)

print(
    "Matching scores:",
    scores_b.cpu().numpy()
)

print(
    "Attention:",
    attention_b.cpu().numpy()
)

print(
    "Action:",
    action_b.cpu().numpy()
)

pred_a = int(
    torch.argmax(
        scores_a,
        dim=-1
    ).item()
)

pred_b = int(
    torch.argmax(
        scores_b,
        dim=-1
    ).item()
)

print(
    "\nPredicted object A:",
    pred_a
)

print(
    "Predicted object B:",
    pred_b
)

print(
    "Expected object A: 0"
)

print(
    "Expected object B: 1"
)


# ================================================================
# 12. TEST 2
#
# SAME LANGUAGE / DIFFERENT WORLD
# ================================================================

print("\n" + "=" * 72)
print("TEST 2: SAME LANGUAGE / DIFFERENT WORLD")
print("=" * 72)

instruction = "push red cube"

idx_a = None
idx_b = None

for i, s in enumerate(
    dataset.samples
):

    combos = [
        (
            obj["color"],
            obj["shape"]
        )
        for obj in s["objects"]
    ]

    if (
        ("red", "cube") in combos
    ):

        if idx_a is None:
            idx_a = i

        elif idx_b is None:

            xyz_a = np.asarray([
                o["xyz"]
                for o in dataset.samples[
                    idx_a
                ]["objects"]
                if (
                    o["color"] == "red"
                    and
                    o["shape"] == "cube"
                )
            ])

            xyz_b = np.asarray([
                o["xyz"]
                for o in s["objects"]
                if (
                    o["color"] == "red"
                    and
                    o["shape"] == "cube"
                )
            ])

            if not np.allclose(
                xyz_a,
                xyz_b
            ):

                idx_b = i
                break

if idx_a is None or idx_b is None:

    raise RuntimeError(
        "Could not find two red-cube worlds."
    )

(
    action_1,
    scores_1,
    attention_1,
    where_1
) = run_vla(
    instruction,
    idx_a
)

(
    action_2,
    scores_2,
    attention_2,
    where_2
) = run_vla(
    instruction,
    idx_b
)

zc1, zs1, zw1 = world_encode(idx_a)
zc2, zs2, zw2 = world_encode(idx_b)

print(
    "\nLanguage:",
    instruction
)

print(
    "\nWorld A:"
)

for i, obj in enumerate(
    dataset.samples[idx_a]["objects"]
):

    print(
        i,
        obj["color"],
        obj["shape"],
        obj["xyz"]
    )

print(
    "\nWorld B:"
)

for i, obj in enumerate(
    dataset.samples[idx_b]["objects"]
):

    print(
        i,
        obj["color"],
        obj["shape"],
        obj["xyz"]
    )

language_zc_1, language_zs_1 = (
    language_encode(instruction)
)

language_zc_2, language_zs_2 = (
    language_encode(instruction)
)

language_diff = (
    torch.mean(
        torch.abs(
            language_zc_1
            -
            language_zc_2
        )
    )
    +
    torch.mean(
        torch.abs(
            language_zs_1
            -
            language_zs_2
        )
    )
)

where_diff = torch.mean(
    torch.abs(
        where_1 - where_2
    )
).item()

action_diff = torch.mean(
    torch.abs(
        action_1 - action_2
    )
).item()

print(
    "\nLanguage representation difference:",
    language_diff.item()
)

print(
    "WHERE difference:",
    where_diff
)

print(
    "Action difference:",
    action_diff
)

print(
    "\nExpected:"
)

print(
    "  Language -> SAME"
)

print(
    "  World WHERE -> DIFFERENT"
)

print(
    "  Action -> DIFFERENT"
)


# ================================================================
# 13. TEST 3
#
# EXPLICIT OBJECT MATCHING ACCURACY
# ================================================================

print("\n" + "=" * 72)
print("TEST 3: OBJECT MATCHING ACCURACY")
print("=" * 72)

correct = 0
total = 0

with torch.no_grad():

    for i in range(
        min(
            len(dataset),
            2000
        )
    ):

        s = dataset.samples[i]

        (
            action,
            scores,
            attention,
            selected_where
        ) = run_vla(
            s["instruction"],
            i
        )

        pred = int(
            torch.argmax(
                scores,
                dim=-1
            ).item()
        )

        if pred == s["target_idx"]:
            correct += 1

        total += 1

matching_accuracy = (
    correct / total
)

print(
    f"\nObject matching accuracy: "
    f"{matching_accuracy:.4f}"
)

print(
    f"Correct: {correct}/{total}"
)


# ================================================================
# 14. TEST 4
#
# WHERE SWAP
#
# Same language WHAT.
# Swap physical locations between the two objects.
#
# The selected object's WHERE should follow the swap.
# ================================================================

print("\n" + "=" * 72)
print("TEST 4: WHERE SWAP")
print("=" * 72)

swap_idx = world_idx

original_objects = dataset.samples[
    swap_idx
]["objects"]

print("\nOriginal:")

for i, obj in enumerate(
    original_objects
):

    print(
        i,
        obj["color"],
        obj["shape"],
        obj["xyz"]
    )

(
    original_action,
    original_scores,
    original_attention,
    original_where
) = run_vla(
    "push red cube",
    swap_idx
)

# ---------------------------------------------------------------
# Create a temporary swapped world.
#
# We cannot mutate the trained dataset permanently.
# We construct a new visual representation.
# ---------------------------------------------------------------

swapped_objects = [

    {
        "color":

_IncompleteInputError: incomplete input (141214872.py, line 2061)

In [ ]:
# ================================================================
# AntEncoder Experiment 10
# Partial WHAT -> Object Selection -> WHERE -> Action
#
# Core hypothesis:
#
#   Teacher = WORLD
#   Student = LANGUAGE
#
#   WORLD
#       -> object-wise WHAT
#       -> object-wise WHERE
#
#   LANGUAGE
#       -> Partial WHAT
#
#   Partial WHAT
#       x
#   World WHAT
#       ->
#   OBJECT SELECTION
#       ->
#   selected WHERE
#       ->
#   ACTION
#
# IMPORTANT:
#   We do NOT require Language to reconstruct the whole world.
#   Language only produces the attributes explicitly present
#   in the instruction.
#
# ================================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ================================================================
# 0. Reproducibility
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("AntEncoder Experiment 10")
print("Partial WHAT -> Object Selection -> WHERE -> Action")
print("=" * 72)

print("Device:", device)


# ================================================================
# 1. Vocabulary
# ================================================================

instructions = [
    "push red object",
    "push blue object",
    "push cube",
    "push elongated object",
    "push red cube",
    "push red elongated",
    "push blue cube",
    "push blue elongated",
    "push the object",
]

all_words = set()

for text in instructions:
    all_words.update(text.lower().split())

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for word in sorted(all_words):
    if word not in vocab:
        vocab[word] = len(vocab)

PAD_IDX = vocab["<PAD>"]
UNK_IDX = vocab["<UNK>"]

print("\nVocabulary size:", len(vocab))
print("Vocabulary:", vocab)


def tokenize(text, max_len=6):

    words = text.lower().split()

    ids = [
        vocab.get(word, UNK_IDX)
        for word in words
    ]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [
            PAD_IDX
        ] * (
            max_len - len(ids)
        )

    return ids


# ================================================================
# 2. Dataset
#
# Two objects per world.
#
# Each object has:
#
#   WHAT:
#       color
#       shape
#
#   WHERE:
#       xyz
#
# The dataset also gives us the ground-truth object index
# for evaluation.
# ================================================================

class Experiment10Dataset(Dataset):

    COLORS = ["red", "blue"]
    SHAPES = ["cube", "elongated"]

    COLOR_VEC = {
        "red": np.array(
            [1.0, 0.0],
            dtype=np.float32
        ),
        "blue": np.array(
            [0.0, 1.0],
            dtype=np.float32
        ),
    }

    SHAPE_VEC = {
        "cube": np.array(
            [1.0, 0.0],
            dtype=np.float32
        ),
        "elongated": np.array(
            [0.0, 1.0],
            dtype=np.float32
        ),
    }

    def __init__(
        self,
        n_samples=8000,
        seed=42
    ):

        self.n_samples = n_samples

        self.rng = np.random.default_rng(seed)

        self.samples = []

        combinations = [
            ("red", "cube"),
            ("red", "elongated"),
            ("blue", "cube"),
            ("blue", "elongated"),
        ]

        # --------------------------------------------------------
        # Fixed visual projection.
        #
        # Physical representation:
        #
        # object:
        #   color 2
        #   shape 2
        #   xyz   3
        #
        # = 7
        #
        # 2 objects = 14
        #
        # --------------------------------------------------------

        physical_dim = 14
        visual_dim = 512

        projection_rng = np.random.default_rng(
            12345
        )

        self.visual_projection = (
            projection_rng.normal(
                0.0,
                1.0 / np.sqrt(physical_dim),
                size=(
                    physical_dim,
                    visual_dim
                )
            ).astype(
                np.float32
            )
        )

        # --------------------------------------------------------
        # Generate worlds
        # --------------------------------------------------------

        for _ in range(n_samples):

            pair = self.rng.choice(
                len(combinations),
                size=2,
                replace=False
            )

            objects = []

            for pair_idx in pair:

                color, shape = combinations[pair_idx]

                xyz = self.rng.uniform(
                    low=[
                        -1.0,
                        -1.0,
                        0.35
                    ],
                    high=[
                        1.0,
                        1.0,
                        1.0
                    ]
                ).astype(
                    np.float32
                )

                objects.append({
                    "color": color,
                    "shape": shape,
                    "xyz": xyz,
                })

            # ----------------------------------------------------
            # Target object
            # ----------------------------------------------------

            target_idx = int(
                self.rng.integers(0, 2)
            )

            target = objects[target_idx]

            target_color = target["color"]
            target_shape = target["shape"]

            # ----------------------------------------------------
            # Language abstraction
            # ----------------------------------------------------

            language_type = int(
                self.rng.integers(0, 4)
            )

            if language_type == 0:

                instruction = (
                    f"push {target_color} object"
                )

                color_specified = True
                shape_specified = False

            elif language_type == 1:

                instruction = (
                    f"push {target_shape} object"
                )

                color_specified = False
                shape_specified = True

            elif language_type == 2:

                instruction = (
                    f"push {target_color} "
                    f"{target_shape}"
                )

                color_specified = True
                shape_specified = True

            else:

                instruction = "push the object"

                color_specified = False
                shape_specified = False

            # ----------------------------------------------------
            # Action target
            #
            # Simple push trajectory target.
            # ----------------------------------------------------

            direction = int(
                self.rng.integers(0, 2)
            )

            if direction == 0:

                action_name = "right"

                action = np.array(
                    [
                        target["xyz"][0] + 0.35,
                        target["xyz"][1],
                        target["xyz"][2],
                    ],
                    dtype=np.float32
                )

            else:

                action_name = "left"

                action = np.array(
                    [
                        target["xyz"][0] - 0.35,
                        target["xyz"][1],
                        target["xyz"][2],
                    ],
                    dtype=np.float32
                )

            # ----------------------------------------------------
            # World WHAT representation
            #
            # One vector per object.
            #
            # color 2
            # shape 2
            #
            # = 4
            # ----------------------------------------------------

            world_what = []

            world_where = []

            physical = []

            for obj in objects:

                what = np.concatenate([
                    self.COLOR_VEC[obj["color"]],
                    self.SHAPE_VEC[obj["shape"]],
                ])

                world_what.append(what)

                world_where.append(
                    obj["xyz"]
                )

                physical.extend(
                    self.COLOR_VEC[obj["color"]]
                )

                physical.extend(
                    self.SHAPE_VEC[obj["shape"]]
                )

                physical.extend(
                    obj["xyz"]
                )

            world_what = np.asarray(
                world_what,
                dtype=np.float32
            )

            world_where = np.asarray(
                world_where,
                dtype=np.float32
            )

            physical = np.asarray(
                physical,
                dtype=np.float32
            )

            # ----------------------------------------------------
            # Physical -> visual
            # ----------------------------------------------------

            visual = (
                physical
                @ self.visual_projection
            )

            visual += self.rng.normal(
                0.0,
                0.02,
                size=512
            ).astype(
                np.float32
            )

            self.samples.append({

                "instruction":
                    instruction,

                "tokens":
                    np.asarray(
                        tokenize(instruction),
                        dtype=np.int64
                    ),

                "visual":
                    visual.astype(
                        np.float32
                    ),

                "world_what":
                    world_what,

                "world_where":
                    world_where,

                "objects":
                    objects,

                "target_idx":
                    target_idx,

                "action":
                    action,

                "action_name":
                    action_name,

                "color_specified":
                    color_specified,

                "shape_specified":
                    shape_specified,

                "target_color":
                    target_color,

                "target_shape":
                    target_shape,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        s = self.samples[idx]

        return {

            "tokens":
                torch.tensor(
                    s["tokens"],
                    dtype=torch.long
                ),

            "visual":
                torch.tensor(
                    s["visual"],
                    dtype=torch.float32
                ),

            "world_what":
                torch.tensor(
                    s["world_what"],
                    dtype=torch.float32
                ),

            "world_where":
                torch.tensor(
                    s["world_where"],
                    dtype=torch.float32
                ),

            "target_idx":
                torch.tensor(
                    s["target_idx"],
                    dtype=torch.long
                ),

            "action":
                torch.tensor(
                    s["action"],
                    dtype=torch.float32
                ),

            "color_specified":
                torch.tensor(
                    s["color_specified"],
                    dtype=torch.bool
                ),

            "shape_specified":
                torch.tensor(
                    s["shape_specified"],
                    dtype=torch.bool
                ),
        }


dataset = Experiment10Dataset(
    n_samples=8000,
    seed=SEED
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

print("\nDataset size:", len(dataset))
print(
    "World WHAT shape:",
    dataset.samples[0]["world_what"].shape
)
print(
    "World WHERE shape:",
    dataset.samples[0]["world_where"].shape
)
print(
    "Visual dimension:",
    dataset.samples[0]["visual"].shape[0]
)


# ================================================================
# 3. WORLD TEACHER
#
# Teacher = WORLD
#
# Vision -> object-wise WHAT
#
# IMPORTANT:
#
# The teacher is NOT given target_idx.
#
# It must represent both objects.
# ================================================================

class WorldWHATTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        what_dim=64
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU()
        )

        self.object_heads = nn.Sequential(

            nn.Linear(
                128,
                what_dim
            ),

            nn.Tanh()
        )

        self.decoder = nn.Linear(
            what_dim,
            4
        )

    def forward(self, visual):

        h = self.encoder(visual)

        # Same world representation is used to construct
        # candidate object slots.
        #
        # The slots are decoded independently.
        #
        # In this synthetic experiment the ordering of the
        # physical representation is fixed.

        z = self.object_heads(h)

        # Two object slots.
        z0 = z
        z1 = z

        return z0, z1


# ================================================================
# Better explicit world teacher:
#
# Because this experiment is about object-wise representations,
# we use separate visual slices for the two candidate objects.
#
# Each object occupies 7 physical dimensions.
#
# We first reconstruct the physical object representation
# from the complete visual world.
# ================================================================

class ObjectWiseWorldTeacher(nn.Module):

    def __init__(
        self,
        input_dim=512,
        what_dim=64,
        where_dim=64
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU()
        )

        self.slot0 = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, what_dim + where_dim),
            nn.Tanh()
        )

        self.slot1 = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, what_dim + where_dim),
            nn.Tanh()
        )

        self.what_decoder = nn.Linear(
            what_dim,
            4
        )

        self.where_decoder = nn.Linear(
            where_dim,
            3
        )

    def forward(self, visual):

        h = self.encoder(visual)

        out0 = self.slot0(h)
        out1 = self.slot1(h)

        z_what0 = out0[:, :64]
        z_where0 = out0[:, 64:]

        z_what1 = out1[:, :64]
        z_where1 = out1[:, 64:]

        return (
            z_what0,
            z_where0,
            z_what1,
            z_where1
        )


teacher = ObjectWiseWorldTeacher().to(device)

teacher_optimizer = torch.optim.AdamW(
    teacher.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

teacher_params = list(
    teacher.parameters()
)

print(
    "\nTeacher parameters:",
    sum(
        p.numel()
        for p in teacher_params
    )
)


# ================================================================
# 4. Teacher training
#
# Vision -> ALL OBJECTS
#
# No target information enters the teacher.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 1: WORLD TEACHER")
print("=" * 72)

teacher_epochs = 60


for epoch in range(teacher_epochs):

    teacher.train()

    total_sum = 0.0
    what_sum = 0.0
    where_sum = 0.0

    for batch in loader:

        visual = batch["visual"].to(device)

        world_what = batch[
            "world_what"
        ].to(device)

        world_where = batch[
            "world_where"
        ].to(device)

        teacher_optimizer.zero_grad()

        (
            z_what0,
            z_where0,
            z_what1,
            z_where1
        ) = teacher(visual)

        # --------------------------------------------------------
        # Decode WHAT
        # --------------------------------------------------------

        what0_pred = teacher.what_decoder(
            z_what0
        )

        what1_pred = teacher.what_decoder(
            z_what1
        )

        # --------------------------------------------------------
        # Decode WHERE
        # --------------------------------------------------------

        where0_pred = teacher.where_decoder(
            z_where0
        )

        where1_pred = teacher.where_decoder(
            z_where1
        )

        # --------------------------------------------------------
        # Target slots
        # --------------------------------------------------------

        what0_target = world_what[:, 0]
        what1_target = world_what[:, 1]

        where0_target = world_where[:, 0]
        where1_target = world_where[:, 1]

        # --------------------------------------------------------
        # Loss
        # --------------------------------------------------------

        what_loss = (
            F.mse_loss(
                what0_pred,
                what0_target
            )
            +
            F.mse_loss(
                what1_pred,
                what1_target
            )
        )

        where_loss = (
            F.mse_loss(
                where0_pred,
                where0_target
            )
            +
            F.mse_loss(
                where1_pred,
                where1_target
            )
        )

        loss = (
            what_loss
            +
            where_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            teacher_params,
            1.0
        )

        teacher_optimizer.step()

        bs = visual.size(0)

        total_sum += loss.item() * bs
        what_sum += what_loss.item() * bs
        where_sum += where_loss.item() * bs

    n = len(dataset)

    total = total_sum / n
    what = what_sum / n
    where = where_sum / n

    if (
        epoch < 5
        or (epoch + 1) % 10 == 0
        or epoch == teacher_epochs - 1
    ):

        print(
            f"Epoch {epoch+1:03d}/{teacher_epochs} | "
            f"Total={total:.6f} | "
            f"WHAT={what:.6f} | "
            f"WHERE={where:.6f}"
        )


# ================================================================
# Freeze teacher
# ================================================================

for p in teacher.parameters():
    p.requires_grad = False

teacher.eval()

print("\nWorld Teacher FROZEN.")


# ================================================================
# 5. LANGUAGE STUDENT
#
# Language -> Partial WHAT
#
# The student does NOT see the world.
#
# ================================================================

class PartialWHATStudent(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=96,
        hidden_dim=128,
        what_dim=64
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.what_head = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                what_dim
            ),

            nn.Tanh()
        )

    def forward(self, tokens):

        x = self.embedding(tokens)

        _, hidden = self.gru(x)

        h = torch.cat(
            [
                hidden[-2],
                hidden[-1]
            ],
            dim=-1
        )

        z_what = self.what_head(h)

        return z_what


student = PartialWHATStudent(
    vocab_size=len(vocab)
).to(device)

student_optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

student_params = list(
    student.parameters()
)

print(
    "\nStudent parameters:",
    sum(
        p.numel()
        for p in student_params
    )
)


# ================================================================
# 6. Student training
#
# Student learns partial WHAT.
#
# To make the representation explicit:
#
#   color-only -> match color portion
#   shape-only -> match shape portion
#   both       -> match both
#   neither    -> no WHAT reconstruction
#
# We derive language targets from the known synthetic object
# attributes, while the WORLD teacher remains the source of
# visual representation.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 2: PARTIAL LANGUAGE WHAT")
print("=" * 72)

student_epochs = 50


# Fixed projections from 4D WHAT -> 64D semantic target.
# These are not learned jointly with the student.
color_basis = torch.tensor(
    np.concatenate([
        np.array([1, 0, 0, 0], dtype=np.float32),
        np.zeros(60, dtype=np.float32)
    ]),
    dtype=torch.float32,
    device=device
).unsqueeze(0)

shape_basis = torch.tensor(
    np.concatenate([
        np.array([0, 0, 1, 0], dtype=np.float32),
        np.zeros(60, dtype=np.float32)
    ]),
    dtype=torch.float32,
    device=device
).unsqueeze(0)


for epoch in range(student_epochs):

    student.train()

    total_sum = 0.0

    for batch in loader:

        tokens = batch["tokens"].to(device)

        color_mask = batch[
            "color_specified"
        ].to(device)

        shape_mask = batch[
            "shape_specified"
        ].to(device)

        student_optimizer.zero_grad()

        z_language = student(tokens)

        # --------------------------------------------------------
        # We use the world teacher to obtain the target object's
        # semantic representation.
        #
        # IMPORTANT:
        # target object is used only for training the language
        # mapping; teacher itself never receives target_idx.
        # --------------------------------------------------------

        with torch.no_grad():

            (
                z0,
                _,
                z1,
                _
            ) = teacher(
                batch["visual"].to(device)
            )

            target_idx = batch[
                "target_idx"
            ].to(device)

            z_world_target = torch.where(
                target_idx.unsqueeze(-1) == 0,
                z0,
                z1
            )

        # --------------------------------------------------------
        # Partial loss
        #
        # We deliberately use only the semantic components
        # requested by language.
        #
        # The teacher representation itself is learned from the
        # complete world.
        # --------------------------------------------------------

        # Since teacher latent is learned, we use cosine-normalized
        # distillation for specified cases.
        teacher_norm = F.normalize(
            z_world_target,
            dim=-1
        )

        student_norm = F.normalize(
            z_language,
            dim=-1
        )

        full_distill = (
            1.0
            -
            (
                student_norm
                *
                teacher_norm
            ).sum(dim=-1)
        )

        specified = (
            color_mask
            |
            shape_mask
        )

        if specified.any():

            loss = full_distill[
                specified
            ].mean()

        else:

            loss = torch.tensor(
                0.0,
                device=device,
                requires_grad=True
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student_params,
            1.0
        )

        student_optimizer.step()

        total_sum += (
            loss.item()
            *
            tokens.size(0)
        )

    total = (
        total_sum
        /
        len(dataset)
    )

    if (
        epoch < 5
        or (epoch + 1) % 10 == 0
        or epoch == student_epochs - 1
    ):

        print(
            f"Student Epoch "
            f"{epoch+1:02d}/{student_epochs} | "
            f"Loss={total:.6f}"
        )


# ================================================================
# Freeze student
# ================================================================

for p in student.parameters():
    p.requires_grad = False

student.eval()

print("\nLanguage Student FROZEN.")


# ================================================================
# 7. Explicit Object Matching
#
# Language WHAT
#       x
# World WHAT object 0
#
# Language WHAT
#       x
# World WHAT object 1
#
# -> scores
# -> selected object
# ================================================================

class ObjectSelector(nn.Module):

    def __init__(
        self,
        what_dim=64
    ):

        super().__init__()

        self.temperature = nn.Parameter(
            torch.tensor(0.1)
        )

    def forward(
        self,
        z_language,
        world_what0,
        world_what1
    ):

        q = F.normalize(
            z_language,
            dim=-1
        )

        k0 = F.normalize(
            world_what0,
            dim=-1
        )

        k1 = F.normalize(
            world_what1,
            dim=-1
        )

        score0 = (
            q * k0
        ).sum(dim=-1)

        score1 = (
            q * k1
        ).sum(dim=-1)

        scores = torch.stack(
            [
                score0,
                score1
            ],
            dim=-1
        )

        return scores


selector = ObjectSelector().to(device)


# ================================================================
# 8. ACTION HEAD
#
# Selected WHERE -> Action
#
# The selector determines WHICH object.
#
# WHERE determines WHERE the action occurs.
# ================================================================

class WhereToAction(nn.Module):

    def __init__(
        self,
        where_dim=64,
        action_dim=3
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                where_dim,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                action_dim
            )
        )

    def forward(self, z_where):

        return self.net(z_where)


action_head = WhereToAction().to(device)

action_optimizer = torch.optim.AdamW(
    action_head.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)


# ================================================================
# 9. PHASE 3
#
# Partial WHAT
#      ->
# Object Selection
#      ->
# Selected WHERE
#      ->
# Action
#
# IMPORTANT:
# Selection itself is supervised in this synthetic experiment.
#
# This gives us a clean test of whether the proposed factorization
# works before moving to weaker supervision.
# ================================================================

print("\n" + "=" * 72)
print("PHASE 3: WHAT -> OBJECT SELECTION -> WHERE -> ACTION")
print("=" * 72)

action_epochs = 40


for epoch in range(action_epochs):

    student.eval()
    teacher.eval()
    action_head.train()

    total_sum = 0.0
    action_sum = 0.0
    selection_sum = 0.0

    for batch in loader:

        tokens = batch["tokens"].to(device)
        visual = batch["visual"].to(device)

        world_where = batch[
            "world_where"
        ].to(device)

        target_idx = batch[
            "target_idx"
        ].to(device)

        action_target = batch[
            "action"
        ].to(device)

        action_optimizer.zero_grad()

        with torch.no_grad():

            z_language = student(
                tokens
            )

            (
                z_what0,
                z_where0,
                z_what1,
                z_where1
            ) = teacher(
                visual
            )

        # --------------------------------------------------------
        # Selection
        # --------------------------------------------------------

        scores = selector(
            z_language,
            z_what0,
            z_what1
        )

        selection_loss = F.cross_entropy(
            scores,
            target_idx
        )

        # --------------------------------------------------------
        # Differentiable WHERE selection
        #
        # Softmax over object candidates.
        # --------------------------------------------------------

        weights = F.softmax(
            scores / 0.1,
            dim=-1
        )

        z_selected_where = (
            weights[:, 0:1]
            * z_where0
            +
            weights[:, 1:2]
            * z_where1
        )

        # --------------------------------------------------------
        # Action
        # --------------------------------------------------------

        action_pred = action_head(
            z_selected_where
        )

        action_loss = F.mse_loss(
            action_pred,
            action_target
        )

        loss = (
            selection_loss
            +
            action_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            action_head.parameters(),
            1.0
        )

        action_optimizer.step()

        bs = tokens.size(0)

        total_sum += (
            loss.item() * bs
        )

        action_sum += (
            action_loss.item() * bs
        )

        selection_sum += (
            selection_loss.item() * bs
        )

    n = len(dataset)

    total = total_sum / n
    action_value = action_sum / n
    selection_value = selection_sum / n

    if (
        epoch < 5
        or (epoch + 1) % 5 == 0
        or epoch == action_epochs - 1
    ):

        print(
            f"Epoch {epoch+1:02d}/{action_epochs} | "
            f"Total={total:.6f} | "
            f"Selection={selection_value:.6f} | "
            f"Action={action_value:.6f}"
        )


# ================================================================
# 10. Helper
# ================================================================

def encode_language(text):

    tokens = torch.tensor(
        [tokenize(text)],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        z = student(tokens)

    return z


def encode_world(index):

    visual = torch.tensor(
        dataset.samples[index]["visual"],
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        (
            z_what0,
            z_where0,
            z_what1,
            z_where1
        ) = teacher(visual)

    return (
        z_what0,
        z_where0,
        z_what1,
        z_where1
    )


def run_selection(
    instruction,
    world_index
):

    z_language = encode_language(
        instruction
    )

    (
        z_what0,
        z_where0,
        z_what1,
        z_where1
    ) = encode_world(
        world_index
    )

    with torch.no_grad():

        scores = selector(
            z_language,
            z_what0,
            z_what1
        )

        selected = int(
            torch.argmax(
                scores,
                dim=-1
            ).item()
        )

        weights = F.softmax(
            scores / 0.1,
            dim=-1
        )

        z_selected_where = (
            weights[:, 0:1]
            * z_where0
            +
            weights[:, 1:2]
            * z_where1
        )

        action = action_head(
            z_selected_where
        )

    return (
        scores,
        weights,
        selected,
        z_selected_where,
        action
    )


# ================================================================
# 11. Find a controlled world
#
# We specifically find:
#
#   object 0 = red cube
#   object 1 = blue elongated
#
# This gives the cleanest WHAT swap test.
# ================================================================

world_index = None

for i, s in enumerate(dataset.samples):

    combos = [
        (
            obj["color"],
            obj["shape"]
        )
        for obj in s["objects"]
    ]

    if (
        ("red", "cube") in combos
        and
        ("blue", "elongated") in combos
    ):

        world_index = i
        break


if world_index is None:

    raise RuntimeError(
        "Controlled world not found."
    )


world = dataset.samples[
    world_index
]

print("\nControlled world index:", world_index)

for i, obj in enumerate(world["objects"]):

    print(
        i,
        obj["color"],
        obj["shape"],
        obj["xyz"]
    )


# ================================================================
# 12. TEST 1
#
# SAME WORLD / LANGUAGE SWAP
#
# ------------------------------------------------
# red cube
# blue elongated
#
# Expected:
#
# WHAT changes
# selected object changes
# WHERE reference changes
# Action changes
# ================================================================

print("\n" + "=" * 72)
print("TEST 1: SAME WORLD / LANGUAGE SWAP")
print("=" * 72)

instruction_a = "push red cube"
instruction_b = "push blue elongated"

(
    scores_a,
    weights_a,
    selected_a,
    selected_where_a,
    action_a
) = run_selection(
    instruction_a,
    world_index
)

(
    scores_b,
    weights_b,
    selected_b,
    selected_where_b,
    action_b
) = run_selection(
    instruction_b,
    world_index
)

print(
    "\nLanguage A:",
    instruction_a
)

print(
    "Scores:",
    scores_a.cpu().numpy()
)

print(
    "Selection weights:",
    weights_a.cpu().numpy()
)

print(
    "Selected object:",
    selected_a
)

print(
    "\nLanguage B:",
    instruction_b
)

print(
    "Scores:",
    scores_b.cpu().numpy()
)

print(
    "Selection weights:",
    weights_b.cpu().numpy()
)

print(
    "Selected object:",
    selected_b
)

print(
    "\nExpected:"
)

print(
    "  red cube       -> object 0"
)

print(
    "  blue elongated -> object 1"
)

print(
    "\nAction A:",
    action_a.cpu().numpy()
)

print(
    "Action B:",
    action_b.cpu().numpy()
)

print(
    "\nAction difference:",
    torch.mean(
        torch.abs(
            action_a - action_b
        )
    ).item()
)


# ================================================================
# 13. TEST 2
#
# SAME LANGUAGE / WHERE SWAP
#
# We create a counterfactual where the objects retain their
# identity but exchange positions.
#
# The language must remain unchanged.
#
# ================================================================

print("\n" + "=" * 72)
print("TEST 2: SAME LANGUAGE / WHERE SWAP")
print("=" * 72)

# ------------------------------------------------
# Original:
#   object 0 red cube
#   object 1 blue elongated
# ------------------------------------------------

original_positions = np.stack([
    world["objects"][0]["xyz"],
    world["objects"][1]["xyz"],
])

swapped_positions = np.stack([
    world["objects"][1]["xyz"],
    world["objects"][0]["xyz"],
])


# ------------------------------------------------
# We cannot directly modify the frozen teacher.
# Instead we measure the downstream effect by replacing
# the WHERE representations with the swapped values after
# encoding them through the same fixed coordinate encoder.
#
# For a clean controlled test, we use the teacher's original
# WHERE latent and exchange the two slots.
# ------------------------------------------------

(
    z_what0,
    z_where0,
    z_what1,
    z_where1
) = encode_world(
    world_index
)

instruction = "push red cube"

with torch.no_grad():

    z_language = encode_language(
        instruction
    )

    scores = selector(
        z_language,
        z_what0,
        z_what1
    )

    weights = F.softmax(
        scores / 0.1,
        dim=-1
    )

    # Original
    where_original = (
        weights[:, 0:1] * z_where0
        +
        weights[:, 1:2] * z_where1
    )

    # Counterfactual WHERE swap
    where_swapped = (
        weights[:, 0:1] * z_where1
        +
        weights[:, 1:2] * z_where0
    )

    action_original = action_head(
        where_original
    )

    action_swapped = action_head(
        where_swapped
    )

print(
    "\nInstruction:",
    instruction
)

print(
    "Original red cube XYZ:",
    original_positions[0]
)

print(
    "Original blue elongated XYZ:",
    original_positions[1]
)

print(
    "\nWHERE slot swap:"
)

print(
    "  red cube receives blue object's WHERE"
)

print(
    "  blue elongated receives red object's WHERE"
)

print(
    "\nOriginal action:",
    action_original.cpu().numpy()
)

print(
    "Action after WHERE swap:",
    action_swapped.cpu().numpy()
)

print(
    "\nAction difference:",
    torch.mean(
        torch.abs(
            action_original
            -
            action_swapped
        )
    ).item()
)


# ================================================================
# 14. TEST 3
#
# OBJECT SELECTION ACCURACY
#
# ================================================================

print("\n" + "=" * 72)
print("TEST 3: OBJECT SELECTION ACCURACY")
print("=" * 72)

teacher.eval()
student.eval()
action_head.eval()

correct = 0
total = 0

for batch in DataLoader(
    dataset,
    batch_size=256,
    shuffle=False
):

    tokens = batch["tokens"].to(device)
    visual = batch["visual"].to(device)
    target_idx = batch[
        "target_idx"
    ].to(device)

    with torch.no_grad():

        z_language = student(tokens)

        (
            z_what0,
            _,
            z_what1,
            _
        ) = teacher(visual)

        scores = selector(
            z_language,
            z_what0,
            z_what1
        )

        prediction = torch.argmax(
            scores,
            dim=-1
        )

        correct += (
            prediction
            ==
            target_idx
        ).sum().item()

        total += (
            target_idx.size(0)
        )


selection_accuracy = (
    correct / total
)

print(
    "\nSelection accuracy:",
    f"{selection_accuracy * 100:.2f}%"
)


# ================================================================
# 15. TEST 4
#
# ABLATION
#
# WHAT = zero
#
# If WHAT is actually responsible for selection,
# removing it should destroy selection performance.
# ================================================================

print("\n" + "=" * 72)
print("TEST 4: WHAT ABLATION")
print("=" * 72)

correct_zero = 0
total_zero = 0

for batch in DataLoader(
    dataset,
    batch_size=256,
    shuffle=False
):

    visual = batch["visual"].to(device)
    target_idx = batch[
        "target_idx"
    ].to(device)

    with torch.no_grad():

        (
            z_what0,
            _,
            z_what1,
            _
        ) = teacher(visual)

        zero_what = torch.zeros(
            visual.size(0),
            64,
            device=device
        )

        scores = selector(
            zero_what,
            z_what0,
            z_what1
        )

        prediction = torch.argmax(
            scores,
            dim=-1
        )

        correct_zero += (
            prediction
            ==
            target_idx
        ).sum().item()

        total_zero += (
            target_idx.size(0)
        )

zero_accuracy = (
    correct_zero / total_zero
)

print(
    "\nNormal selection:",
    f"{selection_accuracy * 100:.2f}%"
)

print(
    "Zero WHAT selection:",
    f"{zero_accuracy * 100:.2f}%"
)

print(
    "\nExpected:"
)

print(
    "  WHAT removal -> selection degradation"
)


# ================================================================
# 16. FINAL REPORT
# ================================================================

print("\n" + "=" * 72)
print("EXPERIMENT 10 FINAL REPORT")
print("=" * 72)

print(
    """
Core hypothesis:

    Teacher = World
    Student = Language

    World:
        object-wise WHAT
        object-wise WHERE

    Language:
        Partial WHAT

    Partial WHAT
        x
    World WHAT
        ->
    Object Selection
        ->
    Selected WHERE
        ->
    Action
"""
)

print(
    "Selection accuracy:",
    f"{selection_accuracy * 100:.2f}%"
)

print(
    "Zero-WHAT accuracy:",
    f"{zero_accuracy * 100:.2f}%"
)

print(
    "\nControlled language swap:"
)

print(
    "  red cube       -> selected:",
    selected_a
)

print(
    "  blue elongated -> selected:",
    selected_b
)

print(
    "\nWHERE swap action difference:",
    torch.mean(
        torch.abs(
            action_original
            -
            action_swapped
        )
    ).item()
)

print(
    """
Interpretation:

    SUCCESS:

        Language WHAT selects the correct object.

        The selected object's WHERE is then used
        to generate the action.

        Changing WHAT changes which WHERE is selected.

        Changing WHERE changes the spatial action
        while preserving the language/object identity.

    FAILURE:

        Selection remains near chance,
        or WHAT changes do not change selected object,
        or WHERE changes do not affect spatial action.

Next step:

    If SUCCESS:

        Partial WHAT
             +
        Object-centric World representation
             ↓
        Object selection
             ↓
        WHERE
             ↓
        Trajectory / Grasp Action
    """
)

print("=" * 72)
print("Experiment 10 finished.")
print("=" * 72)


AntEncoder Experiment 10
Partial WHAT -> Object Selection -> WHERE -> Action
Device: cpu

Vocabulary size: 9
Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'blue': 2, 'cube': 3, 'elongated': 4, 'object': 5, 'push': 6, 'red': 7, 'the': 8}

Dataset size: 8000
World WHAT shape: (2, 4)
World WHERE shape: (2, 3)
Visual dimension: 512

Teacher parameters: 230727

PHASE 1: WORLD TEACHER
Epoch 001/60 | Total=0.211713 | WHAT=0.088535 | WHERE=0.123177
Epoch 002/60 | Total=0.012915 | WHAT=0.001975 | WHERE=0.010940
Epoch 003/60 | Total=0.003795 | WHAT=0.001154 | WHERE=0.002641
Epoch 004/60 | Total=0.002908 | WHAT=0.000649 | WHERE=0.002259
Epoch 005/60 | Total=0.002310 | WHAT=0.000666 | WHERE=0.001644
Epoch 010/60 | Total=0.002101 | WHAT=0.000501 | WHERE=0.001599
Epoch 020/60 | Total=0.001347 | WHAT=0.000248 | WHERE=0.001099
Epoch 030/60 | Total=0.001044 | WHAT=0.000222 | WHERE=0.000822
Epoch 040/60 | Total=0.000684 | WHAT=0.000168 | WHERE=0.000516
Epoch 050/60 | Total=0.000768 | WHAT=0.000099 | WHERE=0.0006